# Preliminaries

In [ ]:
!pip install iterative-stratification

In [ ]:
import csv
import time
import json
import random
import re
import os
import shutil
import zipfile
import math
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import openpyxl
import scipy.stats as stats
from scipy.spatial.distance import cosine, pdist
from scipy.cluster.hierarchy import linkage, leaves_list, dendrogram, fcluster
from scipy.ndimage import label

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from IPython.display import Image, display

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (f1_score, roc_auc_score, average_precision_score,
                             precision_recall_curve, roc_curve, auc)
from sklearn.model_selection import train_test_split
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold, MultilabelStratifiedShuffleSplit

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau

from google.colab import files

print("CUDA available:", torch.cuda.is_available())
print("GPU device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Functions and Defenitions

In [5]:
def load_masked_multilabel_dataset(csv_path, num_labels=122):
    sequences = []
    targets = []
    masks = []
    label_counter = Counter()

    with open(csv_path, newline='') as file:
        reader = csv.DictReader(file)
        for row in reader:
            seq_vector = [int(bit) for bit in row["sequence"]]
            raw_labels = row["labels"].split(";")

            # Initialize target and mask
            target = np.zeros(num_labels, dtype=np.float32)
            mask = np.zeros(num_labels, dtype=np.float32)

            for label_str in raw_labels:
                label = int(label_str)
                abs_label = abs(label)

                if abs_label < 1 or abs_label > num_labels:
                    continue  # skip out-of-range

                index = abs_label - 1  # map label 1-122 → index 0-121

                if label > 0:
                    target[index] = 1
                    mask[index] = 1
                elif label < 0:
                    target[index] = 0
                    mask[index] = 1

                label_counter[index] += 1

            sequences.append(seq_vector)
            targets.append(target)
            masks.append(mask)

    label_counts = [label_counter.get(i, 0) for i in range(num_labels)]

    return np.array(sequences, dtype=np.uint8), np.array(targets), np.array(masks), label_counts


In [6]:
class MaskedMultilabelDataset(Dataset):
    def __init__(self, sequences, targets, masks):
        self.sequences = sequences.reshape(-1, 500, 4).astype(np.float32)
        self.targets = targets.astype(np.float32)
        self.masks = masks.astype(np.float32)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        x = torch.tensor(self.sequences[idx], dtype=torch.float32)
        y = torch.tensor(self.targets[idx], dtype=torch.float32)
        m = torch.tensor(self.masks[idx], dtype=torch.float32)
        return x, y, m

In [7]:
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, pool=True):
        super().__init__()
        self.pool = pool
        self.same_channels = (in_channels == out_channels)

        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, padding=kernel_size // 2)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU()

        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=kernel_size, padding=kernel_size // 2)
        self.bn2 = nn.BatchNorm1d(out_channels)

        if not self.same_channels:
            self.residual = nn.Conv1d(in_channels, out_channels, kernel_size=1)
        else:
            self.residual = nn.Identity()

        self.pool_layer = nn.MaxPool1d(2) if pool else nn.Identity()

    def forward(self, x):
        identity = self.residual(x)

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        out = self.relu(out)
        out = self.pool_layer(out)
        return out


class CNNMultiLabelResidual(nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.encoder = nn.Sequential(
            ResidualBlock1D(4, 64, kernel_size=7, pool=True),     # [B, 64, 250]
            ResidualBlock1D(64, 128, kernel_size=7, pool=True),   # [B, 128, 125]
            ResidualBlock1D(128, 256, kernel_size=5, pool=True),  # [B, 256, ~62]
        )

        self.classifier = nn.Sequential(
            nn.Linear(256 * 62, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, num_labels)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)  # [B, 4, 500]
        x = self.encoder(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


In [8]:
def optimize_thresholds(y_true, y_probs, thresholds=np.linspace(0.1, 0.9, 9)):
    best_thresholds = []

    for i in range(y_true.shape[1]):
        best_f1 = 0
        best_t = 0.5

        for t in thresholds:
            preds = (y_probs[:, i] >= t).astype(int)
            f1 = f1_score(y_true[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        best_thresholds.append(best_t)

    return np.array(best_thresholds)

In [9]:
def get_sorted_by_auc(aurocs, f1s, f1s_05, thresholds, confusions, avg_output_sums):
  # Get valid AUROC scores and sort
    indexed_aurocs = [(i, auc, f1, f1_05, threshold, confusion, avg_output_sum) for i, (auc, f1, f1_05, threshold, confusion, avg_output_sum) in enumerate(zip(aurocs, f1s, f1s_05, thresholds, confusions, avg_output_sums)) if not np.isnan(auc)]
    sorted_by_auc = sorted(indexed_aurocs, key=lambda x: x[1], reverse=True)

    return sorted_by_auc

In [10]:
def masked_bce_with_soft_neg(outputs, targets, masks, lambda_unknown=1.0, lambda_pos=15.0, lambda_neg=1.0):
    """
    - outputs: raw logits from the model [B, L]
    - targets: true labels (1 or 0) [B, L]
    - masks: 1 for known labels, 0 for unknowns [B, L]
    """
    # BCE loss per element
    bce = F.binary_cross_entropy_with_logits(outputs, targets, reduction='none')

    pos_weight = (targets == 1).float() * lambda_pos + (targets == 0).float() * lambda_neg

    masked_loss = (pos_weight * bce * masks).sum() / masks.sum()

    # Encourage model to output low scores for unknown labels
    unknown_mask = (1 - masks).float()
    zero_targets = torch.zeros_like(targets)
    unknown_penalty = F.binary_cross_entropy_with_logits(outputs, zero_targets, reduction='none')
    unknown_penalty = (unknown_penalty * unknown_mask).sum() / unknown_mask.sum()


    return masked_loss + lambda_unknown * unknown_penalty


In [11]:
def evaluate_masked(model, dataloader, device):
    model.eval()
    all_preds, all_targets, all_masks = [], [], []

    with torch.no_grad():
        for batch_x, batch_y, batch_mask in dataloader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            batch_mask = batch_mask.to(device)

            outputs = model(batch_x)
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_preds.append(probs)
            all_targets.append(batch_y.cpu().numpy())
            all_masks.append(batch_mask.cpu().numpy())

    y_true = np.vstack(all_targets)
    y_pred = np.vstack(all_preds)
    y_mask = np.vstack(all_masks)

    n_labels = y_true.shape[1]
    aurocs, f1s, f1s_05, thresholds, confusions = [], [], [], [], []
    avg_output_sums = []

    # Precompute per-sample sums across all labels once
    per_sample_sum = y_pred.sum(axis=1)  # [N]

    for i in range(n_labels):
        valid = y_mask[:, i].astype(bool)
        if valid.sum() == 0:
            aurocs.append(np.nan)
            f1s.append(np.nan)
            f1s_05.append(np.nan)
            thresholds.append(0.5)
            confusions.append((0, 0, 0, 0))
            avg_output_sums.append(np.nan)  # no valid samples for this label
            continue

        avg_output_sums.append(float(per_sample_sum[valid].mean()))

        y_t, y_p = y_true[valid, i], y_pred[valid, i]

        try:
            aurocs.append(roc_auc_score(y_t, y_p))
            precision, recall, thresh = precision_recall_curve(y_t, y_p)
            f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
            best_idx = np.nanargmax(f1_scores)
            best_thresh = thresh[best_idx] if best_idx < len(thresh) else 0.5
            thresholds.append(best_thresh)

            y_pred_bin = (y_p >= best_thresh).astype(int)
            f1s.append(f1_score(y_t, y_pred_bin))

            y_pred_bin_05 = (y_p >= 0.5).astype(int)
            f1s_05.append(f1_score(y_t, y_pred_bin_05))


            TP = np.sum((y_t == 1) & (y_pred_bin == 1))
            FP = np.sum((y_t == 0) & (y_pred_bin == 1))
            FN = np.sum((y_t == 1) & (y_pred_bin == 0))
            TN = np.sum((y_t == 0) & (y_pred_bin == 0))
            confusions.append((TP, FP, FN, TN))
        except ValueError:
            aurocs.append(np.nan)
            f1s.append(np.nan)
            f1s_05.append(np.nan)
            thresholds.append(0.5)
            confusions.append((0, 0, 0, 0))

    mAP = average_precision_score(
        y_true[y_mask.astype(bool)],
        y_pred[y_mask.astype(bool)],
        average="macro"
    )

    return aurocs, f1s, f1s_05, mAP, thresholds, confusions, y_true, y_pred, y_mask, avg_output_sums


In [12]:
def train(model, dataloader, criterion, optimizer, device, lambda_unknown=1.0, lambda_pos=15.0, lambda_neg=1.0):
    model.train()
    total_loss = 0

    for batch_x, batch_y, batch_mask in dataloader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        batch_mask = batch_mask.to(device)

        optimizer.zero_grad()
        outputs = model(batch_x)

        # Apply mask before loss
        loss = masked_bce_with_soft_neg(outputs, batch_y, batch_mask, lambda_unknown, lambda_pos, lambda_neg)

        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch_x.size(0)

    return total_loss / len(dataloader.dataset)

In [13]:
def plot_label_histogram_mask(label_id, y_true, y_pred, mask, threshold, title, scale=""):
    # Extract relevant values for this label
    y_t = y_true[:, label_id]
    y_p = y_pred[:, label_id]
    m = mask[:, label_id]

    # Filter only known labels
    known = m == 1
    y_t = y_t[known]
    y_p = y_p[known]

    pos_preds = y_p[y_t == 1]
    neg_preds = y_p[y_t == 0]

    plt.figure(figsize=(6, 4))
    plt.hist(neg_preds, bins=30, alpha=0.6, label="Negative", color="blue")
    plt.hist(pos_preds, bins=30, alpha=0.6, label="Positive", color="orange")

    plt.axvline(x=threshold, color='red', linestyle='--', label=f"Threshold = {threshold:.2f}")

    plt.title(f"{title} (Label {label_id+1})")
    plt.xlabel("Predicted probability")
    plt.ylabel("Count (log)")
    if scale == "log":
      plt.yscale('log')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [14]:
import matplotlib.pyplot as plt
import numpy as np

def plot_all_label_histogram_masked(y_true, y_pred, mask, thresholds=None, bins=50, scale="log"):
    """
    Aggregate histogram over ALL labels (masked) with a custom overlap color.
    """
    # Keep only known entries
    valid = (mask == 1)
    if not np.any(valid):
        raise ValueError("No known entries (mask==1).")

    y_t_known = y_true[valid].astype(np.int32)
    y_p_known = y_pred[valid].astype(np.float32)

    pos_preds = y_p_known[y_t_known == 1]
    neg_preds = y_p_known[y_t_known == 0]

    # --- Colors ---
    neg_color = "#CB9BCC"
    pos_color = "#CA0080"
    overlap_color = "#1f78b4"

    plt.figure(figsize=(7, 5), dpi=300)

    # Calculate common bin edges
    all_preds = np.concatenate([neg_preds, pos_preds])
    hist_range = (0.0, 1.0) # Probabilities are 0-1

    # Get bin edges first
    _, bin_edges = np.histogram(all_preds, bins=bins, range=hist_range)

    # Compute counts
    neg_counts, _ = np.histogram(neg_preds, bins=bin_edges)
    pos_counts, _ = np.histogram(pos_preds, bins=bin_edges)

    # The 'overlap' is the intersection (minimum height)
    intersect_counts = np.minimum(neg_counts, pos_counts)

    # The 'excess' is the part sticking out above the overlap
    excess_neg = neg_counts - intersect_counts
    excess_pos = pos_counts - intersect_counts

    # Bin properties for plotting
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    bin_width = bin_edges[1] - bin_edges[0]

    # Plot the Overlap base first
    plt.bar(bin_centers, intersect_counts, width=bin_width,
            color=overlap_color, alpha=0.6, align='center')

    # Plot Negative Excess (stacks on top of intersection)
    plt.bar(bin_centers, excess_neg, width=bin_width, bottom=intersect_counts,
            color=neg_color, alpha=0.6, align='center',
            label=f"Negative (n={len(neg_preds)})")

    # Plot Positive Excess (stacks on top of intersection)
    plt.bar(bin_centers, excess_pos, width=bin_width, bottom=intersect_counts,
            color=pos_color, alpha=0.6, align='center',
            label=f"Positive (n={len(pos_preds)})")

    # Threshold handling
    # if thresholds is not None:
    #     if np.isscalar(thresholds):
    #         thr = float(thresholds)
    #         plt.axvline(x=thr, color='red', linestyle='--', label=f"Threshold = {thr:.2f}")
    #     else:
    #         thr_arr = np.asarray(thresholds, dtype=float)
    #         thr_med = np.nanmedian(thr_arr)
    #         thr_mean = np.nanmean(thr_arr)
    #         plt.axvline(x=thr_med, color='red', linestyle='--', label=f"Median thr = {thr_med:.2f}")
    #         plt.axvline(x=thr_mean, color='green', linestyle='--', label=f"Mean thr = {thr_mean:.2f}")


    plt.xlabel("Predicted probability")
    plt.ylabel("Count" + (" (log)" if scale == "log" else ""))

    if scale == "log":
        plt.yscale("log")
        plt.ylim(bottom=0.5)

    plt.grid(True, alpha=0.3)

    plt.legend()
    plt.tight_layout()
    plt.show()

In [15]:
def plot_label_histogram(label_id, y_true, y_pred, threshold, title):
    pos_preds = y_pred[y_true == 1]
    neg_preds = y_pred[y_true == 0]

    plt.figure(figsize=(6, 4))
    plt.hist(neg_preds, bins=30, alpha=0.6, label="Negative", color="blue")
    plt.hist(pos_preds, bins=30, alpha=0.6, label="Positive", color="orange")

    # Plot threshold as vertical line
    plt.axvline(x=threshold, color='red', linestyle='--', label=f"Threshold = {threshold:.2f}")

    plt.title(f"{title} (Label {label_id})")
    plt.xlabel("Predicted probability")
    plt.ylabel("Count (log)")
    plt.yscale('log')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [16]:
Class_mapping = {'AATF': 1, 'NEG_AATF': -1, 'ADAT1': 2, 'NEG_ADAT1': -2, 'AGGF1': 3, 'NEG_AGGF1': -3, 'AKAP1': 4, 'NEG_AKAP1': -4, 'AKAP8L': 5, 'NEG_AKAP8L': -5,
                 'APOBEC3C': 6, 'NEG_APOBEC3C': -6, 'AQR': 7, 'NEG_AQR': -7, 'BCCIP': 8, 'NEG_BCCIP': -8, 'BCLAF1': 9, 'NEG_BCLAF1': -9, 'BUD13': 10, 'NEG_BUD13': -10,
                 'CDC40': 11, 'NEG_CDC40': -11, 'CPSF6': 12, 'NEG_CPSF6': -12, 'CSTF2': 13, 'NEG_CSTF2': -13, 'CSTF2T': 14, 'NEG_CSTF2T': -14, 'DDX1': 15, 'NEG_DDX1': -15,
                 'DDX24': 16, 'NEG_DDX24': -16, 'DDX3X': 17, 'NEG_DDX3X': -17, 'DDX43': 18, 'NEG_DDX43': -18, 'DDX52': 19, 'NEG_DDX52': -19, 'DDX55': 20, 'NEG_DDX55': -20,
                 'DDX59': 21, 'NEG_DDX59': -21, 'DDX6': 22, 'NEG_DDX6': -22, 'DGCR8': 23, 'NEG_DGCR8': -23, 'DHX30': 24, 'NEG_DHX30': -24, 'DKC1': 25, 'NEG_DKC1': -25,
                 'DROSHA': 26, 'NEG_DROSHA': -26, 'EFTUD2': 27, 'NEG_EFTUD2': -27, 'EIF3D': 28, 'NEG_EIF3D': -28, 'EIF3H': 29, 'NEG_EIF3H': -29, 'EIF4E': 30, 'NEG_EIF4E': -30,
                 'EIF4G2': 31, 'NEG_EIF4G2': -31, 'ELAC2': 32, 'NEG_ELAC2': -32, 'ELAVL1': 33, 'NEG_ELAVL1': -33, 'EWSR1': 34, 'NEG_EWSR1': -34, 'EXOSC10': 35, 'NEG_EXOSC10': -35,
                 'EXOSC5': 36, 'NEG_EXOSC5': -36, 'FAM120A': 37, 'NEG_FAM120A': -37, 'FASTKD2': 38, 'NEG_FASTKD2': -38, 'FKBP4': 39, 'NEG_FKBP4': -39, 'FMR1': 40, 'NEG_FMR1': -40,
                 'FTO': 41, 'NEG_FTO': -41, 'FUBP3': 42, 'NEG_FUBP3': -42, 'FUS': 43, 'NEG_FUS': -43, 'FXR1': 44, 'NEG_FXR1': -44, 'FXR2': 45, 'NEG_FXR2': -45,
                 'G3BP1': 46, 'NEG_G3BP1': -46, 'GARS': 47, 'NEG_GARS': -47, 'GEMIN5': 48, 'NEG_GEMIN5': -48, 'GPKOW': 49, 'NEG_GPKOW': -49, 'GRSF1': 50, 'NEG_GRSF1': -50,
                 'GRWD1': 51, 'NEG_GRWD1': -51, 'GTF2F1': 52, 'NEG_GTF2F1': -52, 'HNRNPC': 53, 'NEG_HNRNPC': -53, 'HNRNPK': 54, 'NEG_HNRNPK': -54, 'HNRNPL': 55, 'NEG_HNRNPL': -55,
                 'HNRNPM': 56, 'NEG_HNRNPM': -56, 'IGF2BP1': 57, 'NEG_IGF2BP1': -57, 'IGF2BP2': 58, 'NEG_IGF2BP2': -58, 'IGF2BP3': 59, 'NEG_IGF2BP3': -59, 'ILF3': 60, 'NEG_ILF3': -60,
                 'KHDRBS1': 61, 'NEG_KHDRBS1': -61, 'KHSRP': 62, 'NEG_KHSRP': -62, 'LIN28B': 63, 'NEG_LIN28B': -63, 'LSM11': 64, 'NEG_LSM11': -64, 'MATR3': 65, 'NEG_MATR3': -65,
                 'MBNL1': 66, 'NEG_MBNL1': -66, 'METAP2': 67, 'NEG_METAP2': -67, 'MORC2': 68, 'NEG_MORC2': -68, 'MTPAP': 69, 'NEG_MTPAP': -69, 'NKRF': 70, 'NEG_NKRF': -70,
                 'NOLC1': 71, 'NEG_NOLC1': -71, 'NONO': 72, 'NEG_NONO': -72, 'PABPC4': 73, 'NEG_PABPC4': -73, 'PABPN1': 74, 'NEG_PABPN1': -74, 'PCBP1': 75, 'NEG_PCBP1': -75,
                 'PCBP2': 76, 'NEG_PCBP2': -76, 'PHF6': 77, 'NEG_PHF6': -77, 'PPIG': 78, 'NEG_PPIG': -78, 'PPIL4': 79, 'NEG_PPIL4': -79, 'PRPF4': 80, 'NEG_PRPF4': -80,
                 'PRPF8': 81, 'NEG_PRPF8': -81, 'PTBP1': 82, 'NEG_PTBP1': -82, 'PUM1': 83, 'NEG_PUM1': -83, 'PUM2': 84, 'NEG_PUM2': -84, 'QKI': 85, 'NEG_QKI': -85,
                 'RBFOX2': 86, 'NEG_RBFOX2': -86, 'RBM15': 87, 'NEG_RBM15': -87, 'RBM22': 88, 'NEG_RBM22': -88, 'RBM5': 89, 'NEG_RBM5': -89, 'RNF187': 90, 'NEG_RNF187': -90,
                 'RPS3': 91, 'NEG_RPS3': -91, 'RYBP': 92, 'NEG_RYBP': -92, 'SAFB': 93, 'NEG_SAFB': -93, 'SAFB2': 94, 'NEG_SAFB2': -94, 'SF3A3': 95, 'NEG_SF3A3': -95,
                 'SF3B4': 96, 'NEG_SF3B4': -96, 'SFPQ': 97, 'NEG_SFPQ': -97, 'SLTM': 98, 'NEG_SLTM': -98, 'SMNDC1': 99, 'NEG_SMNDC1': -99, 'SND1': 100, 'NEG_SND1': -100,
                 'SRSF1': 101, 'NEG_SRSF1': -101, 'SRSF7': 102, 'NEG_SRSF7': -102, 'STAU2': 103, 'NEG_STAU2': -103, 'SUB1': 104, 'NEG_SUB1': -104, 'SUGP2': 105, 'NEG_SUGP2': -105,
                 'TAF15': 106, 'NEG_TAF15': -106, 'TARDBP': 107, 'NEG_TARDBP': -107, 'TIA1': 108, 'NEG_TIA1': -108, 'TIAL1': 109, 'NEG_TIAL1': -109, 'TRA2A': 110, 'NEG_TRA2A': -110,
                 'U2AF1': 111, 'NEG_U2AF1': -111, 'U2AF2': 112, 'NEG_U2AF2': -112, 'UCHL5': 113, 'NEG_UCHL5': -113, 'UPF1': 114, 'NEG_UPF1': -114, 'XPO5': 115, 'NEG_XPO5': -115,
                 'XRCC6': 116, 'NEG_XRCC6': -116, 'XRN2': 117, 'NEG_XRN2': -117, 'YBX3': 118, 'NEG_YBX3': -118, 'YWHAG': 119, 'NEG_YWHAG': -119, 'ZC3H11A': 120, 'NEG_ZC3H11A': -120,
                 'ZNF622': 121, 'NEG_ZNF622': -121, 'ZNF800': 122, 'NEG_ZNF800': -122}

Index_mapping = {1: 'AATF', -1: 'NEG_AATF', 2: 'ADAT1', -2: 'NEG_ADAT1', 3: 'AGGF1', -3: 'NEG_AGGF1', 4: 'AKAP1', -4: 'NEG_AKAP1', 5: 'AKAP8L', -5: 'NEG_AKAP8L',
                 6: 'APOBEC3C', -6: 'NEG_APOBEC3C', 7: 'AQR', -7: 'NEG_AQR', 8: 'BCCIP', -8: 'NEG_BCCIP', 9: 'BCLAF1', -9: 'NEG_BCLAF1', 10: 'BUD13', -10: 'NEG_BUD13',
                 11: 'CDC40', -11: 'NEG_CDC40', 12: 'CPSF6', -12: 'NEG_CPSF6', 13: 'CSTF2', -13: 'NEG_CSTF2', 14: 'CSTF2T', -14: 'NEG_CSTF2T', 15: 'DDX1', -15: 'NEG_DDX1',
                 16: 'DDX24', -16: 'NEG_DDX24', 17: 'DDX3X', -17: 'NEG_DDX3X', 18: 'DDX43', -18: 'NEG_DDX43', 19: 'DDX52', -19: 'NEG_DDX52', 20: 'DDX55', -20: 'NEG_DDX55',
                 21: 'DDX59', -21: 'NEG_DDX59', 22: 'DDX6', -22: 'NEG_DDX6', 23: 'DGCR8', -23: 'NEG_DGCR8', 24: 'DHX30', -24: 'NEG_DHX30', 25: 'DKC1', -25: 'NEG_DKC1',
                 26: 'DROSHA', -26: 'NEG_DROSHA', 27: 'EFTUD2', -27: 'NEG_EFTUD2', 28: 'EIF3D', -28: 'NEG_EIF3D', 29: 'EIF3H', -29: 'NEG_EIF3H', 30: 'EIF4E', -30: 'NEG_EIF4E',
                 31: 'EIF4G2', -31: 'NEG_EIF4G2', 32: 'ELAC2', -32: 'NEG_ELAC2', 33: 'ELAVL1', -33: 'NEG_ELAVL1', 34: 'EWSR1', -34: 'NEG_EWSR1', 35: 'EXOSC10', -35: 'NEG_EXOSC10',
                 36: 'EXOSC5', -36: 'NEG_EXOSC5', 37: 'FAM120A', -37: 'NEG_FAM120A', 38: 'FASTKD2', -38: 'NEG_FASTKD2', 39: 'FKBP4', -39: 'NEG_FKBP4', 40: 'FMR1', -40: 'NEG_FMR1',
                 41: 'FTO', -41: 'NEG_FTO', 42: 'FUBP3', -42: 'NEG_FUBP3', 43: 'FUS', -43: 'NEG_FUS', 44: 'FXR1', -44: 'NEG_FXR1', 45: 'FXR2', -45: 'NEG_FXR2',
                 46: 'G3BP1', -46: 'NEG_G3BP1', 47: 'GARS', -47: 'NEG_GARS', 48: 'GEMIN5', -48: 'NEG_GEMIN5', 49: 'GPKOW', -49: 'NEG_GPKOW', 50: 'GRSF1', -50: 'NEG_GRSF1',
                 51: 'GRWD1', -51: 'NEG_GRWD1', 52: 'GTF2F1', -52: 'NEG_GTF2F1', 53: 'HNRNPC', -53: 'NEG_HNRNPC', 54: 'HNRNPK', -54: 'NEG_HNRNPK', 55: 'HNRNPL', -55: 'NEG_HNRNPL',
                 56: 'HNRNPM', -56: 'NEG_HNRNPM', 57: 'IGF2BP1', -57: 'NEG_IGF2BP1', 58: 'IGF2BP2', -58: 'NEG_IGF2BP2', 59: 'IGF2BP3', -59: 'NEG_IGF2BP3', 60: 'ILF3', -60: 'NEG_ILF3',
                 61: 'KHDRBS1', -61: 'NEG_KHDRBS1', 62: 'KHSRP', -62: 'NEG_KHSRP', 63: 'LIN28B', -63: 'NEG_LIN28B', 64: 'LSM11', -64: 'NEG_LSM11', 65: 'MATR3', -65: 'NEG_MATR3',
                 66: 'MBNL1', -66: 'NEG_MBNL1', 67: 'METAP2', -67: 'NEG_METAP2', 68: 'MORC2', -68: 'NEG_MORC2', 69: 'MTPAP', -69: 'NEG_MTPAP', 70: 'NKRF', -70: 'NEG_NKRF',
                 71: 'NOLC1', -71: 'NEG_NOLC1', 72: 'NONO', -72: 'NEG_NONO', 73: 'PABPC4', -73: 'NEG_PABPC4', 74: 'PABPN1', -74: 'NEG_PABPN1', 75: 'PCBP1', -75: 'NEG_PCBP1',
                 76: 'PCBP2', -76: 'NEG_PCBP2', 77: 'PHF6', -77: 'NEG_PHF6', 78: 'PPIG', -78: 'NEG_PPIG', 79: 'PPIL4', -79: 'NEG_PPIL4', 80: 'PRPF4', -80: 'NEG_PRPF4',
                 81: 'PRPF8', -81: 'NEG_PRPF8', 82: 'PTBP1', -82: 'NEG_PTBP1', 83: 'PUM1', -83: 'NEG_PUM1', 84: 'PUM2', -84: 'NEG_PUM2', 85: 'QKI', -85: 'NEG_QKI',
                 86: 'RBFOX2', -86: 'NEG_RBFOX2', 87: 'RBM15', -87: 'NEG_RBM15', 88: 'RBM22', -88: 'NEG_RBM22', 89: 'RBM5', -89: 'NEG_RBM5', 90: 'RNF187', -90: 'NEG_RNF187',
                 91: 'RPS3', -91: 'NEG_RPS3', 92: 'RYBP', -92: 'NEG_RYBP', 93: 'SAFB', -93: 'NEG_SAFB', 94: 'SAFB2', -94: 'NEG_SAFB2', 95: 'SF3A3', -95: 'NEG_SF3A3',
                 96: 'SF3B4', -96: 'NEG_SF3B4', 97: 'SFPQ', -97: 'NEG_SFPQ', 98: 'SLTM', -98: 'NEG_SLTM', 99: 'SMNDC1', -99: 'NEG_SMNDC1', 100: 'SND1', -100: 'NEG_SND1',
                 101: 'SRSF1', -101: 'NEG_SRSF1', 102: 'SRSF7', -102: 'NEG_SRSF7', 103: 'STAU2', -103: 'NEG_STAU2', 104: 'SUB1', -104: 'NEG_SUB1', 105: 'SUGP2', -105: 'NEG_SUGP2',
                 106: 'TAF15', -106: 'NEG_TAF15', 107: 'TARDBP', -107: 'NEG_TARDBP', 108: 'TIA1', -108: 'NEG_TIA1', 109: 'TIAL1', -109: 'NEG_TIAL1', 110: 'TRA2A', -110: 'NEG_TRA2A',
                 111: 'U2AF1', -111: 'NEG_U2AF1', 112: 'U2AF2', -112: 'NEG_U2AF2', 113: 'UCHL5', -113: 'NEG_UCHL5', 114: 'UPF1', -114: 'NEG_UPF1', 115: 'XPO5', -115: 'NEG_XPO5',
                 116: 'XRCC6', -116: 'NEG_XRCC6', 117: 'XRN2', -117: 'NEG_XRN2', 118: 'YBX3', -118: 'NEG_YBX3', 119: 'YWHAG', -119: 'NEG_YWHAG', 120: 'ZC3H11A', -120: 'NEG_ZC3H11A',
                 121: 'ZNF622', -121: 'NEG_ZNF622', 122: 'ZNF800', -122: 'NEG_ZNF800'}

Character_mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
Index_Character_mapping = {0: 'A', 1: 'C', 2: 'G', 3: 'T'}

In [17]:
good_purple = plt.get_cmap('PuRd')(0.85)

# Training

## dataset loading

In [ ]:
csv_path = "/content/drive/MyDrive/intRBP/dataset_K562_multilabel_with_NEGs.csv"
print("Loading CSV and preprocessing...")
sequences, targets, masks, label_counts = load_masked_multilabel_dataset(csv_path)
print(f"Loaded {len(sequences)} sequences.")

print("Creating dataset object...")
dataset = MaskedMultilabelDataset(sequences, targets, masks)

## Define train parameters

In [19]:
num_epochs = 30
batch_size = 256
lr = 1e-3

lambda_unknown = 1
lambda_neg = 1
lambda_pos = 15

## Main train

In [ ]:
start_training = time.time()

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

strat_labels = targets * masks  # use only known labels for stratification

train_idx, val_idx = next(msss.split(sequences, strat_labels))

print(f"\n--- Training Split (80/20) ---")
print(f"Train size: {len(train_idx)}, Val size: {len(val_idx)}")

# Dataset Setup
train_loader = DataLoader(Subset(dataset, train_idx), batch_size=batch_size, num_workers=2, shuffle=True)
val_loader = DataLoader(Subset(dataset, val_idx), batch_size=batch_size, num_workers=2)

print("Initializing model, loss, and optimizer...")
model = CNNMultiLabelResidual(num_labels=targets.shape[1]).to(device)
criterion = nn.BCEWithLogitsLoss(reduction='none')
optimizer = optim.Adam(model.parameters(), lr=lr)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

# Training Loop
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    start_epoch = time.time()

    train_loss = train(model, train_loader, criterion, optimizer, device, lambda_unknown=lambda_unknown, lambda_pos=lambda_pos, lambda_neg=lambda_neg)
    train_time = time.time()

    # Validation
    aurocs, f1s, f1s_05, mAP, thresholds, confusions, y_true, y_pred, y_mask, avg_output_sums = evaluate_masked(model, val_loader, device)
    end_epoch = time.time()

    avg_auroc = np.nanmean(aurocs)
    avg_sum = np.nanmean(avg_output_sums)
    sorted_by_auc = get_sorted_by_auc(aurocs, f1s, f1s_05, thresholds, confusions, avg_output_sums)
    avg_f1_05 = np.nanmean(f1s_05)
    avg_f1 = np.nanmean(f1s)

    scheduler.step(avg_auroc)

    print(f"Loss: {train_loss:.4f} | AUROC: {avg_auroc:.4f} | F1: {avg_f1:.4f} | F1(0.5): {avg_f1_05:.4f} | mAP: {mAP:.4f} | AvgSum: {np.mean(avg_sum):.4f} | Train time: {train_time - start_epoch:.2f}s | Eval time: {end_epoch - train_time:.2f}s")
    print("Average threshold:", np.mean(thresholds))
    print("Thresholds (min/max):", np.min(thresholds), np.max(thresholds))

    print("[INFO] AUROC Top 10 proteins:")
    for idx, auc, f1, f1_05, threshold, confusion, avg_sum in sorted_by_auc[:10]:
        tp, fp, fn, tn = confusion
        print(f"Protein {Index_mapping[idx]} ({idx}): AUROC = {auc:.4f} | F1: {f1:.4f} | F1(0.5): {f1_05:.4f} | Threshold: {threshold:.4f} | TP: {tp:4} | FP: {fp:4} | FN: {fn:4} | TN: {tn:4} | AvgSum: {avg_sum:.2f}")

    print("[INFO] AUROC Bottom 10 proteins:")
    for idx, auc, f1, f1_05, threshold, confusion, avg_sum in sorted_by_auc[-10:]:
        tp, fp, fn, tn = confusion
        print(f"Protein {Index_mapping[idx]} ({idx}): AUROC = {auc:.4f} | F1: {f1:.4f} | F1(0.5): {f1_05:.4f} | Threshold: {threshold:.4f} | TP: {tp:4} | FP: {fp:4} | FN: {fn:4} | TN: {tn:4} | AvgSum: {avg_sum:.2f}")

end_training = time.time()
print(f"\nTraining complete in {end_training - start_training:.2f} seconds.")

# Save Logic
split_dict = { "train_idx": train_idx.tolist(), "val_idx": val_idx.tolist() }

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_name = model.__class__.__name__
model_save_path = f"/content/drive/MyDrive/intRBP/Models/{model_name}_{timestamp}.pth"
split_save_path = f"/content/drive/MyDrive/intRBP/Models/data_split_{timestamp}.json"

torch.save(model.state_dict(), model_save_path)
with open(split_save_path, "w") as f:
    json.dump(split_dict, f)

print(f"Model saved to {model_save_path}")
print(f"Saved train/val split indices to: {split_save_path}")

## Training Performance

### Log

In [21]:
log_data = """

--- Fold 1/5 ---
Train size: 288944, Val size: 72236
Initializing model, loss, and optimizer...

Epoch 1/30
Loss: 3.1640 | AUROC: 0.8666 | F1: 0.8306 | F1(0.5): 0.7882 | mAP: 0.8880 | AvgSum: 69.7001 | Train time: 93.43s | Eval time: 8.45s
Average threshold: 0.72434556
Thresholds (min/max): 0.23377995 0.9487494
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9922 | F1: 0.9721 | F1(0.5): 0.9540 | Threshold: 0.7798 | TP:  383 | FP:    5 | FN:   17 | TN:  322 | AvgSum: 64.98
Protein MTPAP (69): AUROC = 0.9895 | F1: 0.9657 | F1(0.5): 0.9646 | Threshold: 0.5280 | TP:  394 | FP:   22 | FN:    6 | TN:  322 | AvgSum: 71.68
Protein AKAP1 (4): AUROC = 0.9877 | F1: 0.9629 | F1(0.5): 0.9595 | Threshold: 0.5767 | TP:  389 | FP:   19 | FN:   11 | TN:  313 | AvgSum: 68.77
Protein GARS (47): AUROC = 0.9871 | F1: 0.9558 | F1(0.5): 0.9476 | Threshold: 0.6679 | TP:  378 | FP:   13 | FN:   22 | TN:  329 | AvgSum: 66.96
Protein FASTKD2 (38): AUROC = 0.9871 | F1: 0.9567 | F1(0.5): 0.9551 | Threshold: 0.5317 | TP:  287 | FP:   21 | FN:    5 | TN:  262 | AvgSum: 74.97
Protein ADAT1 (2): AUROC = 0.9861 | F1: 0.9509 | F1(0.5): 0.9495 | Threshold: 0.4826 | TP:  378 | FP:   17 | FN:   22 | TN:  306 | AvgSum: 63.71
Protein DGCR8 (23): AUROC = 0.9766 | F1: 0.9388 | F1(0.5): 0.8925 | Threshold: 0.7180 | TP:  184 | FP:   15 | FN:    9 | TN:  168 | AvgSum: 71.13
Protein GRWD1 (51): AUROC = 0.9742 | F1: 0.9370 | F1(0.5): 0.9254 | Threshold: 0.6792 | TP:  379 | FP:   30 | FN:   21 | TN:  307 | AvgSum: 71.87
Protein NOLC1 (71): AUROC = 0.9720 | F1: 0.9310 | F1(0.5): 0.9236 | Threshold: 0.6092 | TP:  391 | FP:   51 | FN:    7 | TN:  277 | AvgSum: 74.89
Protein EIF3H (29): AUROC = 0.9684 | F1: 0.9370 | F1(0.5): 0.8891 | Threshold: 0.7342 | TP:  372 | FP:   22 | FN:   28 | TN:  297 | AvgSum: 73.03
[INFO] AUROC Bottom 10 proteins:
Protein RPS3 (91): AUROC = 0.7379 | F1: 0.7000 | F1(0.5): 0.6575 | Threshold: 0.6487 | TP:  133 | FP:   96 | FN:   18 | TN:   70 | AvgSum: 67.36
Protein FXR2 (45): AUROC = 0.7301 | F1: 0.7650 | F1(0.5): 0.7310 | Threshold: 0.8497 | TP:  350 | FP:  165 | FN:   50 | TN:  154 | AvgSum: 72.12
Protein TIA1 (108): AUROC = 0.7283 | F1: 0.7339 | F1(0.5): 0.7056 | Threshold: 0.7583 | TP:  353 | FP:  209 | FN:   47 | TN:  125 | AvgSum: 66.11
Protein KHSRP (62): AUROC = 0.7270 | F1: 0.7453 | F1(0.5): 0.7205 | Threshold: 0.6748 | TP:  376 | FP:  233 | FN:   24 | TN:  103 | AvgSum: 69.56
Protein MATR3 (65): AUROC = 0.7223 | F1: 0.7195 | F1(0.5): 0.6900 | Threshold: 0.5776 | TP:  227 | FP:  161 | FN:   16 | TN:   71 | AvgSum: 69.54
Protein RYBP (92): AUROC = 0.7040 | F1: 0.6561 | F1(0.5): 0.6498 | Threshold: 0.5374 | TP:  103 | FP:  106 | FN:    2 | TN:   10 | AvgSum: 68.86
Protein AKAP8L (5): AUROC = 0.6757 | F1: 0.7029 | F1(0.5): 0.6858 | Threshold: 0.6069 | TP:  194 | FP:  153 | FN:   11 | TN:   46 | AvgSum: 71.49
Protein CPSF6 (12): AUROC = 0.6679 | F1: 0.7172 | F1(0.5): 0.7112 | Threshold: 0.4380 | TP:  388 | FP:  294 | FN:   12 | TN:   50 | AvgSum: 65.63
Protein IGF2BP1 (57): AUROC = 0.6476 | F1: 0.7467 | F1(0.5): 0.7446 | Threshold: 0.5163 | TP:  395 | FP:  263 | FN:    5 | TN:   31 | AvgSum: 74.52
Protein PPIG (78): AUROC = 0.5806 | F1: 0.6616 | F1(0.5): 0.6290 | Threshold: 0.2338 | TP:  130 | FP:  131 | FN:    2 | TN:    7 | AvgSum: 69.07

Epoch 2/30
Loss: 2.7266 | AUROC: 0.8863 | F1: 0.8448 | F1(0.5): 0.8091 | mAP: 0.9096 | AvgSum: 63.9009 | Train time: 92.68s | Eval time: 8.42s
Average threshold: 0.73793256
Thresholds (min/max): 0.4632825 0.9444481
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9953 | F1: 0.9765 | F1(0.5): 0.9636 | Threshold: 0.6528 | TP:  394 | FP:   13 | FN:    6 | TN:  314 | AvgSum: 56.76
Protein ADAT1 (2): AUROC = 0.9937 | F1: 0.9701 | F1(0.5): 0.9667 | Threshold: 0.5914 | TP:  390 | FP:   14 | FN:   10 | TN:  309 | AvgSum: 54.99
Protein AKAP1 (4): AUROC = 0.9934 | F1: 0.9739 | F1(0.5): 0.9622 | Threshold: 0.8313 | TP:  392 | FP:   13 | FN:    8 | TN:  319 | AvgSum: 62.38
Protein MTPAP (69): AUROC = 0.9890 | F1: 0.9712 | F1(0.5): 0.9556 | Threshold: 0.9246 | TP:  388 | FP:   11 | FN:   12 | TN:  333 | AvgSum: 65.39
Protein GARS (47): AUROC = 0.9874 | F1: 0.9614 | F1(0.5): 0.9588 | Threshold: 0.4633 | TP:  386 | FP:   17 | FN:   14 | TN:  325 | AvgSum: 61.29
Protein FASTKD2 (38): AUROC = 0.9859 | F1: 0.9536 | F1(0.5): 0.9097 | Threshold: 0.8714 | TP:  288 | FP:   24 | FN:    4 | TN:  259 | AvgSum: 70.37
Protein TAF15 (106): AUROC = 0.9845 | F1: 0.9412 | F1(0.5): 0.8894 | Threshold: 0.8193 | TP:  384 | FP:   32 | FN:   16 | TN:  307 | AvgSum: 64.76
Protein DKC1 (25): AUROC = 0.9783 | F1: 0.9367 | F1(0.5): 0.9248 | Threshold: 0.6409 | TP:  370 | FP:   20 | FN:   30 | TN:  304 | AvgSum: 61.56
Protein NOLC1 (71): AUROC = 0.9774 | F1: 0.9479 | F1(0.5): 0.9378 | Threshold: 0.8234 | TP:  382 | FP:   26 | FN:   16 | TN:  302 | AvgSum: 69.32
Protein EIF4E (30): AUROC = 0.9753 | F1: 0.9271 | F1(0.5): 0.8704 | Threshold: 0.8777 | TP:  369 | FP:   27 | FN:   31 | TN:  307 | AvgSum: 68.11
[INFO] AUROC Bottom 10 proteins:
Protein TIA1 (108): AUROC = 0.7640 | F1: 0.7541 | F1(0.5): 0.7173 | Threshold: 0.7817 | TP:  345 | FP:  170 | FN:   55 | TN:  164 | AvgSum: 59.23
Protein FXR2 (45): AUROC = 0.7637 | F1: 0.7662 | F1(0.5): 0.7527 | Threshold: 0.6262 | TP:  367 | FP:  191 | FN:   33 | TN:  128 | AvgSum: 68.20
Protein ILF3 (60): AUROC = 0.7617 | F1: 0.7430 | F1(0.5): 0.7045 | Threshold: 0.8117 | TP:  227 | FP:  139 | FN:   18 | TN:  109 | AvgSum: 47.35
Protein KHSRP (62): AUROC = 0.7617 | F1: 0.7537 | F1(0.5): 0.7279 | Threshold: 0.6929 | TP:  358 | FP:  192 | FN:   42 | TN:  144 | AvgSum: 63.84
Protein CPSF6 (12): AUROC = 0.7480 | F1: 0.7378 | F1(0.5): 0.7066 | Threshold: 0.7919 | TP:  339 | FP:  180 | FN:   61 | TN:  164 | AvgSum: 58.62
Protein G3BP1 (46): AUROC = 0.7267 | F1: 0.7266 | F1(0.5): 0.7055 | Threshold: 0.6163 | TP:   97 | FP:   54 | FN:   19 | TN:   53 | AvgSum: 68.34
Protein RYBP (92): AUROC = 0.7246 | F1: 0.6794 | F1(0.5): 0.6621 | Threshold: 0.6053 | TP:   89 | FP:   68 | FN:   16 | TN:   48 | AvgSum: 61.13
Protein AKAP8L (5): AUROC = 0.7127 | F1: 0.7169 | F1(0.5): 0.7169 | Threshold: 0.5018 | TP:  195 | FP:  144 | FN:   10 | TN:   55 | AvgSum: 67.43
Protein PPIG (78): AUROC = 0.6985 | F1: 0.6645 | F1(0.5): 0.6418 | Threshold: 0.6747 | TP:  100 | FP:   69 | FN:   32 | TN:   69 | AvgSum: 62.57
Protein IGF2BP1 (57): AUROC = 0.6707 | F1: 0.7541 | F1(0.5): 0.7502 | Threshold: 0.5962 | TP:  388 | FP:  241 | FN:   12 | TN:   53 | AvgSum: 70.35

Epoch 3/30
Loss: 2.5634 | AUROC: 0.8880 | F1: 0.8464 | F1(0.5): 0.7984 | mAP: 0.9076 | AvgSum: 66.9949 | Train time: 92.66s | Eval time: 8.36s
Average threshold: 0.7766466
Thresholds (min/max): 0.2368418 0.9251065
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9965 | F1: 0.9751 | F1(0.5): 0.9613 | Threshold: 0.7206 | TP:  391 | FP:   11 | FN:    9 | TN:  316 | AvgSum: 58.44
Protein ADAT1 (2): AUROC = 0.9944 | F1: 0.9736 | F1(0.5): 0.9678 | Threshold: 0.6393 | TP:  387 | FP:    8 | FN:   13 | TN:  315 | AvgSum: 57.88
Protein MTPAP (69): AUROC = 0.9925 | F1: 0.9707 | F1(0.5): 0.9567 | Threshold: 0.7097 | TP:  397 | FP:   21 | FN:    3 | TN:  323 | AvgSum: 68.23
Protein AKAP1 (4): AUROC = 0.9902 | F1: 0.9726 | F1(0.5): 0.9621 | Threshold: 0.7527 | TP:  390 | FP:   12 | FN:   10 | TN:  320 | AvgSum: 65.30
Protein GARS (47): AUROC = 0.9887 | F1: 0.9578 | F1(0.5): 0.9049 | Threshold: 0.7444 | TP:  386 | FP:   20 | FN:   14 | TN:  322 | AvgSum: 64.97
Protein FASTKD2 (38): AUROC = 0.9870 | F1: 0.9633 | F1(0.5): 0.9554 | Threshold: 0.6538 | TP:  289 | FP:   19 | FN:    3 | TN:  264 | AvgSum: 72.27
Protein TAF15 (106): AUROC = 0.9853 | F1: 0.9451 | F1(0.5): 0.8735 | Threshold: 0.9140 | TP:  379 | FP:   23 | FN:   21 | TN:  316 | AvgSum: 67.41
Protein DGCR8 (23): AUROC = 0.9808 | F1: 0.9293 | F1(0.5): 0.9019 | Threshold: 0.8473 | TP:  171 | FP:    4 | FN:   22 | TN:  179 | AvgSum: 68.11
Protein EIF4E (30): AUROC = 0.9806 | F1: 0.9441 | F1(0.5): 0.8922 | Threshold: 0.7945 | TP:  380 | FP:   25 | FN:   20 | TN:  309 | AvgSum: 69.19
Protein GRWD1 (51): AUROC = 0.9787 | F1: 0.9416 | F1(0.5): 0.9222 | Threshold: 0.8777 | TP:  371 | FP:   17 | FN:   29 | TN:  320 | AvgSum: 69.26
[INFO] AUROC Bottom 10 proteins:
Protein AKAP8L (5): AUROC = 0.7610 | F1: 0.7366 | F1(0.5): 0.6891 | Threshold: 0.7477 | TP:  193 | FP:  126 | FN:   12 | TN:   73 | AvgSum: 68.91
Protein KHSRP (62): AUROC = 0.7588 | F1: 0.7537 | F1(0.5): 0.7186 | Threshold: 0.8696 | TP:  332 | FP:  149 | FN:   68 | TN:  187 | AvgSum: 66.50
Protein IGF2BP2 (58): AUROC = 0.7559 | F1: 0.7711 | F1(0.5): 0.7428 | Threshold: 0.6701 | TP:  389 | FP:  220 | FN:   11 | TN:  104 | AvgSum: 72.00
Protein RYBP (92): AUROC = 0.7537 | F1: 0.6850 | F1(0.5): 0.6556 | Threshold: 0.4650 | TP:   87 | FP:   62 | FN:   18 | TN:   54 | AvgSum: 66.78
Protein G3BP1 (46): AUROC = 0.7531 | F1: 0.7361 | F1(0.5): 0.7214 | Threshold: 0.4614 | TP:  106 | FP:   66 | FN:   10 | TN:   41 | AvgSum: 69.80
Protein GPKOW (49): AUROC = 0.7494 | F1: 0.7222 | F1(0.5): 0.6995 | Threshold: 0.8021 | TP:  182 | FP:  109 | FN:   31 | TN:   94 | AvgSum: 75.43
Protein ILF3 (60): AUROC = 0.7471 | F1: 0.7382 | F1(0.5): 0.7118 | Threshold: 0.7082 | TP:  234 | FP:  155 | FN:   11 | TN:   93 | AvgSum: 54.81
Protein CPSF6 (12): AUROC = 0.7269 | F1: 0.7256 | F1(0.5): 0.7157 | Threshold: 0.7740 | TP:  345 | FP:  206 | FN:   55 | TN:  138 | AvgSum: 65.78
Protein PPIG (78): AUROC = 0.7184 | F1: 0.6931 | F1(0.5): 0.6711 | Threshold: 0.2368 | TP:  131 | FP:  115 | FN:    1 | TN:   23 | AvgSum: 66.83
Protein IGF2BP1 (57): AUROC = 0.7059 | F1: 0.7560 | F1(0.5): 0.7449 | Threshold: 0.7177 | TP:  381 | FP:  227 | FN:   19 | TN:   67 | AvgSum: 71.70

Epoch 4/30
Loss: 2.4197 | AUROC: 0.8934 | F1: 0.8499 | F1(0.5): 0.8075 | mAP: 0.9069 | AvgSum: 63.2055 | Train time: 92.26s | Eval time: 8.40s
Average threshold: 0.74861395
Thresholds (min/max): 0.3901128 0.93558955
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9969 | F1: 0.9799 | F1(0.5): 0.9739 | Threshold: 0.7402 | TP:  391 | FP:    7 | FN:    9 | TN:  320 | AvgSum: 54.32
Protein GARS (47): AUROC = 0.9906 | F1: 0.9600 | F1(0.5): 0.9145 | Threshold: 0.7493 | TP:  384 | FP:   16 | FN:   16 | TN:  326 | AvgSum: 61.03
Protein ADAT1 (2): AUROC = 0.9893 | F1: 0.9648 | F1(0.5): 0.9624 | Threshold: 0.5158 | TP:  384 | FP:   12 | FN:   16 | TN:  311 | AvgSum: 53.90
Protein FASTKD2 (38): AUROC = 0.9891 | F1: 0.9584 | F1(0.5): 0.9549 | Threshold: 0.4911 | TP:  288 | FP:   21 | FN:    4 | TN:  262 | AvgSum: 69.80
Protein MTPAP (69): AUROC = 0.9874 | F1: 0.9663 | F1(0.5): 0.9619 | Threshold: 0.6584 | TP:  387 | FP:   14 | FN:   13 | TN:  330 | AvgSum: 66.69
Protein AKAP1 (4): AUROC = 0.9864 | F1: 0.9627 | F1(0.5): 0.9604 | Threshold: 0.5619 | TP:  387 | FP:   17 | FN:   13 | TN:  315 | AvgSum: 62.92
Protein TAF15 (106): AUROC = 0.9834 | F1: 0.9431 | F1(0.5): 0.9227 | Threshold: 0.8712 | TP:  373 | FP:   18 | FN:   27 | TN:  321 | AvgSum: 62.91
Protein U2AF2 (112): AUROC = 0.9812 | F1: 0.9376 | F1(0.5): 0.8529 | Threshold: 0.8638 | TP:  383 | FP:   34 | FN:   17 | TN:  291 | AvgSum: 57.12
Protein DGCR8 (23): AUROC = 0.9788 | F1: 0.9239 | F1(0.5): 0.8863 | Threshold: 0.7437 | TP:  176 | FP:   12 | FN:   17 | TN:  171 | AvgSum: 64.89
Protein GRWD1 (51): AUROC = 0.9784 | F1: 0.9407 | F1(0.5): 0.9225 | Threshold: 0.6911 | TP:  381 | FP:   29 | FN:   19 | TN:  308 | AvgSum: 66.71
[INFO] AUROC Bottom 10 proteins:
Protein G3BP1 (46): AUROC = 0.7809 | F1: 0.7413 | F1(0.5): 0.7299 | Threshold: 0.5824 | TP:   96 | FP:   47 | FN:   20 | TN:   60 | AvgSum: 65.23
Protein KHSRP (62): AUROC = 0.7629 | F1: 0.7523 | F1(0.5): 0.7181 | Threshold: 0.8881 | TP:  331 | FP:  149 | FN:   69 | TN:  187 | AvgSum: 61.88
Protein IGF2BP2 (58): AUROC = 0.7629 | F1: 0.7716 | F1(0.5): 0.7294 | Threshold: 0.8243 | TP:  358 | FP:  170 | FN:   42 | TN:  154 | AvgSum: 67.55
Protein PPIG (78): AUROC = 0.7523 | F1: 0.7122 | F1(0.5): 0.6883 | Threshold: 0.3901 | TP:  120 | FP:   85 | FN:   12 | TN:   53 | AvgSum: 62.85
Protein AKAP8L (5): AUROC = 0.7518 | F1: 0.7322 | F1(0.5): 0.7002 | Threshold: 0.6813 | TP:  190 | FP:  124 | FN:   15 | TN:   75 | AvgSum: 64.96
Protein GPKOW (49): AUROC = 0.7494 | F1: 0.7380 | F1(0.5): 0.7180 | Threshold: 0.6844 | TP:  193 | FP:  117 | FN:   20 | TN:   86 | AvgSum: 71.73
Protein ILF3 (60): AUROC = 0.7487 | F1: 0.7250 | F1(0.5): 0.7196 | Threshold: 0.5606 | TP:  228 | FP:  156 | FN:   17 | TN:   92 | AvgSum: 50.72
Protein IGF2BP1 (57): AUROC = 0.7295 | F1: 0.7548 | F1(0.5): 0.7382 | Threshold: 0.8089 | TP:  371 | FP:  212 | FN:   29 | TN:   82 | AvgSum: 67.19
Protein CPSF6 (12): AUROC = 0.7139 | F1: 0.7245 | F1(0.5): 0.7144 | Threshold: 0.5752 | TP:  376 | FP:  262 | FN:   24 | TN:   82 | AvgSum: 61.82
Protein RYBP (92): AUROC = 0.7056 | F1: 0.6766 | F1(0.5): 0.6714 | Threshold: 0.5446 | TP:   91 | FP:   73 | FN:   14 | TN:   43 | AvgSum: 62.88

Epoch 5/30
Loss: 2.2707 | AUROC: 0.9067 | F1: 0.8632 | F1(0.5): 0.8304 | mAP: 0.9248 | AvgSum: 57.0378 | Train time: 92.34s | Eval time: 8.33s
Average threshold: 0.7465914
Thresholds (min/max): 0.373226 0.9201855
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9969 | F1: 0.9813 | F1(0.5): 0.9718 | Threshold: 0.7092 | TP:  393 | FP:    8 | FN:    7 | TN:  319 | AvgSum: 50.79
Protein ADAT1 (2): AUROC = 0.9962 | F1: 0.9737 | F1(0.5): 0.9692 | Threshold: 0.7593 | TP:  388 | FP:    9 | FN:   12 | TN:  314 | AvgSum: 48.57
Protein GARS (47): AUROC = 0.9929 | F1: 0.9725 | F1(0.5): 0.9667 | Threshold: 0.6186 | TP:  389 | FP:   11 | FN:   11 | TN:  331 | AvgSum: 54.59
Protein AKAP1 (4): AUROC = 0.9924 | F1: 0.9763 | F1(0.5): 0.9634 | Threshold: 0.7835 | TP:  392 | FP:   11 | FN:    8 | TN:  321 | AvgSum: 55.45
Protein FASTKD2 (38): AUROC = 0.9923 | F1: 0.9577 | F1(0.5): 0.9402 | Threshold: 0.8448 | TP:  283 | FP:   16 | FN:    9 | TN:  267 | AvgSum: 63.61
Protein MTPAP (69): AUROC = 0.9916 | F1: 0.9726 | F1(0.5): 0.9625 | Threshold: 0.8505 | TP:  391 | FP:   13 | FN:    9 | TN:  331 | AvgSum: 58.32
Protein TAF15 (106): AUROC = 0.9884 | F1: 0.9527 | F1(0.5): 0.9190 | Threshold: 0.8355 | TP:  383 | FP:   21 | FN:   17 | TN:  318 | AvgSum: 58.16
Protein GRWD1 (51): AUROC = 0.9834 | F1: 0.9487 | F1(0.5): 0.9238 | Threshold: 0.8820 | TP:  379 | FP:   20 | FN:   21 | TN:  317 | AvgSum: 60.45
Protein EIF4E (30): AUROC = 0.9827 | F1: 0.9409 | F1(0.5): 0.9152 | Threshold: 0.7925 | TP:  382 | FP:   30 | FN:   18 | TN:  304 | AvgSum: 61.89
Protein DGCR8 (23): AUROC = 0.9809 | F1: 0.9333 | F1(0.5): 0.9227 | Threshold: 0.7741 | TP:  175 | FP:    7 | FN:   18 | TN:  176 | AvgSum: 57.98
[INFO] AUROC Bottom 10 proteins:
Protein PTBP1 (82): AUROC = 0.7921 | F1: 0.7514 | F1(0.5): 0.7429 | Threshold: 0.7810 | TP:  331 | FP:  150 | FN:   69 | TN:  191 | AvgSum: 56.06
Protein RYBP (92): AUROC = 0.7920 | F1: 0.7500 | F1(0.5): 0.7000 | Threshold: 0.6005 | TP:   78 | FP:   25 | FN:   27 | TN:   91 | AvgSum: 52.89
Protein PPIG (78): AUROC = 0.7907 | F1: 0.7348 | F1(0.5): 0.7279 | Threshold: 0.4081 | TP:  115 | FP:   66 | FN:   17 | TN:   72 | AvgSum: 54.73
Protein IGF2BP2 (58): AUROC = 0.7883 | F1: 0.7974 | F1(0.5): 0.7717 | Threshold: 0.7029 | TP:  366 | FP:  152 | FN:   34 | TN:  172 | AvgSum: 63.23
Protein KHSRP (62): AUROC = 0.7865 | F1: 0.7676 | F1(0.5): 0.7474 | Threshold: 0.6921 | TP:  355 | FP:  170 | FN:   45 | TN:  166 | AvgSum: 56.85
Protein G3BP1 (46): AUROC = 0.7806 | F1: 0.7581 | F1(0.5): 0.7500 | Threshold: 0.5519 | TP:   94 | FP:   38 | FN:   22 | TN:   69 | AvgSum: 62.42
Protein GPKOW (49): AUROC = 0.7779 | F1: 0.7465 | F1(0.5): 0.7345 | Threshold: 0.7039 | TP:  184 | FP:   96 | FN:   29 | TN:  107 | AvgSum: 67.46
Protein ILF3 (60): AUROC = 0.7731 | F1: 0.7418 | F1(0.5): 0.7013 | Threshold: 0.9093 | TP:  214 | FP:  118 | FN:   31 | TN:  130 | AvgSum: 40.30
Protein CPSF6 (12): AUROC = 0.7563 | F1: 0.7423 | F1(0.5): 0.7140 | Threshold: 0.8389 | TP:  350 | FP:  193 | FN:   50 | TN:  151 | AvgSum: 52.23
Protein IGF2BP1 (57): AUROC = 0.7252 | F1: 0.7601 | F1(0.5): 0.7495 | Threshold: 0.6695 | TP:  377 | FP:  215 | FN:   23 | TN:   79 | AvgSum: 63.48

Epoch 6/30
Loss: 2.1045 | AUROC: 0.9065 | F1: 0.8616 | F1(0.5): 0.8419 | mAP: 0.9249 | AvgSum: 50.6923 | Train time: 92.27s | Eval time: 8.43s
Average threshold: 0.6868729
Thresholds (min/max): 0.30761474 0.9146117
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9966 | F1: 0.9777 | F1(0.5): 0.9670 | Threshold: 0.6583 | TP:  395 | FP:   13 | FN:    5 | TN:  314 | AvgSum: 42.99
Protein GARS (47): AUROC = 0.9955 | F1: 0.9762 | F1(0.5): 0.9695 | Threshold: 0.3510 | TP:  389 | FP:    8 | FN:   11 | TN:  334 | AvgSum: 50.00
Protein ADAT1 (2): AUROC = 0.9954 | F1: 0.9727 | F1(0.5): 0.9692 | Threshold: 0.5358 | TP:  392 | FP:   14 | FN:    8 | TN:  309 | AvgSum: 41.17
Protein AKAP1 (4): AUROC = 0.9923 | F1: 0.9788 | F1(0.5): 0.9657 | Threshold: 0.7893 | TP:  393 | FP:   10 | FN:    7 | TN:  322 | AvgSum: 49.98
Protein MTPAP (69): AUROC = 0.9903 | F1: 0.9698 | F1(0.5): 0.9589 | Threshold: 0.9146 | TP:  386 | FP:   10 | FN:   14 | TN:  334 | AvgSum: 53.50
Protein FASTKD2 (38): AUROC = 0.9897 | F1: 0.9612 | F1(0.5): 0.9419 | Threshold: 0.9047 | TP:  285 | FP:   16 | FN:    7 | TN:  267 | AvgSum: 58.31
Protein TAF15 (106): AUROC = 0.9874 | F1: 0.9531 | F1(0.5): 0.9267 | Threshold: 0.7846 | TP:  386 | FP:   24 | FN:   14 | TN:  315 | AvgSum: 49.89
Protein DGCR8 (23): AUROC = 0.9870 | F1: 0.9474 | F1(0.5): 0.9474 | Threshold: 0.5249 | TP:  189 | FP:   17 | FN:    4 | TN:  166 | AvgSum: 52.47
Protein METAP2 (67): AUROC = 0.9854 | F1: 0.9434 | F1(0.5): 0.9409 | Threshold: 0.5082 | TP:  175 | FP:   16 | FN:    5 | TN:  181 | AvgSum: 50.81
Protein FAM120A (37): AUROC = 0.9815 | F1: 0.9431 | F1(0.5): 0.9393 | Threshold: 0.5166 | TP:  116 | FP:    7 | FN:    7 | TN:  115 | AvgSum: 53.07
[INFO] AUROC Bottom 10 proteins:
Protein IGF2BP3 (59): AUROC = 0.8019 | F1: 0.7807 | F1(0.5): 0.7481 | Threshold: 0.7682 | TP:  365 | FP:  171 | FN:   34 | TN:  139 | AvgSum: 42.40
Protein FXR2 (45): AUROC = 0.7954 | F1: 0.7794 | F1(0.5): 0.7709 | Threshold: 0.6781 | TP:  364 | FP:  170 | FN:   36 | TN:  149 | AvgSum: 53.87
Protein AKAP8L (5): AUROC = 0.7930 | F1: 0.7455 | F1(0.5): 0.7214 | Threshold: 0.5742 | TP:  164 | FP:   71 | FN:   41 | TN:  128 | AvgSum: 54.94
Protein ILF3 (60): AUROC = 0.7862 | F1: 0.7389 | F1(0.5): 0.7163 | Threshold: 0.7374 | TP:  208 | FP:  110 | FN:   37 | TN:  138 | AvgSum: 34.70
Protein KHSRP (62): AUROC = 0.7848 | F1: 0.7608 | F1(0.5): 0.7495 | Threshold: 0.7087 | TP:  334 | FP:  144 | FN:   66 | TN:  192 | AvgSum: 49.47
Protein GPKOW (49): AUROC = 0.7786 | F1: 0.7455 | F1(0.5): 0.7263 | Threshold: 0.7101 | TP:  186 | FP:  100 | FN:   27 | TN:  103 | AvgSum: 61.79
Protein CPSF6 (12): AUROC = 0.7673 | F1: 0.7510 | F1(0.5): 0.7295 | Threshold: 0.6266 | TP:  371 | FP:  217 | FN:   29 | TN:  127 | AvgSum: 44.80
Protein PPIG (78): AUROC = 0.7498 | F1: 0.7101 | F1(0.5): 0.6940 | Threshold: 0.6949 | TP:   98 | FP:   46 | FN:   34 | TN:   92 | AvgSum: 47.71
Protein IGF2BP1 (57): AUROC = 0.7464 | F1: 0.7655 | F1(0.5): 0.7602 | Threshold: 0.4844 | TP:  377 | FP:  208 | FN:   23 | TN:   86 | AvgSum: 57.03
Protein RYBP (92): AUROC = 0.7433 | F1: 0.7000 | F1(0.5): 0.6914 | Threshold: 0.6237 | TP:   84 | FP:   51 | FN:   21 | TN:   65 | AvgSum: 45.84

Epoch 7/30
Loss: 1.9340 | AUROC: 0.9086 | F1: 0.8640 | F1(0.5): 0.8415 | mAP: 0.9227 | AvgSum: 50.5887 | Train time: 92.35s | Eval time: 8.37s
Average threshold: 0.699765
Thresholds (min/max): 0.2330964 0.91759443
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9974 | F1: 0.9801 | F1(0.5): 0.9766 | Threshold: 0.6922 | TP:  395 | FP:   11 | FN:    5 | TN:  316 | AvgSum: 44.53
Protein ADAT1 (2): AUROC = 0.9962 | F1: 0.9736 | F1(0.5): 0.9711 | Threshold: 0.5551 | TP:  387 | FP:    8 | FN:   13 | TN:  315 | AvgSum: 43.03
Protein GARS (47): AUROC = 0.9944 | F1: 0.9702 | F1(0.5): 0.9688 | Threshold: 0.4708 | TP:  391 | FP:   15 | FN:    9 | TN:  327 | AvgSum: 49.33
Protein AKAP1 (4): AUROC = 0.9922 | F1: 0.9763 | F1(0.5): 0.9669 | Threshold: 0.7483 | TP:  391 | FP:   10 | FN:    9 | TN:  322 | AvgSum: 51.49
Protein MTPAP (69): AUROC = 0.9888 | F1: 0.9661 | F1(0.5): 0.9561 | Threshold: 0.8242 | TP:  385 | FP:   12 | FN:   15 | TN:  332 | AvgSum: 55.02
Protein FASTKD2 (38): AUROC = 0.9874 | F1: 0.9663 | F1(0.5): 0.9600 | Threshold: 0.5886 | TP:  287 | FP:   15 | FN:    5 | TN:  268 | AvgSum: 58.64
Protein DGCR8 (23): AUROC = 0.9853 | F1: 0.9479 | F1(0.5): 0.9476 | Threshold: 0.4673 | TP:  182 | FP:    9 | FN:   11 | TN:  174 | AvgSum: 52.93
Protein GRWD1 (51): AUROC = 0.9845 | F1: 0.9476 | F1(0.5): 0.9393 | Threshold: 0.6933 | TP:  380 | FP:   22 | FN:   20 | TN:  315 | AvgSum: 55.17
Protein TAF15 (106): AUROC = 0.9844 | F1: 0.9491 | F1(0.5): 0.9354 | Threshold: 0.7617 | TP:  382 | FP:   23 | FN:   18 | TN:  316 | AvgSum: 50.65
Protein EIF4E (30): AUROC = 0.9826 | F1: 0.9384 | F1(0.5): 0.9322 | Threshold: 0.6389 | TP:  381 | FP:   31 | FN:   19 | TN:  303 | AvgSum: 55.73
[INFO] AUROC Bottom 10 proteins:
Protein IGF2BP2 (58): AUROC = 0.8002 | F1: 0.8071 | F1(0.5): 0.7887 | Threshold: 0.6837 | TP:  366 | FP:  141 | FN:   34 | TN:  183 | AvgSum: 56.07
Protein AKAP8L (5): AUROC = 0.7899 | F1: 0.7480 | F1(0.5): 0.7316 | Threshold: 0.7080 | TP:  184 | FP:  103 | FN:   21 | TN:   96 | AvgSum: 52.97
Protein RPS3 (91): AUROC = 0.7881 | F1: 0.7112 | F1(0.5): 0.6914 | Threshold: 0.3949 | TP:  133 | FP:   90 | FN:   18 | TN:   76 | AvgSum: 45.00
Protein KHSRP (62): AUROC = 0.7814 | F1: 0.7669 | F1(0.5): 0.7344 | Threshold: 0.8929 | TP:  329 | FP:  129 | FN:   71 | TN:  207 | AvgSum: 48.80
Protein GPKOW (49): AUROC = 0.7804 | F1: 0.7406 | F1(0.5): 0.7378 | Threshold: 0.6216 | TP:  197 | FP:  122 | FN:   16 | TN:   81 | AvgSum: 61.48
Protein PPIG (78): AUROC = 0.7714 | F1: 0.7215 | F1(0.5): 0.7055 | Threshold: 0.2965 | TP:  114 | FP:   70 | FN:   18 | TN:   68 | AvgSum: 49.38
Protein RYBP (92): AUROC = 0.7646 | F1: 0.7117 | F1(0.5): 0.7074 | Threshold: 0.5636 | TP:   79 | FP:   38 | FN:   26 | TN:   78 | AvgSum: 48.88
Protein IGF2BP1 (57): AUROC = 0.7612 | F1: 0.7694 | F1(0.5): 0.7467 | Threshold: 0.8731 | TP:  337 | FP:  139 | FN:   63 | TN:  155 | AvgSum: 55.62
Protein ILF3 (60): AUROC = 0.7573 | F1: 0.7379 | F1(0.5): 0.7261 | Threshold: 0.7241 | TP:  221 | FP:  133 | FN:   24 | TN:  115 | AvgSum: 34.98
Protein CPSF6 (12): AUROC = 0.7496 | F1: 0.7564 | F1(0.5): 0.7239 | Threshold: 0.6849 | TP:  357 | FP:  187 | FN:   43 | TN:  157 | AvgSum: 45.65

Epoch 8/30
Loss: 1.7645 | AUROC: 0.9101 | F1: 0.8652 | F1(0.5): 0.8478 | mAP: 0.9237 | AvgSum: 46.3624 | Train time: 92.22s | Eval time: 8.34s
Average threshold: 0.6732404
Thresholds (min/max): 0.14250462 0.94974357
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9975 | F1: 0.9812 | F1(0.5): 0.9740 | Threshold: 0.8127 | TP:  391 | FP:    6 | FN:    9 | TN:  321 | AvgSum: 41.17
Protein ADAT1 (2): AUROC = 0.9964 | F1: 0.9759 | F1(0.5): 0.9726 | Threshold: 0.8610 | TP:  385 | FP:    4 | FN:   15 | TN:  319 | AvgSum: 39.47
Protein GARS (47): AUROC = 0.9949 | F1: 0.9763 | F1(0.5): 0.9684 | Threshold: 0.3981 | TP:  392 | FP:   11 | FN:    8 | TN:  331 | AvgSum: 45.32
Protein AKAP1 (4): AUROC = 0.9940 | F1: 0.9740 | F1(0.5): 0.9705 | Threshold: 0.6281 | TP:  393 | FP:   14 | FN:    7 | TN:  318 | AvgSum: 46.64
Protein FASTKD2 (38): AUROC = 0.9914 | F1: 0.9664 | F1(0.5): 0.9448 | Threshold: 0.8614 | TP:  288 | FP:   16 | FN:    4 | TN:  267 | AvgSum: 54.01
Protein MTPAP (69): AUROC = 0.9894 | F1: 0.9691 | F1(0.5): 0.9563 | Threshold: 0.7497 | TP:  392 | FP:   17 | FN:    8 | TN:  327 | AvgSum: 49.43
Protein TAF15 (106): AUROC = 0.9882 | F1: 0.9525 | F1(0.5): 0.9106 | Threshold: 0.8970 | TP:  381 | FP:   19 | FN:   19 | TN:  320 | AvgSum: 46.47
Protein DGCR8 (23): AUROC = 0.9869 | F1: 0.9433 | F1(0.5): 0.9409 | Threshold: 0.5685 | TP:  183 | FP:   12 | FN:   10 | TN:  171 | AvgSum: 47.81
Protein PCBP2 (76): AUROC = 0.9851 | F1: 0.9391 | F1(0.5): 0.9278 | Threshold: 0.6484 | TP:  131 | FP:    8 | FN:    9 | TN:  126 | AvgSum: 51.58
Protein FAM120A (37): AUROC = 0.9843 | F1: 0.9492 | F1(0.5): 0.9417 | Threshold: 0.6758 | TP:  112 | FP:    1 | FN:   11 | TN:  121 | AvgSum: 47.64
[INFO] AUROC Bottom 10 proteins:
Protein PTBP1 (82): AUROC = 0.8091 | F1: 0.7720 | F1(0.5): 0.7541 | Threshold: 0.7143 | TP:  347 | FP:  152 | FN:   53 | TN:  189 | AvgSum: 46.07
Protein FXR2 (45): AUROC = 0.8066 | F1: 0.7829 | F1(0.5): 0.7734 | Threshold: 0.6031 | TP:  357 | FP:  155 | FN:   43 | TN:  164 | AvgSum: 48.78
Protein PPIG (78): AUROC = 0.8050 | F1: 0.7446 | F1(0.5): 0.7329 | Threshold: 0.2477 | TP:  121 | FP:   72 | FN:   11 | TN:   66 | AvgSum: 43.71
Protein RPS3 (91): AUROC = 0.7976 | F1: 0.7268 | F1(0.5): 0.7108 | Threshold: 0.3129 | TP:  133 | FP:   82 | FN:   18 | TN:   84 | AvgSum: 40.27
Protein GPKOW (49): AUROC = 0.7816 | F1: 0.7481 | F1(0.5): 0.7427 | Threshold: 0.3211 | TP:  202 | FP:  125 | FN:   11 | TN:   78 | AvgSum: 56.31
Protein KHSRP (62): AUROC = 0.7735 | F1: 0.7619 | F1(0.5): 0.7302 | Threshold: 0.8541 | TP:  328 | FP:  133 | FN:   72 | TN:  203 | AvgSum: 45.13
Protein IGF2BP1 (57): AUROC = 0.7718 | F1: 0.7784 | F1(0.5): 0.7669 | Threshold: 0.6244 | TP:  360 | FP:  165 | FN:   40 | TN:  129 | AvgSum: 51.64
Protein CPSF6 (12): AUROC = 0.7714 | F1: 0.7471 | F1(0.5): 0.7254 | Threshold: 0.8986 | TP:  325 | FP:  145 | FN:   75 | TN:  199 | AvgSum: 40.89
Protein RYBP (92): AUROC = 0.7562 | F1: 0.6926 | F1(0.5): 0.6763 | Threshold: 0.1425 | TP:   98 | FP:   80 | FN:    7 | TN:   36 | AvgSum: 42.42
Protein ILF3 (60): AUROC = 0.7536 | F1: 0.7244 | F1(0.5): 0.7112 | Threshold: 0.6414 | TP:  230 | FP:  160 | FN:   15 | TN:   88 | AvgSum: 31.73

Epoch 9/30
Loss: 1.6250 | AUROC: 0.9123 | F1: 0.8671 | F1(0.5): 0.8542 | mAP: 0.9272 | AvgSum: 41.2170 | Train time: 92.15s | Eval time: 8.35s
Average threshold: 0.5977836
Thresholds (min/max): 0.088461064 0.8798443
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9982 | F1: 0.9838 | F1(0.5): 0.9825 | Threshold: 0.5183 | TP:  394 | FP:    7 | FN:    6 | TN:  320 | AvgSum: 39.18
Protein ADAT1 (2): AUROC = 0.9941 | F1: 0.9722 | F1(0.5): 0.9708 | Threshold: 0.4851 | TP:  384 | FP:    6 | FN:   16 | TN:  317 | AvgSum: 36.60
Protein GARS (47): AUROC = 0.9940 | F1: 0.9689 | F1(0.5): 0.9647 | Threshold: 0.3792 | TP:  389 | FP:   14 | FN:   11 | TN:  328 | AvgSum: 41.44
Protein AKAP1 (4): AUROC = 0.9919 | F1: 0.9738 | F1(0.5): 0.9699 | Threshold: 0.4486 | TP:  391 | FP:   12 | FN:    9 | TN:  320 | AvgSum: 43.90
Protein FASTKD2 (38): AUROC = 0.9917 | F1: 0.9668 | F1(0.5): 0.9633 | Threshold: 0.4917 | TP:  291 | FP:   19 | FN:    1 | TN:  264 | AvgSum: 50.60
Protein MTPAP (69): AUROC = 0.9910 | F1: 0.9681 | F1(0.5): 0.9600 | Threshold: 0.3293 | TP:  394 | FP:   20 | FN:    6 | TN:  324 | AvgSum: 46.24
Protein NOLC1 (71): AUROC = 0.9871 | F1: 0.9524 | F1(0.5): 0.9506 | Threshold: 0.3854 | TP:  390 | FP:   31 | FN:    8 | TN:  297 | AvgSum: 48.17
Protein PCBP2 (76): AUROC = 0.9866 | F1: 0.9527 | F1(0.5): 0.9493 | Threshold: 0.5239 | TP:  131 | FP:    4 | FN:    9 | TN:  130 | AvgSum: 47.31
Protein DGCR8 (23): AUROC = 0.9854 | F1: 0.9452 | F1(0.5): 0.9396 | Threshold: 0.4693 | TP:  181 | FP:    9 | FN:   12 | TN:  174 | AvgSum: 43.87
Protein TAF15 (106): AUROC = 0.9851 | F1: 0.9452 | F1(0.5): 0.9356 | Threshold: 0.8468 | TP:  371 | FP:   14 | FN:   29 | TN:  325 | AvgSum: 41.74
[INFO] AUROC Bottom 10 proteins:
Protein PPIG (78): AUROC = 0.8108 | F1: 0.7456 | F1(0.5): 0.6996 | Threshold: 0.0885 | TP:  126 | FP:   80 | FN:    6 | TN:   58 | AvgSum: 37.87
Protein RPS3 (91): AUROC = 0.8083 | F1: 0.7301 | F1(0.5): 0.7267 | Threshold: 0.3788 | TP:  119 | FP:   56 | FN:   32 | TN:  110 | AvgSum: 33.86
Protein KHSRP (62): AUROC = 0.8027 | F1: 0.7799 | F1(0.5): 0.7502 | Threshold: 0.8162 | TP:  342 | FP:  135 | FN:   58 | TN:  201 | AvgSum: 38.92
Protein FXR2 (45): AUROC = 0.8014 | F1: 0.7950 | F1(0.5): 0.7899 | Threshold: 0.7443 | TP:  347 | FP:  126 | FN:   53 | TN:  193 | AvgSum: 42.67
Protein IGF2BP2 (58): AUROC = 0.7931 | F1: 0.7913 | F1(0.5): 0.7868 | Threshold: 0.4797 | TP:  364 | FP:  156 | FN:   36 | TN:  168 | AvgSum: 45.76
Protein RYBP (92): AUROC = 0.7888 | F1: 0.7217 | F1(0.5): 0.6931 | Threshold: 0.2847 | TP:   83 | FP:   42 | FN:   22 | TN:   74 | AvgSum: 36.96
Protein GPKOW (49): AUROC = 0.7783 | F1: 0.7462 | F1(0.5): 0.7379 | Threshold: 0.4815 | TP:  194 | FP:  113 | FN:   19 | TN:   90 | AvgSum: 51.87
Protein ILF3 (60): AUROC = 0.7659 | F1: 0.7406 | F1(0.5): 0.7209 | Threshold: 0.7045 | TP:  227 | FP:  141 | FN:   18 | TN:  107 | AvgSum: 27.16
Protein CPSF6 (12): AUROC = 0.7635 | F1: 0.7530 | F1(0.5): 0.7368 | Threshold: 0.7371 | TP:  343 | FP:  168 | FN:   57 | TN:  176 | AvgSum: 35.42
Protein IGF2BP1 (57): AUROC = 0.7495 | F1: 0.7602 | F1(0.5): 0.7542 | Threshold: 0.6266 | TP:  344 | FP:  161 | FN:   56 | TN:  133 | AvgSum: 45.90

Epoch 10/30
Loss: 1.5075 | AUROC: 0.9129 | F1: 0.8665 | F1(0.5): 0.8559 | mAP: 0.9304 | AvgSum: 36.7392 | Train time: 92.43s | Eval time: 8.37s
Average threshold: 0.5005668
Thresholds (min/max): 0.07407407 0.8671057
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9982 | F1: 0.9812 | F1(0.5): 0.9776 | Threshold: 0.7087 | TP:  391 | FP:    6 | FN:    9 | TN:  321 | AvgSum: 34.10
Protein ADAT1 (2): AUROC = 0.9950 | F1: 0.9724 | F1(0.5): 0.9711 | Threshold: 0.4860 | TP:  388 | FP:   10 | FN:   12 | TN:  313 | AvgSum: 31.72
Protein GARS (47): AUROC = 0.9946 | F1: 0.9709 | F1(0.5): 0.9603 | Threshold: 0.2587 | TP:  384 | FP:    7 | FN:   16 | TN:  335 | AvgSum: 37.68
Protein FASTKD2 (38): AUROC = 0.9923 | F1: 0.9623 | F1(0.5): 0.9565 | Threshold: 0.7205 | TP:  281 | FP:   11 | FN:   11 | TN:  272 | AvgSum: 47.30
Protein AKAP1 (4): AUROC = 0.9917 | F1: 0.9691 | F1(0.5): 0.9677 | Threshold: 0.3392 | TP:  392 | FP:   17 | FN:    8 | TN:  315 | AvgSum: 39.69
Protein MTPAP (69): AUROC = 0.9911 | F1: 0.9686 | F1(0.5): 0.9653 | Threshold: 0.7460 | TP:  385 | FP:   10 | FN:   15 | TN:  334 | AvgSum: 42.67
Protein DGCR8 (23): AUROC = 0.9870 | F1: 0.9474 | F1(0.5): 0.9440 | Threshold: 0.4283 | TP:  180 | FP:    7 | FN:   13 | TN:  176 | AvgSum: 38.90
Protein TAF15 (106): AUROC = 0.9863 | F1: 0.9450 | F1(0.5): 0.9370 | Threshold: 0.6559 | TP:  378 | FP:   22 | FN:   22 | TN:  317 | AvgSum: 36.68
Protein NOLC1 (71): AUROC = 0.9861 | F1: 0.9523 | F1(0.5): 0.9523 | Threshold: 0.5043 | TP:  389 | FP:   30 | FN:    9 | TN:  298 | AvgSum: 44.43
Protein PCBP2 (76): AUROC = 0.9837 | F1: 0.9404 | F1(0.5): 0.9366 | Threshold: 0.4955 | TP:  134 | FP:   11 | FN:    6 | TN:  123 | AvgSum: 43.07
[INFO] AUROC Bottom 10 proteins:
Protein PPIG (78): AUROC = 0.8017 | F1: 0.7455 | F1(0.5): 0.7050 | Threshold: 0.3926 | TP:  104 | FP:   43 | FN:   28 | TN:   95 | AvgSum: 31.87
Protein RPS3 (91): AUROC = 0.7953 | F1: 0.7189 | F1(0.5): 0.6908 | Threshold: 0.1508 | TP:  133 | FP:   86 | FN:   18 | TN:   80 | AvgSum: 26.73
Protein IGF2BP2 (58): AUROC = 0.7923 | F1: 0.7931 | F1(0.5): 0.7735 | Threshold: 0.3974 | TP:  368 | FP:  160 | FN:   32 | TN:  164 | AvgSum: 41.42
Protein FXR2 (45): AUROC = 0.7919 | F1: 0.7796 | F1(0.5): 0.7740 | Threshold: 0.4049 | TP:  359 | FP:  162 | FN:   41 | TN:  157 | AvgSum: 39.08
Protein KHSRP (62): AUROC = 0.7900 | F1: 0.7755 | F1(0.5): 0.7598 | Threshold: 0.5953 | TP:  335 | FP:  129 | FN:   65 | TN:  207 | AvgSum: 34.08
Protein CPSF6 (12): AUROC = 0.7880 | F1: 0.7492 | F1(0.5): 0.7418 | Threshold: 0.3315 | TP:  372 | FP:  221 | FN:   28 | TN:  123 | AvgSum: 27.86
Protein ILF3 (60): AUROC = 0.7806 | F1: 0.7409 | F1(0.5): 0.7211 | Threshold: 0.8399 | TP:  213 | FP:  117 | FN:   32 | TN:  131 | AvgSum: 18.27
Protein RYBP (92): AUROC = 0.7800 | F1: 0.7266 | F1(0.5): 0.6765 | Threshold: 0.1561 | TP:   93 | FP:   58 | FN:   12 | TN:   58 | AvgSum: 29.73
Protein GPKOW (49): AUROC = 0.7672 | F1: 0.7444 | F1(0.5): 0.7216 | Threshold: 0.2463 | TP:  198 | FP:  121 | FN:   15 | TN:   82 | AvgSum: 48.72
Protein IGF2BP1 (57): AUROC = 0.7522 | F1: 0.7723 | F1(0.5): 0.7592 | Threshold: 0.2890 | TP:  373 | FP:  193 | FN:   27 | TN:  101 | AvgSum: 41.81

Epoch 11/30
Loss: 1.4016 | AUROC: 0.9099 | F1: 0.8646 | F1(0.5): 0.8526 | mAP: 0.9255 | AvgSum: 36.3517 | Train time: 92.21s | Eval time: 8.39s
Average threshold: 0.52890164
Thresholds (min/max): 0.101866394 0.87916267
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9982 | F1: 0.9828 | F1(0.5): 0.9825 | Threshold: 0.2945 | TP:  399 | FP:   13 | FN:    1 | TN:  314 | AvgSum: 34.21
Protein GARS (47): AUROC = 0.9944 | F1: 0.9722 | F1(0.5): 0.9561 | Threshold: 0.3083 | TP:  385 | FP:    7 | FN:   15 | TN:  335 | AvgSum: 36.57
Protein ADAT1 (2): AUROC = 0.9937 | F1: 0.9774 | F1(0.5): 0.9772 | Threshold: 0.2850 | TP:  389 | FP:    7 | FN:   11 | TN:  316 | AvgSum: 31.73
Protein AKAP1 (4): AUROC = 0.9913 | F1: 0.9738 | F1(0.5): 0.9725 | Threshold: 0.4417 | TP:  391 | FP:   12 | FN:    9 | TN:  320 | AvgSum: 39.46
Protein FASTKD2 (38): AUROC = 0.9911 | F1: 0.9747 | F1(0.5): 0.9730 | Threshold: 0.4770 | TP:  289 | FP:   12 | FN:    3 | TN:  271 | AvgSum: 46.16
Protein MTPAP (69): AUROC = 0.9902 | F1: 0.9618 | F1(0.5): 0.9618 | Threshold: 0.5320 | TP:  390 | FP:   21 | FN:   10 | TN:  323 | AvgSum: 42.31
Protein TAF15 (106): AUROC = 0.9862 | F1: 0.9518 | F1(0.5): 0.9465 | Threshold: 0.7150 | TP:  385 | FP:   24 | FN:   15 | TN:  315 | AvgSum: 36.48
Protein DGCR8 (23): AUROC = 0.9856 | F1: 0.9485 | F1(0.5): 0.9268 | Threshold: 0.1309 | TP:  184 | FP:   11 | FN:    9 | TN:  172 | AvgSum: 38.74
Protein GRWD1 (51): AUROC = 0.9843 | F1: 0.9435 | F1(0.5): 0.9394 | Threshold: 0.6409 | TP:  376 | FP:   21 | FN:   24 | TN:  316 | AvgSum: 42.54
Protein EIF4E (30): AUROC = 0.9827 | F1: 0.9491 | F1(0.5): 0.9365 | Threshold: 0.7091 | TP:  373 | FP:   13 | FN:   27 | TN:  321 | AvgSum: 42.83
[INFO] AUROC Bottom 10 proteins:
Protein G3BP1 (46): AUROC = 0.7988 | F1: 0.7607 | F1(0.5): 0.7306 | Threshold: 0.3429 | TP:   89 | FP:   29 | FN:   27 | TN:   78 | AvgSum: 38.15
Protein KHSRP (62): AUROC = 0.7954 | F1: 0.7799 | F1(0.5): 0.7742 | Threshold: 0.4296 | TP:  372 | FP:  182 | FN:   28 | TN:  154 | AvgSum: 33.98
Protein PPIG (78): AUROC = 0.7946 | F1: 0.7376 | F1(0.5): 0.6935 | Threshold: 0.3321 | TP:   97 | FP:   34 | FN:   35 | TN:  104 | AvgSum: 34.11
Protein FXR2 (45): AUROC = 0.7943 | F1: 0.7862 | F1(0.5): 0.7684 | Threshold: 0.7332 | TP:  364 | FP:  162 | FN:   36 | TN:  157 | AvgSum: 37.32
Protein IGF2BP2 (58): AUROC = 0.7916 | F1: 0.7970 | F1(0.5): 0.7775 | Threshold: 0.4135 | TP:  367 | FP:  154 | FN:   33 | TN:  170 | AvgSum: 40.81
Protein GPKOW (49): AUROC = 0.7799 | F1: 0.7571 | F1(0.5): 0.7389 | Threshold: 0.4179 | TP:  187 | FP:   94 | FN:   26 | TN:  109 | AvgSum: 47.18
Protein ILF3 (60): AUROC = 0.7615 | F1: 0.7337 | F1(0.5): 0.7121 | Threshold: 0.7890 | TP:  197 | FP:   95 | FN:   48 | TN:  153 | AvgSum: 20.81
Protein CPSF6 (12): AUROC = 0.7584 | F1: 0.7446 | F1(0.5): 0.7355 | Threshold: 0.2234 | TP:  360 | FP:  207 | FN:   40 | TN:  137 | AvgSum: 29.77
Protein IGF2BP1 (57): AUROC = 0.7581 | F1: 0.7673 | F1(0.5): 0.7509 | Threshold: 0.3238 | TP:  366 | FP:  188 | FN:   34 | TN:  106 | AvgSum: 40.74
Protein RYBP (92): AUROC = 0.7462 | F1: 0.6920 | F1(0.5): 0.6321 | Threshold: 0.1019 | TP:   91 | FP:   67 | FN:   14 | TN:   49 | AvgSum: 31.89

Epoch 12/30
Loss: 1.3088 | AUROC: 0.9134 | F1: 0.8679 | F1(0.5): 0.8556 | mAP: 0.9277 | AvgSum: 33.2505 | Train time: 92.07s | Eval time: 8.36s
Average threshold: 0.45588937
Thresholds (min/max): 0.0329424 0.89856774
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9976 | F1: 0.9798 | F1(0.5): 0.9786 | Threshold: 0.4968 | TP:  389 | FP:    5 | FN:   11 | TN:  322 | AvgSum: 31.27
Protein ADAT1 (2): AUROC = 0.9953 | F1: 0.9763 | F1(0.5): 0.9747 | Threshold: 0.1878 | TP:  392 | FP:   11 | FN:    8 | TN:  312 | AvgSum: 29.06
Protein AKAP1 (4): AUROC = 0.9943 | F1: 0.9739 | F1(0.5): 0.9673 | Threshold: 0.2574 | TP:  392 | FP:   13 | FN:    8 | TN:  319 | AvgSum: 36.50
Protein GARS (47): AUROC = 0.9942 | F1: 0.9711 | F1(0.5): 0.9602 | Threshold: 0.2876 | TP:  386 | FP:    9 | FN:   14 | TN:  333 | AvgSum: 34.16
Protein FASTKD2 (38): AUROC = 0.9909 | F1: 0.9646 | F1(0.5): 0.9592 | Threshold: 0.3869 | TP:  286 | FP:   15 | FN:    6 | TN:  268 | AvgSum: 42.94
Protein MTPAP (69): AUROC = 0.9908 | F1: 0.9737 | F1(0.5): 0.9737 | Threshold: 0.5063 | TP:  388 | FP:    9 | FN:   12 | TN:  335 | AvgSum: 39.20
Protein DGCR8 (23): AUROC = 0.9861 | F1: 0.9471 | F1(0.5): 0.9210 | Threshold: 0.0550 | TP:  188 | FP:   16 | FN:    5 | TN:  167 | AvgSum: 35.89
Protein METAP2 (67): AUROC = 0.9836 | F1: 0.9359 | F1(0.5): 0.9155 | Threshold: 0.2243 | TP:  168 | FP:   11 | FN:   12 | TN:  186 | AvgSum: 35.09
Protein NOLC1 (71): AUROC = 0.9835 | F1: 0.9514 | F1(0.5): 0.9455 | Threshold: 0.3669 | TP:  382 | FP:   23 | FN:   16 | TN:  305 | AvgSum: 40.56
Protein TAF15 (106): AUROC = 0.9828 | F1: 0.9413 | F1(0.5): 0.9402 | Threshold: 0.5084 | TP:  385 | FP:   33 | FN:   15 | TN:  306 | AvgSum: 33.45
[INFO] AUROC Bottom 10 proteins:
Protein G3BP1 (46): AUROC = 0.8132 | F1: 0.7679 | F1(0.5): 0.6875 | Threshold: 0.1881 | TP:   86 | FP:   22 | FN:   30 | TN:   85 | AvgSum: 35.63
Protein RPS3 (91): AUROC = 0.8049 | F1: 0.7172 | F1(0.5): 0.7059 | Threshold: 0.6011 | TP:  104 | FP:   35 | FN:   47 | TN:  131 | AvgSum: 25.78
Protein IGF2BP2 (58): AUROC = 0.8046 | F1: 0.7894 | F1(0.5): 0.7869 | Threshold: 0.2165 | TP:  373 | FP:  172 | FN:   27 | TN:  152 | AvgSum: 36.93
Protein FXR2 (45): AUROC = 0.7975 | F1: 0.7866 | F1(0.5): 0.7797 | Threshold: 0.6394 | TP:  352 | FP:  143 | FN:   48 | TN:  176 | AvgSum: 34.54
Protein KHSRP (62): AUROC = 0.7923 | F1: 0.7695 | F1(0.5): 0.7643 | Threshold: 0.4028 | TP:  374 | FP:  198 | FN:   26 | TN:  138 | AvgSum: 31.13
Protein GPKOW (49): AUROC = 0.7838 | F1: 0.7449 | F1(0.5): 0.7359 | Threshold: 0.6002 | TP:  165 | FP:   65 | FN:   48 | TN:  138 | AvgSum: 43.88
Protein ILF3 (60): AUROC = 0.7807 | F1: 0.7414 | F1(0.5): 0.7219 | Threshold: 0.8986 | TP:  205 | FP:  103 | FN:   40 | TN:  145 | AvgSum: 19.07
Protein CPSF6 (12): AUROC = 0.7785 | F1: 0.7489 | F1(0.5): 0.7430 | Threshold: 0.5285 | TP:  334 | FP:  158 | FN:   66 | TN:  186 | AvgSum: 26.81
Protein IGF2BP1 (57): AUROC = 0.7720 | F1: 0.7621 | F1(0.5): 0.7578 | Threshold: 0.1817 | TP:  378 | FP:  214 | FN:   22 | TN:   80 | AvgSum: 37.03
Protein RYBP (92): AUROC = 0.7590 | F1: 0.7137 | F1(0.5): 0.6495 | Threshold: 0.0582 | TP:   91 | FP:   59 | FN:   14 | TN:   57 | AvgSum: 27.88

Epoch 13/30
Loss: 1.2405 | AUROC: 0.9113 | F1: 0.8659 | F1(0.5): 0.8527 | mAP: 0.9252 | AvgSum: 33.2455 | Train time: 92.22s | Eval time: 8.44s
Average threshold: 0.47203043
Thresholds (min/max): 0.07157138 0.88836795
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9981 | F1: 0.9825 | F1(0.5): 0.9773 | Threshold: 0.3895 | TP:  393 | FP:    7 | FN:    7 | TN:  320 | AvgSum: 31.61
Protein ADAT1 (2): AUROC = 0.9930 | F1: 0.9737 | F1(0.5): 0.9719 | Threshold: 0.2192 | TP:  389 | FP:   10 | FN:   11 | TN:  313 | AvgSum: 29.17
Protein AKAP1 (4): AUROC = 0.9926 | F1: 0.9726 | F1(0.5): 0.9688 | Threshold: 0.3453 | TP:  391 | FP:   13 | FN:    9 | TN:  319 | AvgSum: 35.78
Protein MTPAP (69): AUROC = 0.9926 | F1: 0.9737 | F1(0.5): 0.9726 | Threshold: 0.5817 | TP:  389 | FP:   10 | FN:   11 | TN:  334 | AvgSum: 38.44
Protein GARS (47): AUROC = 0.9926 | F1: 0.9661 | F1(0.5): 0.9509 | Threshold: 0.2278 | TP:  385 | FP:   12 | FN:   15 | TN:  330 | AvgSum: 34.02
Protein FASTKD2 (38): AUROC = 0.9910 | F1: 0.9692 | F1(0.5): 0.9659 | Threshold: 0.5980 | TP:  283 | FP:    9 | FN:    9 | TN:  274 | AvgSum: 42.62
Protein CSTF2T (14): AUROC = 0.9859 | F1: 0.9429 | F1(0.5): 0.9372 | Threshold: 0.5581 | TP:  388 | FP:   35 | FN:   12 | TN:  315 | AvgSum: 43.22
Protein DGCR8 (23): AUROC = 0.9853 | F1: 0.9436 | F1(0.5): 0.9330 | Threshold: 0.1860 | TP:  184 | FP:   13 | FN:    9 | TN:  170 | AvgSum: 34.98
Protein TAF15 (106): AUROC = 0.9848 | F1: 0.9422 | F1(0.5): 0.9356 | Threshold: 0.7245 | TP:  375 | FP:   21 | FN:   25 | TN:  318 | AvgSum: 33.95
Protein GRWD1 (51): AUROC = 0.9848 | F1: 0.9494 | F1(0.5): 0.9472 | Threshold: 0.6053 | TP:  385 | FP:   26 | FN:   15 | TN:  311 | AvgSum: 38.92
[INFO] AUROC Bottom 10 proteins:
Protein G3BP1 (46): AUROC = 0.8158 | F1: 0.7607 | F1(0.5): 0.7010 | Threshold: 0.0771 | TP:   89 | FP:   29 | FN:   27 | TN:   78 | AvgSum: 36.06
Protein SF3B4 (96): AUROC = 0.8077 | F1: 0.7782 | F1(0.5): 0.7706 | Threshold: 0.6513 | TP:  342 | FP:  137 | FN:   58 | TN:  196 | AvgSum: 37.41
Protein FXR2 (45): AUROC = 0.8021 | F1: 0.7846 | F1(0.5): 0.7812 | Threshold: 0.5130 | TP:  357 | FP:  153 | FN:   43 | TN:  166 | AvgSum: 34.98
Protein ILF3 (60): AUROC = 0.7856 | F1: 0.7547 | F1(0.5): 0.7350 | Threshold: 0.7873 | TP:  220 | FP:  118 | FN:   25 | TN:  130 | AvgSum: 18.24
Protein IGF2BP2 (58): AUROC = 0.7822 | F1: 0.7871 | F1(0.5): 0.7787 | Threshold: 0.3228 | TP:  355 | FP:  147 | FN:   45 | TN:  177 | AvgSum: 37.33
Protein KHSRP (62): AUROC = 0.7770 | F1: 0.7641 | F1(0.5): 0.7548 | Threshold: 0.3330 | TP:  353 | FP:  171 | FN:   47 | TN:  165 | AvgSum: 31.19
Protein RYBP (92): AUROC = 0.7755 | F1: 0.7094 | F1(0.5): 0.6667 | Threshold: 0.1419 | TP:   83 | FP:   46 | FN:   22 | TN:   70 | AvgSum: 27.81
Protein GPKOW (49): AUROC = 0.7699 | F1: 0.7359 | F1(0.5): 0.7124 | Threshold: 0.0722 | TP:  202 | FP:  134 | FN:   11 | TN:   69 | AvgSum: 43.87
Protein CPSF6 (12): AUROC = 0.7556 | F1: 0.7456 | F1(0.5): 0.7380 | Threshold: 0.3664 | TP:  362 | FP:  209 | FN:   38 | TN:  135 | AvgSum: 26.88
Protein IGF2BP1 (57): AUROC = 0.7510 | F1: 0.7607 | F1(0.5): 0.7458 | Threshold: 0.3319 | TP:  345 | FP:  162 | FN:   55 | TN:  132 | AvgSum: 37.20

Epoch 14/30
Loss: 1.1808 | AUROC: 0.9105 | F1: 0.8648 | F1(0.5): 0.8495 | mAP: 0.9241 | AvgSum: 31.6300 | Train time: 92.28s | Eval time: 8.38s
Average threshold: 0.39915854
Thresholds (min/max): 0.0034700457 0.85473746
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9979 | F1: 0.9836 | F1(0.5): 0.9836 | Threshold: 0.5044 | TP:  391 | FP:    4 | FN:    9 | TN:  323 | AvgSum: 32.38
Protein GARS (47): AUROC = 0.9954 | F1: 0.9752 | F1(0.5): 0.9560 | Threshold: 0.0376 | TP:  394 | FP:   14 | FN:    6 | TN:  328 | AvgSum: 32.50
Protein AKAP1 (4): AUROC = 0.9932 | F1: 0.9739 | F1(0.5): 0.9687 | Threshold: 0.2054 | TP:  392 | FP:   13 | FN:    8 | TN:  319 | AvgSum: 34.91
Protein ADAT1 (2): AUROC = 0.9929 | F1: 0.9728 | F1(0.5): 0.9643 | Threshold: 0.0967 | TP:  393 | FP:   15 | FN:    7 | TN:  308 | AvgSum: 29.97
Protein FASTKD2 (38): AUROC = 0.9921 | F1: 0.9711 | F1(0.5): 0.9675 | Threshold: 0.4464 | TP:  286 | FP:   11 | FN:    6 | TN:  272 | AvgSum: 40.40
Protein MTPAP (69): AUROC = 0.9903 | F1: 0.9691 | F1(0.5): 0.9662 | Threshold: 0.3454 | TP:  392 | FP:   17 | FN:    8 | TN:  327 | AvgSum: 37.22
Protein TAF15 (106): AUROC = 0.9849 | F1: 0.9370 | F1(0.5): 0.9300 | Threshold: 0.7651 | TP:  372 | FP:   22 | FN:   28 | TN:  317 | AvgSum: 32.16
Protein PCBP2 (76): AUROC = 0.9846 | F1: 0.9328 | F1(0.5): 0.9272 | Threshold: 0.3889 | TP:  125 | FP:    3 | FN:   15 | TN:  131 | AvgSum: 36.81
Protein DGCR8 (23): AUROC = 0.9837 | F1: 0.9430 | F1(0.5): 0.9210 | Threshold: 0.1683 | TP:  182 | FP:   11 | FN:   11 | TN:  172 | AvgSum: 34.12
Protein METAP2 (67): AUROC = 0.9836 | F1: 0.9366 | F1(0.5): 0.9118 | Threshold: 0.1562 | TP:  170 | FP:   13 | FN:   10 | TN:  184 | AvgSum: 33.45
[INFO] AUROC Bottom 10 proteins:
Protein FXR2 (45): AUROC = 0.8091 | F1: 0.7893 | F1(0.5): 0.7783 | Threshold: 0.8073 | TP:  339 | FP:  120 | FN:   61 | TN:  199 | AvgSum: 33.22
Protein PPIG (78): AUROC = 0.8089 | F1: 0.7467 | F1(0.5): 0.6609 | Threshold: 0.0590 | TP:  112 | FP:   56 | FN:   20 | TN:   82 | AvgSum: 28.75
Protein IGF2BP2 (58): AUROC = 0.8081 | F1: 0.8027 | F1(0.5): 0.7766 | Threshold: 0.3621 | TP:  360 | FP:  137 | FN:   40 | TN:  187 | AvgSum: 34.21
Protein SF3B4 (96): AUROC = 0.8070 | F1: 0.7887 | F1(0.5): 0.7781 | Threshold: 0.6709 | TP:  334 | FP:  113 | FN:   66 | TN:  220 | AvgSum: 34.90
Protein KHSRP (62): AUROC = 0.7924 | F1: 0.7717 | F1(0.5): 0.7666 | Threshold: 0.5486 | TP:  333 | FP:  130 | FN:   67 | TN:  206 | AvgSum: 29.39
Protein GPKOW (49): AUROC = 0.7833 | F1: 0.7445 | F1(0.5): 0.7223 | Threshold: 0.2137 | TP:  185 | FP:   99 | FN:   28 | TN:  104 | AvgSum: 41.27
Protein RYBP (92): AUROC = 0.7717 | F1: 0.7067 | F1(0.5): 0.6535 | Threshold: 0.0234 | TP:  100 | FP:   78 | FN:    5 | TN:   38 | AvgSum: 26.80
Protein CPSF6 (12): AUROC = 0.7695 | F1: 0.7571 | F1(0.5): 0.7463 | Threshold: 0.4145 | TP:  346 | FP:  168 | FN:   54 | TN:  176 | AvgSum: 25.16
Protein ILF3 (60): AUROC = 0.7690 | F1: 0.7429 | F1(0.5): 0.7375 | Threshold: 0.7496 | TP:  208 | FP:  107 | FN:   37 | TN:  141 | AvgSum: 18.18
Protein IGF2BP1 (57): AUROC = 0.7507 | F1: 0.7601 | F1(0.5): 0.7348 | Threshold: 0.0289 | TP:  396 | FP:  246 | FN:    4 | TN:   48 | AvgSum: 34.71

Epoch 15/30
Loss: 1.1331 | AUROC: 0.9095 | F1: 0.8651 | F1(0.5): 0.8497 | mAP: 0.9248 | AvgSum: 31.0868 | Train time: 92.35s | Eval time: 8.40s
Average threshold: 0.4045884
Thresholds (min/max): 0.009276774 0.84430844
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9970 | F1: 0.9837 | F1(0.5): 0.9812 | Threshold: 0.4577 | TP:  393 | FP:    6 | FN:    7 | TN:  321 | AvgSum: 29.01
Protein GARS (47): AUROC = 0.9940 | F1: 0.9747 | F1(0.5): 0.9587 | Threshold: 0.2074 | TP:  386 | FP:    6 | FN:   14 | TN:  336 | AvgSum: 32.34
Protein ADAT1 (2): AUROC = 0.9939 | F1: 0.9812 | F1(0.5): 0.9654 | Threshold: 0.1905 | TP:  391 | FP:    6 | FN:    9 | TN:  317 | AvgSum: 27.04
Protein AKAP1 (4): AUROC = 0.9925 | F1: 0.9724 | F1(0.5): 0.9701 | Threshold: 0.6579 | TP:  388 | FP:   10 | FN:   12 | TN:  322 | AvgSum: 34.11
Protein FASTKD2 (38): AUROC = 0.9919 | F1: 0.9695 | F1(0.5): 0.9695 | Threshold: 0.5171 | TP:  286 | FP:   12 | FN:    6 | TN:  271 | AvgSum: 40.14
Protein MTPAP (69): AUROC = 0.9906 | F1: 0.9688 | F1(0.5): 0.9631 | Threshold: 0.6614 | TP:  388 | FP:   13 | FN:   12 | TN:  331 | AvgSum: 36.72
Protein PCBP2 (76): AUROC = 0.9864 | F1: 0.9362 | F1(0.5): 0.9098 | Threshold: 0.1955 | TP:  132 | FP:   10 | FN:    8 | TN:  124 | AvgSum: 36.01
Protein TAF15 (106): AUROC = 0.9842 | F1: 0.9492 | F1(0.5): 0.9417 | Threshold: 0.8443 | TP:  374 | FP:   14 | FN:   26 | TN:  325 | AvgSum: 30.93
Protein DKC1 (25): AUROC = 0.9833 | F1: 0.9447 | F1(0.5): 0.9427 | Threshold: 0.5420 | TP:  367 | FP:   10 | FN:   33 | TN:  314 | AvgSum: 33.30
Protein NOLC1 (71): AUROC = 0.9825 | F1: 0.9463 | F1(0.5): 0.9428 | Threshold: 0.3560 | TP:  388 | FP:   34 | FN:   10 | TN:  294 | AvgSum: 37.58
[INFO] AUROC Bottom 10 proteins:
Protein IGF2BP2 (58): AUROC = 0.8115 | F1: 0.7970 | F1(0.5): 0.7715 | Threshold: 0.1413 | TP:  373 | FP:  163 | FN:   27 | TN:  161 | AvgSum: 34.45
Protein FXR2 (45): AUROC = 0.7905 | F1: 0.7772 | F1(0.5): 0.7682 | Threshold: 0.2699 | TP:  368 | FP:  179 | FN:   32 | TN:  140 | AvgSum: 33.01
Protein G3BP1 (46): AUROC = 0.7862 | F1: 0.7429 | F1(0.5): 0.7100 | Threshold: 0.0093 | TP:  104 | FP:   60 | FN:   12 | TN:   47 | AvgSum: 32.86
Protein KHSRP (62): AUROC = 0.7818 | F1: 0.7624 | F1(0.5): 0.7561 | Threshold: 0.3791 | TP:  337 | FP:  147 | FN:   63 | TN:  189 | AvgSum: 28.64
Protein ILF3 (60): AUROC = 0.7812 | F1: 0.7411 | F1(0.5): 0.7252 | Threshold: 0.7834 | TP:  209 | FP:  110 | FN:   36 | TN:  138 | AvgSum: 16.23
Protein RYBP (92): AUROC = 0.7805 | F1: 0.7203 | F1(0.5): 0.6667 | Threshold: 0.0833 | TP:   85 | FP:   46 | FN:   20 | TN:   70 | AvgSum: 25.48
Protein GPKOW (49): AUROC = 0.7703 | F1: 0.7409 | F1(0.5): 0.7200 | Threshold: 0.1961 | TP:  193 | FP:  115 | FN:   20 | TN:   88 | AvgSum: 41.74
Protein PPIG (78): AUROC = 0.7690 | F1: 0.7273 | F1(0.5): 0.6640 | Threshold: 0.0262 | TP:  120 | FP:   78 | FN:   12 | TN:   60 | AvgSum: 27.46
Protein CPSF6 (12): AUROC = 0.7651 | F1: 0.7489 | F1(0.5): 0.7332 | Threshold: 0.3707 | TP:  328 | FP:  148 | FN:   72 | TN:  196 | AvgSum: 24.06
Protein IGF2BP1 (57): AUROC = 0.7560 | F1: 0.7602 | F1(0.5): 0.7424 | Threshold: 0.1087 | TP:  371 | FP:  205 | FN:   29 | TN:   89 | AvgSum: 34.75

Epoch 16/30
Loss: 0.9898 | AUROC: 0.9127 | F1: 0.8675 | F1(0.5): 0.8468 | mAP: 0.9276 | AvgSum: 27.3264 | Train time: 92.14s | Eval time: 8.36s
Average threshold: 0.31219926
Thresholds (min/max): 0.0062490334 0.8064947
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9976 | F1: 0.9813 | F1(0.5): 0.9786 | Threshold: 0.2535 | TP:  393 | FP:    8 | FN:    7 | TN:  319 | AvgSum: 27.02
Protein GARS (47): AUROC = 0.9949 | F1: 0.9751 | F1(0.5): 0.9492 | Threshold: 0.0379 | TP:  391 | FP:   11 | FN:    9 | TN:  331 | AvgSum: 29.28
Protein ADAT1 (2): AUROC = 0.9948 | F1: 0.9774 | F1(0.5): 0.9654 | Threshold: 0.1581 | TP:  389 | FP:    7 | FN:   11 | TN:  316 | AvgSum: 24.47
Protein FASTKD2 (38): AUROC = 0.9921 | F1: 0.9698 | F1(0.5): 0.9625 | Threshold: 0.1798 | TP:  289 | FP:   15 | FN:    3 | TN:  268 | AvgSum: 36.36
Protein AKAP1 (4): AUROC = 0.9913 | F1: 0.9726 | F1(0.5): 0.9698 | Threshold: 0.1913 | TP:  391 | FP:   13 | FN:    9 | TN:  319 | AvgSum: 30.96
Protein MTPAP (69): AUROC = 0.9906 | F1: 0.9690 | F1(0.5): 0.9650 | Threshold: 0.3393 | TP:  391 | FP:   16 | FN:    9 | TN:  328 | AvgSum: 33.47
Protein PCBP2 (76): AUROC = 0.9869 | F1: 0.9507 | F1(0.5): 0.9077 | Threshold: 0.1153 | TP:  135 | FP:    9 | FN:    5 | TN:  125 | AvgSum: 32.41
Protein GRWD1 (51): AUROC = 0.9830 | F1: 0.9509 | F1(0.5): 0.9479 | Threshold: 0.3555 | TP:  387 | FP:   27 | FN:   13 | TN:  310 | AvgSum: 33.67
Protein METAP2 (67): AUROC = 0.9830 | F1: 0.9412 | F1(0.5): 0.9181 | Threshold: 0.1492 | TP:  168 | FP:    9 | FN:   12 | TN:  188 | AvgSum: 29.37
Protein CSTF2T (14): AUROC = 0.9824 | F1: 0.9518 | F1(0.5): 0.9495 | Threshold: 0.5300 | TP:  395 | FP:   35 | FN:    5 | TN:  315 | AvgSum: 37.08
[INFO] AUROC Bottom 10 proteins:
Protein FXR2 (45): AUROC = 0.8011 | F1: 0.7882 | F1(0.5): 0.7856 | Threshold: 0.6859 | TP:  333 | FP:  112 | FN:   67 | TN:  207 | AvgSum: 28.77
Protein G3BP1 (46): AUROC = 0.7985 | F1: 0.7402 | F1(0.5): 0.6559 | Threshold: 0.0156 | TP:   94 | FP:   44 | FN:   22 | TN:   63 | AvgSum: 29.18
Protein IGF2BP2 (58): AUROC = 0.7985 | F1: 0.7956 | F1(0.5): 0.7468 | Threshold: 0.1514 | TP:  362 | FP:  148 | FN:   38 | TN:  176 | AvgSum: 29.75
Protein PPIG (78): AUROC = 0.7946 | F1: 0.7395 | F1(0.5): 0.6606 | Threshold: 0.0256 | TP:  115 | FP:   64 | FN:   17 | TN:   74 | AvgSum: 24.02
Protein KHSRP (62): AUROC = 0.7945 | F1: 0.7731 | F1(0.5): 0.7694 | Threshold: 0.4329 | TP:  339 | FP:  138 | FN:   61 | TN:  198 | AvgSum: 24.83
Protein GPKOW (49): AUROC = 0.7909 | F1: 0.7561 | F1(0.5): 0.6969 | Threshold: 0.1335 | TP:  186 | FP:   93 | FN:   27 | TN:  110 | AvgSum: 37.15
Protein CPSF6 (12): AUROC = 0.7773 | F1: 0.7536 | F1(0.5): 0.7488 | Threshold: 0.0820 | TP:  370 | FP:  212 | FN:   30 | TN:  132 | AvgSum: 20.87
Protein ILF3 (60): AUROC = 0.7623 | F1: 0.7278 | F1(0.5): 0.7153 | Threshold: 0.1142 | TP:  234 | FP:  164 | FN:   11 | TN:   84 | AvgSum: 14.41
Protein RYBP (92): AUROC = 0.7615 | F1: 0.7068 | F1(0.5): 0.5955 | Threshold: 0.0415 | TP:   88 | FP:   56 | FN:   17 | TN:   60 | AvgSum: 22.28
Protein IGF2BP1 (57): AUROC = 0.7525 | F1: 0.7623 | F1(0.5): 0.7108 | Threshold: 0.0484 | TP:  372 | FP:  204 | FN:   28 | TN:   90 | AvgSum: 30.22

Epoch 17/30
Loss: 0.8948 | AUROC: 0.9135 | F1: 0.8674 | F1(0.5): 0.8392 | mAP: 0.9296 | AvgSum: 24.2486 | Train time: 92.52s | Eval time: 8.38s
Average threshold: 0.24508049
Thresholds (min/max): 0.00051728945 0.8045142
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9981 | F1: 0.9888 | F1(0.5): 0.9785 | Threshold: 0.3715 | TP:  397 | FP:    6 | FN:    3 | TN:  321 | AvgSum: 24.34
Protein GARS (47): AUROC = 0.9958 | F1: 0.9763 | F1(0.5): 0.9490 | Threshold: 0.0222 | TP:  392 | FP:   11 | FN:    8 | TN:  331 | AvgSum: 25.76
Protein ADAT1 (2): AUROC = 0.9942 | F1: 0.9774 | F1(0.5): 0.9628 | Threshold: 0.1193 | TP:  390 | FP:    8 | FN:   10 | TN:  315 | AvgSum: 22.22
Protein FASTKD2 (38): AUROC = 0.9915 | F1: 0.9691 | F1(0.5): 0.9673 | Threshold: 0.4838 | TP:  282 | FP:    8 | FN:   10 | TN:  275 | AvgSum: 32.98
Protein AKAP1 (4): AUROC = 0.9912 | F1: 0.9726 | F1(0.5): 0.9670 | Threshold: 0.1615 | TP:  391 | FP:   13 | FN:    9 | TN:  319 | AvgSum: 28.22
Protein MTPAP (69): AUROC = 0.9903 | F1: 0.9703 | F1(0.5): 0.9635 | Threshold: 0.2315 | TP:  392 | FP:   16 | FN:    8 | TN:  328 | AvgSum: 30.20
Protein PCBP2 (76): AUROC = 0.9886 | F1: 0.9496 | F1(0.5): 0.9062 | Threshold: 0.0645 | TP:  132 | FP:    6 | FN:    8 | TN:  128 | AvgSum: 28.82
Protein GRWD1 (51): AUROC = 0.9840 | F1: 0.9496 | F1(0.5): 0.9373 | Threshold: 0.2714 | TP:  386 | FP:   27 | FN:   14 | TN:  310 | AvgSum: 30.50
Protein TAF15 (106): AUROC = 0.9833 | F1: 0.9459 | F1(0.5): 0.9343 | Threshold: 0.7136 | TP:  376 | FP:   19 | FN:   24 | TN:  320 | AvgSum: 24.16
Protein NOLC1 (71): AUROC = 0.9831 | F1: 0.9488 | F1(0.5): 0.9410 | Threshold: 0.2698 | TP:  380 | FP:   23 | FN:   18 | TN:  305 | AvgSum: 30.19
[INFO] AUROC Bottom 10 proteins:
Protein IGF2BP2 (58): AUROC = 0.8021 | F1: 0.7983 | F1(0.5): 0.7383 | Threshold: 0.0599 | TP:  374 | FP:  163 | FN:   26 | TN:  161 | AvgSum: 26.30
Protein PPIG (78): AUROC = 0.8003 | F1: 0.7526 | F1(0.5): 0.6368 | Threshold: 0.0412 | TP:  108 | FP:   47 | FN:   24 | TN:   91 | AvgSum: 20.81
Protein G3BP1 (46): AUROC = 0.8000 | F1: 0.7525 | F1(0.5): 0.6702 | Threshold: 0.0005 | TP:  111 | FP:   68 | FN:    5 | TN:   39 | AvgSum: 25.07
Protein FXR2 (45): AUROC = 0.7978 | F1: 0.7884 | F1(0.5): 0.7796 | Threshold: 0.3670 | TP:  352 | FP:  141 | FN:   48 | TN:  178 | AvgSum: 25.86
Protein KHSRP (62): AUROC = 0.7956 | F1: 0.7661 | F1(0.5): 0.7620 | Threshold: 0.1792 | TP:  344 | FP:  154 | FN:   56 | TN:  182 | AvgSum: 21.98
Protein RYBP (92): AUROC = 0.7891 | F1: 0.7296 | F1(0.5): 0.5521 | Threshold: 0.0440 | TP:   85 | FP:   43 | FN:   20 | TN:   73 | AvgSum: 19.09
Protein CPSF6 (12): AUROC = 0.7846 | F1: 0.7561 | F1(0.5): 0.7412 | Threshold: 0.0824 | TP:  355 | FP:  184 | FN:   45 | TN:  160 | AvgSum: 18.24
Protein GPKOW (49): AUROC = 0.7798 | F1: 0.7447 | F1(0.5): 0.6768 | Threshold: 0.0221 | TP:  194 | FP:  114 | FN:   19 | TN:   89 | AvgSum: 32.95
Protein ILF3 (60): AUROC = 0.7778 | F1: 0.7446 | F1(0.5): 0.7273 | Threshold: 0.6166 | TP:  207 | FP:  104 | FN:   38 | TN:  144 | AvgSum: 12.65
Protein IGF2BP1 (57): AUROC = 0.7592 | F1: 0.7628 | F1(0.5): 0.7160 | Threshold: 0.0106 | TP:  386 | FP:  226 | FN:   14 | TN:   68 | AvgSum: 26.48

Epoch 18/30
Loss: 0.8454 | AUROC: 0.9117 | F1: 0.8666 | F1(0.5): 0.8369 | mAP: 0.9279 | AvgSum: 23.7836 | Train time: 92.28s | Eval time: 8.40s
Average threshold: 0.23706256
Thresholds (min/max): 0.0008540444 0.89057887
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9975 | F1: 0.9801 | F1(0.5): 0.9773 | Threshold: 0.2444 | TP:  394 | FP:   10 | FN:    6 | TN:  317 | AvgSum: 24.57
Protein GARS (47): AUROC = 0.9946 | F1: 0.9684 | F1(0.5): 0.9504 | Threshold: 0.1173 | TP:  383 | FP:    8 | FN:   17 | TN:  334 | AvgSum: 24.72
Protein ADAT1 (2): AUROC = 0.9942 | F1: 0.9761 | F1(0.5): 0.9654 | Threshold: 0.1714 | TP:  388 | FP:    7 | FN:   12 | TN:  316 | AvgSum: 22.29
Protein MTPAP (69): AUROC = 0.9911 | F1: 0.9677 | F1(0.5): 0.9662 | Threshold: 0.3738 | TP:  390 | FP:   16 | FN:   10 | TN:  328 | AvgSum: 29.18
Protein AKAP1 (4): AUROC = 0.9909 | F1: 0.9738 | F1(0.5): 0.9634 | Threshold: 0.2491 | TP:  391 | FP:   12 | FN:    9 | TN:  320 | AvgSum: 27.48
Protein PCBP2 (76): AUROC = 0.9894 | F1: 0.9534 | F1(0.5): 0.9195 | Threshold: 0.1104 | TP:  133 | FP:    6 | FN:    7 | TN:  128 | AvgSum: 28.32
Protein FASTKD2 (38): AUROC = 0.9887 | F1: 0.9764 | F1(0.5): 0.9673 | Threshold: 0.2427 | TP:  289 | FP:   11 | FN:    3 | TN:  272 | AvgSum: 31.95
Protein TAF15 (106): AUROC = 0.9836 | F1: 0.9383 | F1(0.5): 0.9317 | Threshold: 0.8906 | TP:  365 | FP:   13 | FN:   35 | TN:  326 | AvgSum: 24.42
Protein METAP2 (67): AUROC = 0.9830 | F1: 0.9370 | F1(0.5): 0.9086 | Threshold: 0.0564 | TP:  171 | FP:   14 | FN:    9 | TN:  183 | AvgSum: 25.49
Protein U2AF2 (112): AUROC = 0.9818 | F1: 0.9416 | F1(0.5): 0.9362 | Threshold: 0.3043 | TP:  379 | FP:   26 | FN:   21 | TN:  299 | AvgSum: 20.68
[INFO] AUROC Bottom 10 proteins:
Protein IGF2BP2 (58): AUROC = 0.8063 | F1: 0.7996 | F1(0.5): 0.7302 | Threshold: 0.0660 | TP:  363 | FP:  145 | FN:   37 | TN:  179 | AvgSum: 25.62
Protein FXR2 (45): AUROC = 0.8042 | F1: 0.7921 | F1(0.5): 0.7849 | Threshold: 0.3257 | TP:  339 | FP:  117 | FN:   61 | TN:  202 | AvgSum: 25.08
Protein PPIG (78): AUROC = 0.8004 | F1: 0.7398 | F1(0.5): 0.6425 | Threshold: 0.0090 | TP:  118 | FP:   69 | FN:   14 | TN:   69 | AvgSum: 21.08
Protein G3BP1 (46): AUROC = 0.7958 | F1: 0.7616 | F1(0.5): 0.6559 | Threshold: 0.0014 | TP:  107 | FP:   58 | FN:    9 | TN:   49 | AvgSum: 24.36
Protein KHSRP (62): AUROC = 0.7912 | F1: 0.7768 | F1(0.5): 0.7600 | Threshold: 0.1483 | TP:  355 | FP:  159 | FN:   45 | TN:  177 | AvgSum: 21.60
Protein GPKOW (49): AUROC = 0.7855 | F1: 0.7505 | F1(0.5): 0.6751 | Threshold: 0.0243 | TP:  197 | FP:  115 | FN:   16 | TN:   88 | AvgSum: 32.10
Protein CPSF6 (12): AUROC = 0.7723 | F1: 0.7497 | F1(0.5): 0.7360 | Threshold: 0.1390 | TP:  352 | FP:  187 | FN:   48 | TN:  157 | AvgSum: 18.12
Protein RYBP (92): AUROC = 0.7671 | F1: 0.7164 | F1(0.5): 0.5576 | Threshold: 0.0060 | TP:   96 | FP:   67 | FN:    9 | TN:   49 | AvgSum: 19.09
Protein ILF3 (60): AUROC = 0.7645 | F1: 0.7356 | F1(0.5): 0.7295 | Threshold: 0.2109 | TP:  224 | FP:  140 | FN:   21 | TN:  108 | AvgSum: 12.64
Protein IGF2BP1 (57): AUROC = 0.7541 | F1: 0.7639 | F1(0.5): 0.6965 | Threshold: 0.0538 | TP:  351 | FP:  168 | FN:   49 | TN:  126 | AvgSum: 25.89

Epoch 19/30
Loss: 0.8136 | AUROC: 0.9110 | F1: 0.8646 | F1(0.5): 0.8361 | mAP: 0.9266 | AvgSum: 23.8127 | Train time: 92.27s | Eval time: 8.40s
Average threshold: 0.22953473
Thresholds (min/max): 0.0016809609 0.7222387
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9970 | F1: 0.9789 | F1(0.5): 0.9747 | Threshold: 0.0689 | TP:  394 | FP:   11 | FN:    6 | TN:  316 | AvgSum: 24.98
Protein GARS (47): AUROC = 0.9935 | F1: 0.9700 | F1(0.5): 0.9393 | Threshold: 0.0333 | TP:  388 | FP:   12 | FN:   12 | TN:  330 | AvgSum: 24.81
Protein ADAT1 (2): AUROC = 0.9923 | F1: 0.9711 | F1(0.5): 0.9600 | Threshold: 0.0861 | TP:  387 | FP:   10 | FN:   13 | TN:  313 | AvgSum: 21.98
Protein FASTKD2 (38): AUROC = 0.9915 | F1: 0.9695 | F1(0.5): 0.9640 | Threshold: 0.1958 | TP:  286 | FP:   12 | FN:    6 | TN:  271 | AvgSum: 32.12
Protein AKAP1 (4): AUROC = 0.9906 | F1: 0.9691 | F1(0.5): 0.9659 | Threshold: 0.1329 | TP:  392 | FP:   17 | FN:    8 | TN:  315 | AvgSum: 27.61
Protein MTPAP (69): AUROC = 0.9901 | F1: 0.9713 | F1(0.5): 0.9646 | Threshold: 0.3664 | TP:  389 | FP:   12 | FN:   11 | TN:  332 | AvgSum: 29.38
Protein PCBP2 (76): AUROC = 0.9883 | F1: 0.9433 | F1(0.5): 0.9062 | Threshold: 0.0816 | TP:  133 | FP:    9 | FN:    7 | TN:  125 | AvgSum: 28.36
Protein CSTF2T (14): AUROC = 0.9831 | F1: 0.9481 | F1(0.5): 0.9440 | Threshold: 0.6254 | TP:  384 | FP:   26 | FN:   16 | TN:  324 | AvgSum: 32.74
Protein METAP2 (67): AUROC = 0.9826 | F1: 0.9301 | F1(0.5): 0.8889 | Threshold: 0.0194 | TP:  173 | FP:   19 | FN:    7 | TN:  178 | AvgSum: 25.54
Protein NOLC1 (71): AUROC = 0.9819 | F1: 0.9440 | F1(0.5): 0.9394 | Threshold: 0.6479 | TP:  371 | FP:   17 | FN:   27 | TN:  311 | AvgSum: 29.73
[INFO] AUROC Bottom 10 proteins:
Protein G3BP1 (46): AUROC = 0.8020 | F1: 0.7500 | F1(0.5): 0.6489 | Threshold: 0.0017 | TP:  105 | FP:   59 | FN:   11 | TN:   48 | AvgSum: 24.45
Protein FXR2 (45): AUROC = 0.7967 | F1: 0.7844 | F1(0.5): 0.7735 | Threshold: 0.3367 | TP:  351 | FP:  144 | FN:   49 | TN:  175 | AvgSum: 25.32
Protein PPIG (78): AUROC = 0.7951 | F1: 0.7533 | F1(0.5): 0.6455 | Threshold: 0.0280 | TP:  113 | FP:   55 | FN:   19 | TN:   83 | AvgSum: 20.80
Protein RYBP (92): AUROC = 0.7929 | F1: 0.7266 | F1(0.5): 0.5799 | Threshold: 0.0105 | TP:   93 | FP:   58 | FN:   12 | TN:   58 | AvgSum: 19.03
Protein IGF2BP2 (58): AUROC = 0.7878 | F1: 0.7796 | F1(0.5): 0.7356 | Threshold: 0.0375 | TP:  366 | FP:  173 | FN:   34 | TN:  151 | AvgSum: 25.96
Protein KHSRP (62): AUROC = 0.7838 | F1: 0.7680 | F1(0.5): 0.7540 | Threshold: 0.2062 | TP:  336 | FP:  139 | FN:   64 | TN:  197 | AvgSum: 21.65
Protein CPSF6 (12): AUROC = 0.7757 | F1: 0.7529 | F1(0.5): 0.7315 | Threshold: 0.0727 | TP:  361 | FP:  198 | FN:   39 | TN:  146 | AvgSum: 17.86
Protein GPKOW (49): AUROC = 0.7728 | F1: 0.7368 | F1(0.5): 0.6888 | Threshold: 0.0179 | TP:  196 | FP:  123 | FN:   17 | TN:   80 | AvgSum: 32.33
Protein ILF3 (60): AUROC = 0.7615 | F1: 0.7319 | F1(0.5): 0.7177 | Threshold: 0.0598 | TP:  228 | FP:  150 | FN:   17 | TN:   98 | AvgSum: 12.67
Protein IGF2BP1 (57): AUROC = 0.7590 | F1: 0.7733 | F1(0.5): 0.7115 | Threshold: 0.0890 | TP:  353 | FP:  160 | FN:   47 | TN:  134 | AvgSum: 25.80

Epoch 20/30
Loss: 0.7861 | AUROC: 0.9107 | F1: 0.8649 | F1(0.5): 0.8336 | mAP: 0.9261 | AvgSum: 22.8517 | Train time: 92.21s | Eval time: 8.38s
Average threshold: 0.21518774
Thresholds (min/max): 0.00536555 0.87609875
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9974 | F1: 0.9839 | F1(0.5): 0.9735 | Threshold: 0.0669 | TP:  397 | FP:   10 | FN:    3 | TN:  317 | AvgSum: 24.63
Protein GARS (47): AUROC = 0.9941 | F1: 0.9727 | F1(0.5): 0.9407 | Threshold: 0.0188 | TP:  392 | FP:   14 | FN:    8 | TN:  328 | AvgSum: 24.00
Protein ADAT1 (2): AUROC = 0.9931 | F1: 0.9773 | F1(0.5): 0.9587 | Threshold: 0.1172 | TP:  388 | FP:    6 | FN:   12 | TN:  317 | AvgSum: 21.67
Protein AKAP1 (4): AUROC = 0.9917 | F1: 0.9727 | F1(0.5): 0.9711 | Threshold: 0.1723 | TP:  392 | FP:   14 | FN:    8 | TN:  318 | AvgSum: 27.05
Protein FASTKD2 (38): AUROC = 0.9917 | F1: 0.9713 | F1(0.5): 0.9677 | Threshold: 0.3499 | TP:  288 | FP:   13 | FN:    4 | TN:  270 | AvgSum: 31.07
Protein MTPAP (69): AUROC = 0.9910 | F1: 0.9702 | F1(0.5): 0.9662 | Threshold: 0.4119 | TP:  391 | FP:   15 | FN:    9 | TN:  329 | AvgSum: 28.59
Protein PCBP2 (76): AUROC = 0.9871 | F1: 0.9496 | F1(0.5): 0.9154 | Threshold: 0.0930 | TP:  132 | FP:    6 | FN:    8 | TN:  128 | AvgSum: 27.43
Protein DGCR8 (23): AUROC = 0.9847 | F1: 0.9427 | F1(0.5): 0.9197 | Threshold: 0.0420 | TP:  181 | FP:   10 | FN:   12 | TN:  173 | AvgSum: 25.09
Protein NOLC1 (71): AUROC = 0.9827 | F1: 0.9514 | F1(0.5): 0.9501 | Threshold: 0.4700 | TP:  382 | FP:   23 | FN:   16 | TN:  305 | AvgSum: 28.48
Protein METAP2 (67): AUROC = 0.9823 | F1: 0.9355 | F1(0.5): 0.8949 | Threshold: 0.0105 | TP:  174 | FP:   18 | FN:    6 | TN:  179 | AvgSum: 24.46
[INFO] AUROC Bottom 10 proteins:
Protein G3BP1 (46): AUROC = 0.8099 | F1: 0.7521 | F1(0.5): 0.6593 | Threshold: 0.0120 | TP:   91 | FP:   35 | FN:   25 | TN:   72 | AvgSum: 23.14
Protein PPIG (78): AUROC = 0.7977 | F1: 0.7461 | F1(0.5): 0.6055 | Threshold: 0.0054 | TP:  119 | FP:   68 | FN:   13 | TN:   70 | AvgSum: 20.20
Protein FXR2 (45): AUROC = 0.7906 | F1: 0.7858 | F1(0.5): 0.7759 | Threshold: 0.3615 | TP:  343 | FP:  130 | FN:   57 | TN:  189 | AvgSum: 23.88
Protein IGF2BP2 (58): AUROC = 0.7903 | F1: 0.7798 | F1(0.5): 0.7067 | Threshold: 0.0143 | TP:  370 | FP:  179 | FN:   30 | TN:  145 | AvgSum: 24.73
Protein KHSRP (62): AUROC = 0.7880 | F1: 0.7687 | F1(0.5): 0.7268 | Threshold: 0.0341 | TP:  359 | FP:  175 | FN:   41 | TN:  161 | AvgSum: 20.47
Protein RYBP (92): AUROC = 0.7841 | F1: 0.7227 | F1(0.5): 0.5868 | Threshold: 0.0157 | TP:   86 | FP:   47 | FN:   19 | TN:   69 | AvgSum: 18.07
Protein CPSF6 (12): AUROC = 0.7794 | F1: 0.7537 | F1(0.5): 0.7372 | Threshold: 0.2331 | TP:  329 | FP:  144 | FN:   71 | TN:  200 | AvgSum: 17.11
Protein GPKOW (49): AUROC = 0.7780 | F1: 0.7510 | F1(0.5): 0.6799 | Threshold: 0.0747 | TP:  184 | FP:   93 | FN:   29 | TN:  110 | AvgSum: 31.56
Protein ILF3 (60): AUROC = 0.7748 | F1: 0.7368 | F1(0.5): 0.7166 | Threshold: 0.2432 | TP:  217 | FP:  127 | FN:   28 | TN:  121 | AvgSum: 11.81
Protein IGF2BP1 (57): AUROC = 0.7502 | F1: 0.7593 | F1(0.5): 0.6958 | Threshold: 0.0070 | TP:  377 | FP:  216 | FN:   23 | TN:   78 | AvgSum: 25.00

Epoch 21/30
Loss: 0.7212 | AUROC: 0.9121 | F1: 0.8663 | F1(0.5): 0.8225 | mAP: 0.9286 | AvgSum: 20.3532 | Train time: 92.23s | Eval time: 8.37s
Average threshold: 0.1596823
Thresholds (min/max): 3.488309e-05 0.7264948
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9977 | F1: 0.9851 | F1(0.5): 0.9773 | Threshold: 0.0929 | TP:  396 | FP:    8 | FN:    4 | TN:  319 | AvgSum: 21.72
Protein GARS (47): AUROC = 0.9949 | F1: 0.9703 | F1(0.5): 0.9278 | Threshold: 0.0064 | TP:  392 | FP:   16 | FN:    8 | TN:  326 | AvgSum: 21.81
Protein ADAT1 (2): AUROC = 0.9937 | F1: 0.9773 | F1(0.5): 0.9600 | Threshold: 0.0893 | TP:  388 | FP:    6 | FN:   12 | TN:  317 | AvgSum: 19.27
Protein FASTKD2 (38): AUROC = 0.9925 | F1: 0.9714 | F1(0.5): 0.9622 | Threshold: 0.1698 | TP:  289 | FP:   14 | FN:    3 | TN:  269 | AvgSum: 28.40
Protein MTPAP (69): AUROC = 0.9913 | F1: 0.9737 | F1(0.5): 0.9698 | Threshold: 0.3368 | TP:  389 | FP:   10 | FN:   11 | TN:  334 | AvgSum: 26.05
Protein AKAP1 (4): AUROC = 0.9913 | F1: 0.9714 | F1(0.5): 0.9672 | Threshold: 0.1429 | TP:  391 | FP:   14 | FN:    9 | TN:  318 | AvgSum: 24.45
Protein PCBP2 (76): AUROC = 0.9899 | F1: 0.9668 | F1(0.5): 0.8710 | Threshold: 0.1164 | TP:  131 | FP:    0 | FN:    9 | TN:  134 | AvgSum: 24.53
Protein DGCR8 (23): AUROC = 0.9843 | F1: 0.9499 | F1(0.5): 0.8920 | Threshold: 0.0551 | TP:  180 | FP:    6 | FN:   13 | TN:  177 | AvgSum: 22.65
Protein CSTF2T (14): AUROC = 0.9842 | F1: 0.9510 | F1(0.5): 0.9476 | Threshold: 0.5488 | TP:  388 | FP:   28 | FN:   12 | TN:  322 | AvgSum: 28.68
Protein GRWD1 (51): AUROC = 0.9834 | F1: 0.9446 | F1(0.5): 0.9305 | Threshold: 0.1762 | TP:  384 | FP:   29 | FN:   16 | TN:  308 | AvgSum: 26.21
[INFO] AUROC Bottom 10 proteins:
Protein G3BP1 (46): AUROC = 0.8061 | F1: 0.7550 | F1(0.5): 0.6409 | Threshold: 0.0000 | TP:  114 | FP:   72 | FN:    2 | TN:   35 | AvgSum: 20.72
Protein IGF2BP2 (58): AUROC = 0.8020 | F1: 0.7850 | F1(0.5): 0.6829 | Threshold: 0.0487 | TP:  345 | FP:  134 | FN:   55 | TN:  190 | AvgSum: 21.65
Protein FXR2 (45): AUROC = 0.7959 | F1: 0.7863 | F1(0.5): 0.7679 | Threshold: 0.2000 | TP:  344 | FP:  131 | FN:   56 | TN:  188 | AvgSum: 21.50
Protein PPIG (78): AUROC = 0.7951 | F1: 0.7537 | F1(0.5): 0.5473 | Threshold: 0.0008 | TP:  127 | FP:   78 | FN:    5 | TN:   60 | AvgSum: 17.84
Protein KHSRP (62): AUROC = 0.7913 | F1: 0.7689 | F1(0.5): 0.7302 | Threshold: 0.2684 | TP:  311 | FP:   98 | FN:   89 | TN:  238 | AvgSum: 18.18
Protein RYBP (92): AUROC = 0.7886 | F1: 0.7225 | F1(0.5): 0.5513 | Threshold: 0.0168 | TP:   82 | FP:   40 | FN:   23 | TN:   76 | AvgSum: 15.71
Protein CPSF6 (12): AUROC = 0.7804 | F1: 0.7572 | F1(0.5): 0.7134 | Threshold: 0.0783 | TP:  343 | FP:  163 | FN:   57 | TN:  181 | AvgSum: 14.65
Protein GPKOW (49): AUROC = 0.7781 | F1: 0.7410 | F1(0.5): 0.6684 | Threshold: 0.0055 | TP:  196 | FP:  120 | FN:   17 | TN:   83 | AvgSum: 28.21
Protein ILF3 (60): AUROC = 0.7699 | F1: 0.7376 | F1(0.5): 0.7236 | Threshold: 0.3015 | TP:  208 | FP:  111 | FN:   37 | TN:  137 | AvgSum: 10.08
Protein IGF2BP1 (57): AUROC = 0.7448 | F1: 0.7568 | F1(0.5): 0.6686 | Threshold: 0.0094 | TP:  361 | FP:  193 | FN:   39 | TN:  101 | AvgSum: 22.00

Epoch 22/30
Loss: 0.6857 | AUROC: 0.9118 | F1: 0.8656 | F1(0.5): 0.8176 | mAP: 0.9278 | AvgSum: 19.5363 | Train time: 92.35s | Eval time: 8.40s
Average threshold: 0.13146977
Thresholds (min/max): 0.0014866889 0.767245
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9972 | F1: 0.9812 | F1(0.5): 0.9759 | Threshold: 0.2968 | TP:  392 | FP:    7 | FN:    8 | TN:  320 | AvgSum: 20.34
Protein GARS (47): AUROC = 0.9948 | F1: 0.9690 | F1(0.5): 0.9264 | Threshold: 0.0056 | TP:  391 | FP:   16 | FN:    9 | TN:  326 | AvgSum: 21.09
Protein FASTKD2 (38): AUROC = 0.9936 | F1: 0.9699 | F1(0.5): 0.9656 | Threshold: 0.0455 | TP:  290 | FP:   16 | FN:    2 | TN:  267 | AvgSum: 27.75
Protein ADAT1 (2): AUROC = 0.9935 | F1: 0.9773 | F1(0.5): 0.9573 | Threshold: 0.0888 | TP:  388 | FP:    6 | FN:   12 | TN:  317 | AvgSum: 18.30
Protein AKAP1 (4): AUROC = 0.9914 | F1: 0.9714 | F1(0.5): 0.9646 | Threshold: 0.1235 | TP:  391 | FP:   14 | FN:    9 | TN:  318 | AvgSum: 23.65
Protein MTPAP (69): AUROC = 0.9906 | F1: 0.9738 | F1(0.5): 0.9697 | Threshold: 0.2372 | TP:  390 | FP:   11 | FN:   10 | TN:  333 | AvgSum: 25.24
Protein PCBP2 (76): AUROC = 0.9906 | F1: 0.9556 | F1(0.5): 0.8618 | Threshold: 0.0773 | TP:  129 | FP:    1 | FN:   11 | TN:  133 | AvgSum: 23.62
Protein CSTF2T (14): AUROC = 0.9845 | F1: 0.9488 | F1(0.5): 0.9458 | Threshold: 0.3174 | TP:  389 | FP:   31 | FN:   11 | TN:  319 | AvgSum: 28.00
Protein DGCR8 (23): AUROC = 0.9839 | F1: 0.9496 | F1(0.5): 0.8832 | Threshold: 0.0770 | TP:  179 | FP:    5 | FN:   14 | TN:  178 | AvgSum: 21.96
Protein NOLC1 (71): AUROC = 0.9831 | F1: 0.9518 | F1(0.5): 0.9407 | Threshold: 0.1287 | TP:  385 | FP:   26 | FN:   13 | TN:  302 | AvgSum: 24.81
[INFO] AUROC Bottom 10 proteins:
Protein PPIG (78): AUROC = 0.8095 | F1: 0.7607 | F1(0.5): 0.5951 | Threshold: 0.0015 | TP:  124 | FP:   70 | FN:    8 | TN:   68 | AvgSum: 16.83
Protein G3BP1 (46): AUROC = 0.8044 | F1: 0.7534 | F1(0.5): 0.6067 | Threshold: 0.0292 | TP:   84 | FP:   23 | FN:   32 | TN:   84 | AvgSum: 19.82
Protein FXR2 (45): AUROC = 0.7940 | F1: 0.7824 | F1(0.5): 0.7692 | Threshold: 0.1185 | TP:  347 | FP:  140 | FN:   53 | TN:  179 | AvgSum: 20.96
Protein IGF2BP2 (58): AUROC = 0.7925 | F1: 0.7817 | F1(0.5): 0.6706 | Threshold: 0.0085 | TP:  367 | FP:  172 | FN:   33 | TN:  152 | AvgSum: 20.92
Protein KHSRP (62): AUROC = 0.7868 | F1: 0.7698 | F1(0.5): 0.7098 | Threshold: 0.1058 | TP:  326 | FP:  121 | FN:   74 | TN:  215 | AvgSum: 17.32
Protein RYBP (92): AUROC = 0.7813 | F1: 0.7109 | F1(0.5): 0.5478 | Threshold: 0.0321 | TP:   75 | FP:   31 | FN:   30 | TN:   85 | AvgSum: 14.89
Protein CPSF6 (12): AUROC = 0.7760 | F1: 0.7551 | F1(0.5): 0.7087 | Threshold: 0.1062 | TP:  333 | FP:  149 | FN:   67 | TN:  195 | AvgSum: 13.82
Protein ILF3 (60): AUROC = 0.7699 | F1: 0.7395 | F1(0.5): 0.7185 | Threshold: 0.0682 | TP:  230 | FP:  147 | FN:   15 | TN:  101 | AvgSum: 9.56
Protein GPKOW (49): AUROC = 0.7684 | F1: 0.7349 | F1(0.5): 0.6649 | Threshold: 0.0588 | TP:  176 | FP:   90 | FN:   37 | TN:  113 | AvgSum: 27.23
Protein IGF2BP1 (57): AUROC = 0.7448 | F1: 0.7571 | F1(0.5): 0.6657 | Threshold: 0.0045 | TP:  371 | FP:  209 | FN:   29 | TN:   85 | AvgSum: 21.02

Epoch 23/30
Loss: 0.6634 | AUROC: 0.9101 | F1: 0.8635 | F1(0.5): 0.8161 | mAP: 0.9268 | AvgSum: 19.4456 | Train time: 92.19s | Eval time: 8.34s
Average threshold: 0.1194087
Thresholds (min/max): 0.00047286367 0.55241346
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9972 | F1: 0.9801 | F1(0.5): 0.9748 | Threshold: 0.0357 | TP:  395 | FP:   11 | FN:    5 | TN:  316 | AvgSum: 20.53
Protein GARS (47): AUROC = 0.9943 | F1: 0.9698 | F1(0.5): 0.9349 | Threshold: 0.0159 | TP:  386 | FP:   10 | FN:   14 | TN:  332 | AvgSum: 20.68
Protein ADAT1 (2): AUROC = 0.9929 | F1: 0.9773 | F1(0.5): 0.9560 | Threshold: 0.0574 | TP:  388 | FP:    6 | FN:   12 | TN:  317 | AvgSum: 18.30
Protein FASTKD2 (38): AUROC = 0.9918 | F1: 0.9764 | F1(0.5): 0.9619 | Threshold: 0.1085 | TP:  290 | FP:   12 | FN:    2 | TN:  271 | AvgSum: 26.93
Protein MTPAP (69): AUROC = 0.9905 | F1: 0.9723 | F1(0.5): 0.9710 | Threshold: 0.4579 | TP:  386 | FP:    8 | FN:   14 | TN:  336 | AvgSum: 24.72
Protein AKAP1 (4): AUROC = 0.9902 | F1: 0.9677 | F1(0.5): 0.9634 | Threshold: 0.1223 | TP:  389 | FP:   15 | FN:   11 | TN:  317 | AvgSum: 23.30
Protein CSTF2T (14): AUROC = 0.9846 | F1: 0.9480 | F1(0.5): 0.9420 | Threshold: 0.2691 | TP:  392 | FP:   35 | FN:    8 | TN:  315 | AvgSum: 27.28
Protein PCBP2 (76): AUROC = 0.9836 | F1: 0.9407 | F1(0.5): 0.8755 | Threshold: 0.0889 | TP:  127 | FP:    3 | FN:   13 | TN:  131 | AvgSum: 23.61
Protein TAF15 (106): AUROC = 0.9828 | F1: 0.9425 | F1(0.5): 0.9391 | Threshold: 0.5524 | TP:  377 | FP:   23 | FN:   23 | TN:  316 | AvgSum: 19.76
Protein GRWD1 (51): AUROC = 0.9819 | F1: 0.9409 | F1(0.5): 0.9338 | Threshold: 0.3712 | TP:  374 | FP:   21 | FN:   26 | TN:  316 | AvgSum: 24.99
[INFO] AUROC Bottom 10 proteins:
Protein IGF2BP2 (58): AUROC = 0.7977 | F1: 0.7862 | F1(0.5): 0.6950 | Threshold: 0.0136 | TP:  364 | FP:  162 | FN:   36 | TN:  162 | AvgSum: 20.43
Protein G3BP1 (46): AUROC = 0.7941 | F1: 0.7458 | F1(0.5): 0.5989 | Threshold: 0.0097 | TP:   88 | FP:   32 | FN:   28 | TN:   75 | AvgSum: 19.53
Protein FXR2 (45): AUROC = 0.7923 | F1: 0.7786 | F1(0.5): 0.7584 | Threshold: 0.0240 | TP:  371 | FP:  182 | FN:   29 | TN:  137 | AvgSum: 20.69
Protein PPIG (78): AUROC = 0.7918 | F1: 0.7429 | F1(0.5): 0.6197 | Threshold: 0.0044 | TP:  117 | FP:   66 | FN:   15 | TN:   72 | AvgSum: 17.10
Protein KHSRP (62): AUROC = 0.7871 | F1: 0.7736 | F1(0.5): 0.7175 | Threshold: 0.0552 | TP:  340 | FP:  139 | FN:   60 | TN:  197 | AvgSum: 17.65
Protein RYBP (92): AUROC = 0.7814 | F1: 0.7176 | F1(0.5): 0.5385 | Threshold: 0.0019 | TP:   94 | FP:   63 | FN:   11 | TN:   53 | AvgSum: 14.89
Protein GPKOW (49): AUROC = 0.7787 | F1: 0.7414 | F1(0.5): 0.6558 | Threshold: 0.0609 | TP:  172 | FP:   79 | FN:   41 | TN:  124 | AvgSum: 26.53
Protein CPSF6 (12): AUROC = 0.7777 | F1: 0.7530 | F1(0.5): 0.7070 | Threshold: 0.0646 | TP:  346 | FP:  173 | FN:   54 | TN:  171 | AvgSum: 14.51
Protein ILF3 (60): AUROC = 0.7683 | F1: 0.7391 | F1(0.5): 0.7201 | Threshold: 0.0871 | TP:  228 | FP:  144 | FN:   17 | TN:  104 | AvgSum: 10.36
Protein IGF2BP1 (57): AUROC = 0.7489 | F1: 0.7602 | F1(0.5): 0.6570 | Threshold: 0.0380 | TP:  336 | FP:  148 | FN:   64 | TN:  146 | AvgSum: 20.55

Epoch 24/30
Loss: 0.6309 | AUROC: 0.9115 | F1: 0.8651 | F1(0.5): 0.8127 | mAP: 0.9276 | AvgSum: 18.6729 | Train time: 92.31s | Eval time: 8.31s
Average threshold: 0.116540216
Thresholds (min/max): 2.0349478e-05 0.6182935
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9973 | F1: 0.9815 | F1(0.5): 0.9772 | Threshold: 0.0245 | TP:  397 | FP:   12 | FN:    3 | TN:  315 | AvgSum: 20.20
Protein GARS (47): AUROC = 0.9945 | F1: 0.9701 | F1(0.5): 0.9220 | Threshold: 0.0049 | TP:  390 | FP:   14 | FN:   10 | TN:  328 | AvgSum: 20.20
Protein ADAT1 (2): AUROC = 0.9935 | F1: 0.9760 | F1(0.5): 0.9587 | Threshold: 0.0740 | TP:  386 | FP:    5 | FN:   14 | TN:  318 | AvgSum: 17.72
Protein FASTKD2 (38): AUROC = 0.9928 | F1: 0.9746 | F1(0.5): 0.9618 | Threshold: 0.1515 | TP:  288 | FP:   11 | FN:    4 | TN:  272 | AvgSum: 26.47
Protein AKAP1 (4): AUROC = 0.9907 | F1: 0.9714 | F1(0.5): 0.9659 | Threshold: 0.1305 | TP:  390 | FP:   13 | FN:   10 | TN:  319 | AvgSum: 22.80
Protein MTPAP (69): AUROC = 0.9902 | F1: 0.9727 | F1(0.5): 0.9684 | Threshold: 0.1316 | TP:  392 | FP:   14 | FN:    8 | TN:  330 | AvgSum: 24.25
Protein PCBP2 (76): AUROC = 0.9881 | F1: 0.9466 | F1(0.5): 0.8618 | Threshold: 0.0123 | TP:  133 | FP:    8 | FN:    7 | TN:  126 | AvgSum: 22.79
Protein CSTF2T (14): AUROC = 0.9850 | F1: 0.9496 | F1(0.5): 0.9467 | Threshold: 0.4797 | TP:  386 | FP:   27 | FN:   14 | TN:  323 | AvgSum: 26.71
Protein DGCR8 (23): AUROC = 0.9833 | F1: 0.9375 | F1(0.5): 0.8793 | Threshold: 0.0253 | TP:  180 | FP:   11 | FN:   13 | TN:  172 | AvgSum: 21.06
Protein TAF15 (106): AUROC = 0.9833 | F1: 0.9431 | F1(0.5): 0.9385 | Threshold: 0.6183 | TP:  373 | FP:   18 | FN:   27 | TN:  321 | AvgSum: 19.01
[INFO] AUROC Bottom 10 proteins:
Protein PPIG (78): AUROC = 0.8029 | F1: 0.7575 | F1(0.5): 0.5800 | Threshold: 0.0062 | TP:  114 | FP:   55 | FN:   18 | TN:   83 | AvgSum: 16.38
Protein G3BP1 (46): AUROC = 0.7985 | F1: 0.7417 | F1(0.5): 0.5780 | Threshold: 0.0000 | TP:  112 | FP:   74 | FN:    4 | TN:   33 | AvgSum: 18.80
Protein IGF2BP2 (58): AUROC = 0.7935 | F1: 0.7796 | F1(0.5): 0.6782 | Threshold: 0.0127 | TP:  359 | FP:  162 | FN:   41 | TN:  162 | AvgSum: 19.68
Protein FXR2 (45): AUROC = 0.7918 | F1: 0.7780 | F1(0.5): 0.7531 | Threshold: 0.0435 | TP:  361 | FP:  167 | FN:   39 | TN:  152 | AvgSum: 19.92
Protein KHSRP (62): AUROC = 0.7892 | F1: 0.7719 | F1(0.5): 0.7154 | Threshold: 0.0445 | TP:  340 | FP:  141 | FN:   60 | TN:  195 | AvgSum: 16.69
Protein CPSF6 (12): AUROC = 0.7815 | F1: 0.7576 | F1(0.5): 0.7079 | Threshold: 0.0781 | TP:  336 | FP:  151 | FN:   64 | TN:  193 | AvgSum: 13.33
Protein RYBP (92): AUROC = 0.7805 | F1: 0.7218 | F1(0.5): 0.5229 | Threshold: 0.0009 | TP:   96 | FP:   65 | FN:    9 | TN:   51 | AvgSum: 14.11
Protein GPKOW (49): AUROC = 0.7709 | F1: 0.7360 | F1(0.5): 0.6649 | Threshold: 0.0460 | TP:  177 | FP:   91 | FN:   36 | TN:  112 | AvgSum: 25.91
Protein ILF3 (60): AUROC = 0.7632 | F1: 0.7378 | F1(0.5): 0.7111 | Threshold: 0.0250 | TP:  235 | FP:  157 | FN:   10 | TN:   91 | AvgSum: 9.18
Protein IGF2BP1 (57): AUROC = 0.7488 | F1: 0.7569 | F1(0.5): 0.6753 | Threshold: 0.0036 | TP:  372 | FP:  211 | FN:   28 | TN:   83 | AvgSum: 19.91

Epoch 25/30
Loss: 0.6125 | AUROC: 0.9112 | F1: 0.8652 | F1(0.5): 0.8102 | mAP: 0.9276 | AvgSum: 18.3285 | Train time: 92.24s | Eval time: 8.41s
Average threshold: 0.110608034
Thresholds (min/max): 1.1943164e-05 0.7194091
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9976 | F1: 0.9812 | F1(0.5): 0.9772 | Threshold: 0.3366 | TP:  391 | FP:    6 | FN:    9 | TN:  321 | AvgSum: 20.15
Protein GARS (47): AUROC = 0.9944 | F1: 0.9688 | F1(0.5): 0.9292 | Threshold: 0.0059 | TP:  388 | FP:   13 | FN:   12 | TN:  329 | AvgSum: 19.85
Protein ADAT1 (2): AUROC = 0.9933 | F1: 0.9785 | F1(0.5): 0.9587 | Threshold: 0.1539 | TP:  386 | FP:    3 | FN:   14 | TN:  320 | AvgSum: 17.71
Protein FASTKD2 (38): AUROC = 0.9926 | F1: 0.9743 | F1(0.5): 0.9707 | Threshold: 0.4797 | TP:  284 | FP:    7 | FN:    8 | TN:  276 | AvgSum: 26.13
Protein AKAP1 (4): AUROC = 0.9908 | F1: 0.9713 | F1(0.5): 0.9645 | Threshold: 0.1750 | TP:  389 | FP:   12 | FN:   11 | TN:  320 | AvgSum: 22.55
Protein MTPAP (69): AUROC = 0.9904 | F1: 0.9738 | F1(0.5): 0.9672 | Threshold: 0.1690 | TP:  391 | FP:   12 | FN:    9 | TN:  332 | AvgSum: 23.96
Protein PCBP2 (76): AUROC = 0.9874 | F1: 0.9493 | F1(0.5): 0.8477 | Threshold: 0.0231 | TP:  131 | FP:    5 | FN:    9 | TN:  129 | AvgSum: 22.38
Protein CSTF2T (14): AUROC = 0.9857 | F1: 0.9492 | F1(0.5): 0.9420 | Threshold: 0.2138 | TP:  392 | FP:   34 | FN:    8 | TN:  316 | AvgSum: 26.44
Protein DGCR8 (23): AUROC = 0.9848 | F1: 0.9468 | F1(0.5): 0.8793 | Threshold: 0.0449 | TP:  178 | FP:    5 | FN:   15 | TN:  178 | AvgSum: 20.76
Protein GRWD1 (51): AUROC = 0.9837 | F1: 0.9434 | F1(0.5): 0.9352 | Threshold: 0.3456 | TP:  375 | FP:   20 | FN:   25 | TN:  317 | AvgSum: 24.05
[INFO] AUROC Bottom 10 proteins:
Protein PPIG (78): AUROC = 0.7994 | F1: 0.7556 | F1(0.5): 0.5584 | Threshold: 0.0015 | TP:  119 | FP:   64 | FN:   13 | TN:   74 | AvgSum: 15.79
Protein G3BP1 (46): AUROC = 0.7984 | F1: 0.7410 | F1(0.5): 0.6102 | Threshold: 0.0000 | TP:  113 | FP:   76 | FN:    3 | TN:   31 | AvgSum: 18.36
Protein IGF2BP2 (58): AUROC = 0.7945 | F1: 0.7783 | F1(0.5): 0.6676 | Threshold: 0.0015 | TP:  381 | FP:  198 | FN:   19 | TN:  126 | AvgSum: 19.25
Protein FXR2 (45): AUROC = 0.7929 | F1: 0.7832 | F1(0.5): 0.7526 | Threshold: 0.1418 | TP:  345 | FP:  136 | FN:   55 | TN:  183 | AvgSum: 19.52
Protein KHSRP (62): AUROC = 0.7894 | F1: 0.7705 | F1(0.5): 0.7059 | Threshold: 0.1031 | TP:  324 | FP:  117 | FN:   76 | TN:  219 | AvgSum: 16.17
Protein RYBP (92): AUROC = 0.7829 | F1: 0.7252 | F1(0.5): 0.5263 | Threshold: 0.0008 | TP:   95 | FP:   62 | FN:   10 | TN:   54 | AvgSum: 13.53
Protein CPSF6 (12): AUROC = 0.7796 | F1: 0.7551 | F1(0.5): 0.7101 | Threshold: 0.0127 | TP:  367 | FP:  205 | FN:   33 | TN:  139 | AvgSum: 13.07
Protein GPKOW (49): AUROC = 0.7675 | F1: 0.7299 | F1(0.5): 0.6542 | Threshold: 0.0327 | TP:  177 | FP:   95 | FN:   36 | TN:  108 | AvgSum: 25.64
Protein ILF3 (60): AUROC = 0.7665 | F1: 0.7402 | F1(0.5): 0.7178 | Threshold: 0.1646 | TP:  218 | FP:  126 | FN:   27 | TN:  122 | AvgSum: 9.06
Protein IGF2BP1 (57): AUROC = 0.7481 | F1: 0.7616 | F1(0.5): 0.6609 | Threshold: 0.0022 | TP:  377 | FP:  213 | FN:   23 | TN:   81 | AvgSum: 19.47

Epoch 26/30
Loss: 0.5950 | AUROC: 0.9112 | F1: 0.8653 | F1(0.5): 0.8082 | mAP: 0.9277 | AvgSum: 17.9901 | Train time: 92.24s | Eval time: 8.34s
Average threshold: 0.10101934
Thresholds (min/max): 0.00057495653 0.6066241
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9977 | F1: 0.9825 | F1(0.5): 0.9785 | Threshold: 0.2266 | TP:  393 | FP:    7 | FN:    7 | TN:  320 | AvgSum: 20.04
Protein GARS (47): AUROC = 0.9942 | F1: 0.9698 | F1(0.5): 0.9294 | Threshold: 0.0192 | TP:  385 | FP:    9 | FN:   15 | TN:  333 | AvgSum: 19.11
Protein ADAT1 (2): AUROC = 0.9937 | F1: 0.9785 | F1(0.5): 0.9560 | Threshold: 0.0938 | TP:  386 | FP:    3 | FN:   14 | TN:  320 | AvgSum: 17.66
Protein AKAP1 (4): AUROC = 0.9916 | F1: 0.9711 | F1(0.5): 0.9659 | Threshold: 0.3166 | TP:  387 | FP:   10 | FN:   13 | TN:  322 | AvgSum: 22.08
Protein FASTKD2 (38): AUROC = 0.9914 | F1: 0.9731 | F1(0.5): 0.9655 | Threshold: 0.0769 | TP:  289 | FP:   13 | FN:    3 | TN:  270 | AvgSum: 25.54
Protein MTPAP (69): AUROC = 0.9902 | F1: 0.9714 | F1(0.5): 0.9697 | Threshold: 0.2403 | TP:  390 | FP:   13 | FN:   10 | TN:  331 | AvgSum: 23.33
Protein PCBP2 (76): AUROC = 0.9879 | F1: 0.9527 | F1(0.5): 0.8571 | Threshold: 0.0240 | TP:  131 | FP:    4 | FN:    9 | TN:  130 | AvgSum: 21.91
Protein DGCR8 (23): AUROC = 0.9844 | F1: 0.9496 | F1(0.5): 0.8889 | Threshold: 0.0618 | TP:  179 | FP:    5 | FN:   14 | TN:  178 | AvgSum: 20.31
Protein CSTF2T (14): AUROC = 0.9842 | F1: 0.9487 | F1(0.5): 0.9433 | Threshold: 0.4241 | TP:  388 | FP:   30 | FN:   12 | TN:  320 | AvgSum: 25.91
Protein TAF15 (106): AUROC = 0.9830 | F1: 0.9430 | F1(0.5): 0.9384 | Threshold: 0.6066 | TP:  372 | FP:   17 | FN:   28 | TN:  322 | AvgSum: 18.49
[INFO] AUROC Bottom 10 proteins:
Protein G3BP1 (46): AUROC = 0.8062 | F1: 0.7478 | F1(0.5): 0.6171 | Threshold: 0.0070 | TP:   86 | FP:   28 | FN:   30 | TN:   79 | AvgSum: 17.98
Protein PPIG (78): AUROC = 0.8000 | F1: 0.7569 | F1(0.5): 0.5389 | Threshold: 0.0006 | TP:  123 | FP:   70 | FN:    9 | TN:   68 | AvgSum: 15.67
Protein FXR2 (45): AUROC = 0.7970 | F1: 0.7823 | F1(0.5): 0.7570 | Threshold: 0.1247 | TP:  345 | FP:  137 | FN:   55 | TN:  182 | AvgSum: 19.16
Protein IGF2BP2 (58): AUROC = 0.7952 | F1: 0.7821 | F1(0.5): 0.6647 | Threshold: 0.0052 | TP:  368 | FP:  173 | FN:   32 | TN:  151 | AvgSum: 19.04
Protein KHSRP (62): AUROC = 0.7885 | F1: 0.7717 | F1(0.5): 0.6958 | Threshold: 0.0416 | TP:  333 | FP:  130 | FN:   67 | TN:  206 | AvgSum: 15.85
Protein CPSF6 (12): AUROC = 0.7813 | F1: 0.7550 | F1(0.5): 0.6953 | Threshold: 0.0182 | TP:  356 | FP:  187 | FN:   44 | TN:  157 | AvgSum: 12.72
Protein RYBP (92): AUROC = 0.7785 | F1: 0.7213 | F1(0.5): 0.5263 | Threshold: 0.0025 | TP:   88 | FP:   51 | FN:   17 | TN:   65 | AvgSum: 13.65
Protein GPKOW (49): AUROC = 0.7695 | F1: 0.7306 | F1(0.5): 0.6432 | Threshold: 0.0247 | TP:  179 | FP:   98 | FN:   34 | TN:  105 | AvgSum: 25.00
Protein ILF3 (60): AUROC = 0.7615 | F1: 0.7398 | F1(0.5): 0.7081 | Threshold: 0.0120 | TP:  236 | FP:  157 | FN:    9 | TN:   91 | AvgSum: 8.53
Protein IGF2BP1 (57): AUROC = 0.7436 | F1: 0.7596 | F1(0.5): 0.6482 | Threshold: 0.0017 | TP:  376 | FP:  214 | FN:   24 | TN:   80 | AvgSum: 19.21

Epoch 27/30
Loss: 0.5783 | AUROC: 0.9110 | F1: 0.8650 | F1(0.5): 0.8049 | mAP: 0.9275 | AvgSum: 17.4814 | Train time: 92.08s | Eval time: 8.39s
Average threshold: 0.09998644
Thresholds (min/max): 1.4781478e-05 0.58693147
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9977 | F1: 0.9825 | F1(0.5): 0.9772 | Threshold: 0.1806 | TP:  393 | FP:    7 | FN:    7 | TN:  320 | AvgSum: 19.29
Protein GARS (47): AUROC = 0.9939 | F1: 0.9652 | F1(0.5): 0.9249 | Threshold: 0.0060 | TP:  388 | FP:   16 | FN:   12 | TN:  326 | AvgSum: 18.92
Protein ADAT1 (2): AUROC = 0.9933 | F1: 0.9786 | F1(0.5): 0.9532 | Threshold: 0.0271 | TP:  388 | FP:    5 | FN:   12 | TN:  318 | AvgSum: 16.96
Protein FASTKD2 (38): AUROC = 0.9923 | F1: 0.9748 | F1(0.5): 0.9635 | Threshold: 0.0421 | TP:  290 | FP:   13 | FN:    2 | TN:  270 | AvgSum: 24.93
Protein AKAP1 (4): AUROC = 0.9910 | F1: 0.9726 | F1(0.5): 0.9657 | Threshold: 0.1368 | TP:  391 | FP:   13 | FN:    9 | TN:  319 | AvgSum: 21.62
Protein MTPAP (69): AUROC = 0.9899 | F1: 0.9727 | F1(0.5): 0.9619 | Threshold: 0.1227 | TP:  392 | FP:   14 | FN:    8 | TN:  330 | AvgSum: 22.85
Protein PCBP2 (76): AUROC = 0.9870 | F1: 0.9524 | F1(0.5): 0.8477 | Threshold: 0.0333 | TP:  130 | FP:    3 | FN:   10 | TN:  131 | AvgSum: 21.34
Protein CSTF2T (14): AUROC = 0.9842 | F1: 0.9480 | F1(0.5): 0.9480 | Threshold: 0.5023 | TP:  383 | FP:   25 | FN:   17 | TN:  325 | AvgSum: 25.19
Protein DGCR8 (23): AUROC = 0.9828 | F1: 0.9471 | F1(0.5): 0.8825 | Threshold: 0.0322 | TP:  179 | FP:    6 | FN:   14 | TN:  177 | AvgSum: 19.79
Protein TAF15 (106): AUROC = 0.9825 | F1: 0.9412 | F1(0.5): 0.9395 | Threshold: 0.4063 | TP:  376 | FP:   23 | FN:   24 | TN:  316 | AvgSum: 17.78
[INFO] AUROC Bottom 10 proteins:
Protein PPIG (78): AUROC = 0.8066 | F1: 0.7586 | F1(0.5): 0.5158 | Threshold: 0.0009 | TP:  121 | FP:   66 | FN:   11 | TN:   72 | AvgSum: 15.26
Protein G3BP1 (46): AUROC = 0.8004 | F1: 0.7400 | F1(0.5): 0.6215 | Threshold: 0.0000 | TP:  111 | FP:   73 | FN:    5 | TN:   34 | AvgSum: 17.50
Protein FXR2 (45): AUROC = 0.7930 | F1: 0.7853 | F1(0.5): 0.7566 | Threshold: 0.0839 | TP:  353 | FP:  146 | FN:   47 | TN:  173 | AvgSum: 18.62
Protein IGF2BP2 (58): AUROC = 0.7924 | F1: 0.7785 | F1(0.5): 0.6686 | Threshold: 0.0022 | TP:  376 | FP:  190 | FN:   24 | TN:  134 | AvgSum: 18.29
Protein KHSRP (62): AUROC = 0.7895 | F1: 0.7729 | F1(0.5): 0.7098 | Threshold: 0.0911 | TP:  325 | FP:  116 | FN:   75 | TN:  220 | AvgSum: 15.44
Protein CPSF6 (12): AUROC = 0.7816 | F1: 0.7566 | F1(0.5): 0.6967 | Threshold: 0.0730 | TP:  331 | FP:  144 | FN:   69 | TN:  200 | AvgSum: 12.36
Protein RYBP (92): AUROC = 0.7789 | F1: 0.7250 | F1(0.5): 0.5369 | Threshold: 0.0023 | TP:   87 | FP:   48 | FN:   18 | TN:   68 | AvgSum: 13.06
Protein GPKOW (49): AUROC = 0.7706 | F1: 0.7354 | F1(0.5): 0.6393 | Threshold: 0.0139 | TP:  182 | FP:  100 | FN:   31 | TN:  103 | AvgSum: 24.31
Protein ILF3 (60): AUROC = 0.7616 | F1: 0.7363 | F1(0.5): 0.7154 | Threshold: 0.0469 | TP:  229 | FP:  148 | FN:   16 | TN:  100 | AvgSum: 8.58
Protein IGF2BP1 (57): AUROC = 0.7461 | F1: 0.7626 | F1(0.5): 0.6501 | Threshold: 0.0106 | TP:  355 | FP:  176 | FN:   45 | TN:  118 | AvgSum: 18.58

Epoch 28/30
Loss: 0.5687 | AUROC: 0.9111 | F1: 0.8648 | F1(0.5): 0.8014 | mAP: 0.9278 | AvgSum: 17.0733 | Train time: 92.20s | Eval time: 8.34s
Average threshold: 0.08398078
Thresholds (min/max): 6.392447e-05 0.66431147
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9975 | F1: 0.9813 | F1(0.5): 0.9759 | Threshold: 0.1755 | TP:  393 | FP:    8 | FN:    7 | TN:  319 | AvgSum: 19.00
Protein GARS (47): AUROC = 0.9937 | F1: 0.9674 | F1(0.5): 0.9162 | Threshold: 0.0079 | TP:  386 | FP:   12 | FN:   14 | TN:  330 | AvgSum: 18.59
Protein ADAT1 (2): AUROC = 0.9931 | F1: 0.9797 | F1(0.5): 0.9546 | Threshold: 0.0814 | TP:  387 | FP:    3 | FN:   13 | TN:  320 | AvgSum: 16.58
Protein FASTKD2 (38): AUROC = 0.9921 | F1: 0.9731 | F1(0.5): 0.9653 | Threshold: 0.0559 | TP:  289 | FP:   13 | FN:    3 | TN:  270 | AvgSum: 24.47
Protein AKAP1 (4): AUROC = 0.9911 | F1: 0.9714 | F1(0.5): 0.9644 | Threshold: 0.1619 | TP:  390 | FP:   13 | FN:   10 | TN:  319 | AvgSum: 21.22
Protein MTPAP (69): AUROC = 0.9899 | F1: 0.9724 | F1(0.5): 0.9684 | Threshold: 0.3284 | TP:  387 | FP:    9 | FN:   13 | TN:  335 | AvgSum: 22.49
Protein PCBP2 (76): AUROC = 0.9874 | F1: 0.9527 | F1(0.5): 0.8525 | Threshold: 0.0134 | TP:  131 | FP:    4 | FN:    9 | TN:  130 | AvgSum: 20.94
Protein CSTF2T (14): AUROC = 0.9849 | F1: 0.9483 | F1(0.5): 0.9457 | Threshold: 0.4754 | TP:  385 | FP:   27 | FN:   15 | TN:  323 | AvgSum: 24.73
Protein DGCR8 (23): AUROC = 0.9836 | F1: 0.9471 | F1(0.5): 0.8793 | Threshold: 0.0341 | TP:  179 | FP:    6 | FN:   14 | TN:  177 | AvgSum: 19.32
Protein GRWD1 (51): AUROC = 0.9828 | F1: 0.9415 | F1(0.5): 0.9362 | Threshold: 0.0584 | TP:  386 | FP:   34 | FN:   14 | TN:  303 | AvgSum: 22.71
[INFO] AUROC Bottom 10 proteins:
Protein PPIG (78): AUROC = 0.8072 | F1: 0.7539 | F1(0.5): 0.5340 | Threshold: 0.0008 | TP:  121 | FP:   68 | FN:   11 | TN:   70 | AvgSum: 14.74
Protein G3BP1 (46): AUROC = 0.8029 | F1: 0.7500 | F1(0.5): 0.5896 | Threshold: 0.0050 | TP:   87 | FP:   29 | FN:   29 | TN:   78 | AvgSum: 17.08
Protein FXR2 (45): AUROC = 0.7943 | F1: 0.7871 | F1(0.5): 0.7519 | Threshold: 0.0551 | TP:  355 | FP:  147 | FN:   45 | TN:  172 | AvgSum: 18.21
Protein IGF2BP2 (58): AUROC = 0.7942 | F1: 0.7795 | F1(0.5): 0.6446 | Threshold: 0.0009 | TP:  380 | FP:  195 | FN:   20 | TN:  129 | AvgSum: 17.81
Protein KHSRP (62): AUROC = 0.7897 | F1: 0.7728 | F1(0.5): 0.7070 | Threshold: 0.0454 | TP:  330 | FP:  124 | FN:   70 | TN:  212 | AvgSum: 14.97
Protein CPSF6 (12): AUROC = 0.7817 | F1: 0.7549 | F1(0.5): 0.6914 | Threshold: 0.0076 | TP:  365 | FP:  202 | FN:   35 | TN:  142 | AvgSum: 11.82
Protein RYBP (92): AUROC = 0.7782 | F1: 0.7222 | F1(0.5): 0.5166 | Threshold: 0.0010 | TP:   91 | FP:   56 | FN:   14 | TN:   60 | AvgSum: 12.66
Protein GPKOW (49): AUROC = 0.7709 | F1: 0.7360 | F1(0.5): 0.6240 | Threshold: 0.0074 | TP:  184 | FP:  103 | FN:   29 | TN:  100 | AvgSum: 23.86
Protein ILF3 (60): AUROC = 0.7633 | F1: 0.7358 | F1(0.5): 0.7057 | Threshold: 0.0226 | TP:  234 | FP:  157 | FN:   11 | TN:   91 | AvgSum: 8.07
Protein IGF2BP1 (57): AUROC = 0.7455 | F1: 0.7574 | F1(0.5): 0.6430 | Threshold: 0.0017 | TP:  370 | FP:  207 | FN:   30 | TN:   87 | AvgSum: 18.10

Epoch 29/30
Loss: 0.5622 | AUROC: 0.9111 | F1: 0.8647 | F1(0.5): 0.8010 | mAP: 0.9276 | AvgSum: 16.9236 | Train time: 92.29s | Eval time: 8.45s
Average threshold: 0.08274664
Thresholds (min/max): 0.00013906245 0.4541718
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9976 | F1: 0.9813 | F1(0.5): 0.9772 | Threshold: 0.1462 | TP:  393 | FP:    8 | FN:    7 | TN:  319 | AvgSum: 18.87
Protein GARS (47): AUROC = 0.9938 | F1: 0.9661 | F1(0.5): 0.9147 | Threshold: 0.0067 | TP:  385 | FP:   12 | FN:   15 | TN:  330 | AvgSum: 18.14
Protein ADAT1 (2): AUROC = 0.9929 | F1: 0.9785 | F1(0.5): 0.9560 | Threshold: 0.0544 | TP:  387 | FP:    4 | FN:   13 | TN:  319 | AvgSum: 16.44
Protein FASTKD2 (38): AUROC = 0.9920 | F1: 0.9748 | F1(0.5): 0.9525 | Threshold: 0.0406 | TP:  290 | FP:   13 | FN:    2 | TN:  270 | AvgSum: 24.19
Protein AKAP1 (4): AUROC = 0.9913 | F1: 0.9714 | F1(0.5): 0.9617 | Threshold: 0.0858 | TP:  390 | FP:   13 | FN:   10 | TN:  319 | AvgSum: 21.07
Protein MTPAP (69): AUROC = 0.9900 | F1: 0.9724 | F1(0.5): 0.9632 | Threshold: 0.1816 | TP:  388 | FP:   10 | FN:   12 | TN:  334 | AvgSum: 22.20
Protein PCBP2 (76): AUROC = 0.9866 | F1: 0.9524 | F1(0.5): 0.8333 | Threshold: 0.0236 | TP:  130 | FP:    3 | FN:   10 | TN:  131 | AvgSum: 20.78
Protein CSTF2T (14): AUROC = 0.9852 | F1: 0.9481 | F1(0.5): 0.9398 | Threshold: 0.3785 | TP:  384 | FP:   26 | FN:   16 | TN:  324 | AvgSum: 24.42
Protein DGCR8 (23): AUROC = 0.9832 | F1: 0.9443 | F1(0.5): 0.8696 | Threshold: 0.0305 | TP:  178 | FP:    6 | FN:   15 | TN:  177 | AvgSum: 19.28
Protein NOLC1 (71): AUROC = 0.9830 | F1: 0.9487 | F1(0.5): 0.9321 | Threshold: 0.1056 | TP:  379 | FP:   22 | FN:   19 | TN:  306 | AvgSum: 21.48
[INFO] AUROC Bottom 10 proteins:
Protein PPIG (78): AUROC = 0.8074 | F1: 0.7586 | F1(0.5): 0.5319 | Threshold: 0.0007 | TP:  121 | FP:   66 | FN:   11 | TN:   72 | AvgSum: 14.79
Protein G3BP1 (46): AUROC = 0.8019 | F1: 0.7456 | F1(0.5): 0.6092 | Threshold: 0.0051 | TP:   85 | FP:   27 | FN:   31 | TN:   80 | AvgSum: 16.93
Protein FXR2 (45): AUROC = 0.7941 | F1: 0.7841 | F1(0.5): 0.7569 | Threshold: 0.0496 | TP:  356 | FP:  152 | FN:   44 | TN:  167 | AvgSum: 18.16
Protein IGF2BP2 (58): AUROC = 0.7924 | F1: 0.7805 | F1(0.5): 0.6495 | Threshold: 0.0157 | TP:  345 | FP:  139 | FN:   55 | TN:  185 | AvgSum: 17.76
Protein KHSRP (62): AUROC = 0.7875 | F1: 0.7738 | F1(0.5): 0.6992 | Threshold: 0.0364 | TP:  337 | FP:  134 | FN:   63 | TN:  202 | AvgSum: 14.90
Protein CPSF6 (12): AUROC = 0.7820 | F1: 0.7537 | F1(0.5): 0.6978 | Threshold: 0.0457 | TP:  335 | FP:  154 | FN:   65 | TN:  190 | AvgSum: 11.81
Protein RYBP (92): AUROC = 0.7804 | F1: 0.7206 | F1(0.5): 0.5306 | Threshold: 0.0015 | TP:   89 | FP:   53 | FN:   16 | TN:   63 | AvgSum: 12.63
Protein GPKOW (49): AUROC = 0.7707 | F1: 0.7392 | F1(0.5): 0.6257 | Threshold: 0.0123 | TP:  180 | FP:   94 | FN:   33 | TN:  109 | AvgSum: 23.55
Protein ILF3 (60): AUROC = 0.7646 | F1: 0.7377 | F1(0.5): 0.7167 | Threshold: 0.0374 | TP:  232 | FP:  152 | FN:   13 | TN:   96 | AvgSum: 8.12
Protein IGF2BP1 (57): AUROC = 0.7470 | F1: 0.7601 | F1(0.5): 0.6440 | Threshold: 0.0056 | TP:  358 | FP:  184 | FN:   42 | TN:  110 | AvgSum: 17.95

Epoch 30/30
Loss: 0.5534 | AUROC: 0.9107 | F1: 0.8643 | F1(0.5): 0.8005 | mAP: 0.9276 | AvgSum: 16.8872 | Train time: 92.16s | Eval time: 8.43s
Average threshold: 0.08090814
Thresholds (min/max): 5.45284e-06 0.45624414
[INFO] AUROC Top 10 proteins:
Protein EIF4G2 (31): AUROC = 0.9975 | F1: 0.9812 | F1(0.5): 0.9772 | Threshold: 0.1720 | TP:  392 | FP:    7 | FN:    8 | TN:  320 | AvgSum: 18.82
Protein GARS (47): AUROC = 0.9937 | F1: 0.9686 | F1(0.5): 0.9177 | Threshold: 0.0084 | TP:  386 | FP:   11 | FN:   14 | TN:  331 | AvgSum: 18.27
Protein ADAT1 (2): AUROC = 0.9928 | F1: 0.9785 | F1(0.5): 0.9546 | Threshold: 0.0514 | TP:  387 | FP:    4 | FN:   13 | TN:  319 | AvgSum: 16.49
Protein FASTKD2 (38): AUROC = 0.9918 | F1: 0.9748 | F1(0.5): 0.9599 | Threshold: 0.0336 | TP:  290 | FP:   13 | FN:    2 | TN:  270 | AvgSum: 24.16
Protein AKAP1 (4): AUROC = 0.9911 | F1: 0.9701 | F1(0.5): 0.9657 | Threshold: 0.1005 | TP:  390 | FP:   14 | FN:   10 | TN:  318 | AvgSum: 20.97
Protein MTPAP (69): AUROC = 0.9897 | F1: 0.9713 | F1(0.5): 0.9671 | Threshold: 0.1693 | TP:  389 | FP:   12 | FN:   11 | TN:  332 | AvgSum: 22.18
Protein PCBP2 (76): AUROC = 0.9869 | F1: 0.9524 | F1(0.5): 0.8285 | Threshold: 0.0287 | TP:  130 | FP:    3 | FN:   10 | TN:  131 | AvgSum: 20.74
Protein CSTF2T (14): AUROC = 0.9846 | F1: 0.9494 | F1(0.5): 0.9467 | Threshold: 0.4562 | TP:  385 | FP:   26 | FN:   15 | TN:  324 | AvgSum: 24.41
Protein NOLC1 (71): AUROC = 0.9826 | F1: 0.9488 | F1(0.5): 0.9377 | Threshold: 0.1108 | TP:  380 | FP:   23 | FN:   18 | TN:  305 | AvgSum: 21.53
Protein TAF15 (106): AUROC = 0.9825 | F1: 0.9410 | F1(0.5): 0.9379 | Threshold: 0.3985 | TP:  375 | FP:   22 | FN:   25 | TN:  317 | AvgSum: 17.23
[INFO] AUROC Bottom 10 proteins:
Protein PPIG (78): AUROC = 0.8059 | F1: 0.7508 | F1(0.5): 0.5312 | Threshold: 0.0013 | TP:  116 | FP:   61 | FN:   16 | TN:   77 | AvgSum: 14.64
Protein G3BP1 (46): AUROC = 0.7993 | F1: 0.7434 | F1(0.5): 0.6057 | Threshold: 0.0000 | TP:  113 | FP:   75 | FN:    3 | TN:   32 | AvgSum: 16.88
Protein IGF2BP2 (58): AUROC = 0.7963 | F1: 0.7795 | F1(0.5): 0.6415 | Threshold: 0.0007 | TP:  380 | FP:  195 | FN:   20 | TN:  129 | AvgSum: 17.67
Protein FXR2 (45): AUROC = 0.7940 | F1: 0.7849 | F1(0.5): 0.7506 | Threshold: 0.0566 | TP:  354 | FP:  148 | FN:   46 | TN:  171 | AvgSum: 18.01
Protein KHSRP (62): AUROC = 0.7900 | F1: 0.7747 | F1(0.5): 0.6847 | Threshold: 0.0279 | TP:  337 | FP:  133 | FN:   63 | TN:  203 | AvgSum: 14.82
Protein CPSF6 (12): AUROC = 0.7786 | F1: 0.7503 | F1(0.5): 0.6951 | Threshold: 0.0046 | TP:  368 | FP:  213 | FN:   32 | TN:  131 | AvgSum: 11.77
Protein RYBP (92): AUROC = 0.7778 | F1: 0.7244 | F1(0.5): 0.5200 | Threshold: 0.0009 | TP:   92 | FP:   57 | FN:   13 | TN:   59 | AvgSum: 12.58
Protein GPKOW (49): AUROC = 0.7691 | F1: 0.7347 | F1(0.5): 0.6354 | Threshold: 0.0115 | TP:  180 | FP:   97 | FN:   33 | TN:  106 | AvgSum: 23.57
Protein ILF3 (60): AUROC = 0.7636 | F1: 0.7367 | F1(0.5): 0.7065 | Threshold: 0.0518 | TP:  228 | FP:  146 | FN:   17 | TN:  102 | AvgSum: 8.16
Protein IGF2BP1 (57): AUROC = 0.7456 | F1: 0.7587 | F1(0.5): 0.6380 | Threshold: 0.0014 | TP:  371 | FP:  207 | FN:   29 | TN:   87 | AvgSum: 17.86

Training complete in 3032.55 seconds.
Model saved to /content/drive/MyDrive/RBPmap_plus/Models/CNNMultiLabelResidual_20250812_123241.pth
Saved train/val split indices to: /content/drive/MyDrive/RBPmap_plus/Models/data_split_20250812_123241.json
"""


### Plotting

In [ ]:
sns.set_context("paper", font_scale=1.4)
sns.set_style("whitegrid")

def parse_training_log(log_data):

    overall_metrics = []
    lines = log_data.split('\n')

    # Regex to capture Epoch number (used on one line)
    epoch_num_pattern = re.compile(r'Epoch (\d+)/\d+')

    # Regex to capture metrics (used on the subsequent line)
    metrics_pattern = re.compile(
        r'Loss: ([\d.]+) \| AUROC: ([\d.]+) \| F1: ([\d.]+) \| F1\(0\.5\): ([\d.]+) \| mAP: ([\d.]+) \| AvgSum: ([\d.]+)'
    )

    for i, line in enumerate(lines):
        epoch_match = epoch_num_pattern.search(line)
        if epoch_match:
            try:
                metrics_line = lines[i + 1]
                metrics_match = metrics_pattern.search(metrics_line)

                if metrics_match:
                    epoch = int(epoch_match.group(1))
                    epoch_data = [epoch] + [float(metrics_match.group(j)) for j in range(1, 7)]
                    overall_metrics.append(epoch_data)
            except IndexError:
                pass
            except Exception as e:
                print(f"Error processing metrics after Epoch {epoch_match.group(1)}: {e}")
                pass

    df_overall = pd.DataFrame(
        overall_metrics,
        columns=['Epoch', 'Loss', 'AUROC', 'F1', 'F1(0.5)', 'mAP', 'AvgSum']
    )
    df_overall = df_overall.set_index('Epoch')

    return df_overall, pd.DataFrame()

def plot_dual_axis(df, col_left_list, col_left_color, col_right, filename, title=None, y_label_left="", y_label_right=""):

    fig, ax1 = plt.subplots(figsize=(12, 8), dpi=300)

    cmap = plt.get_cmap("PuRd_r")

    if len(col_left_list) == len(col_left_color):
        colors = col_left_color
    else:
        color_indices = np.linspace(0.25, 0.55, len(col_left_list))
        colors = [cmap(i) for i in color_indices]


    markers = ['o', 's', '^', 'D']

    lines = []

    for i, col in enumerate(col_left_list):
        line, = ax1.plot(df.index, df[col], label=col,
                         color=colors[i],
                         marker=markers[i % len(markers)],
                         linewidth=2.5, markersize=8)
        lines.append(line)

    ax1.set_xlabel('Epoch', fontweight='bold')
    ax1.set_ylabel(y_label_left, fontweight='bold', color='#444444')
    ax1.tick_params(axis='y', labelcolor='#444444')
    ax1.grid(True, linestyle='--', alpha=0.4)

    ax2 = ax1.twinx()
    color_loss = '#1f78b4'
    line_loss, = ax2.plot(df.index, df[col_right], label=col_right,
                          color=color_loss, linestyle='--',
                          linewidth=2.5, alpha=0.8)
    lines.append(line_loss)

    ax2.set_ylabel(y_label_right, fontweight='bold', color=color_loss, rotation=270, labelpad=20)
    ax2.tick_params(axis='y', labelcolor=color_loss)
    ax2.grid(False)

    labels = [l.get_label() for l in lines]

    ax1.legend(lines, labels,
               loc='lower center',
               ncol=len(lines),
               frameon=True,
               fancybox=True,
               shadow=False,
               framealpha=0.9)

    if title:
        plt.title(title, fontsize=16, fontweight='bold', pad=15)

    plt.tight_layout()

df_overall, df_protein = parse_training_log(log_data)

light_color = plt.get_cmap("PuRd_r")(0.55)
middle_color = plt.get_cmap("PuRd_r")(0.3)
dark_color = plt.get_cmap("PuRd_r")(0.05)

print("Generating Plot 1: Global Validation Performance...")
plot_dual_axis(
    df_overall,
    col_left_list=['mAP', 'AUROC', 'F1'],
    col_left_color = [dark_color, middle_color, light_color],
    col_right='Loss',
    filename="thesis_global_performance",
    y_label_left='Validation Metrics',
    y_label_right='Training Loss'
)

print("Generating Plot 2: Semi-Supervised Convergence (AvgSum)...")
plot_dual_axis(
    df_overall,
    col_left_list=['AvgSum'],
    col_left_color = [middle_color],
    col_right='Loss',
    filename="thesis_avgsum_convergence",
    y_label_left='AvgSum (Mean Predicted Probability)',
    y_label_right='Training Loss'
)

print("Plots Generated Successfully.")

# Evaluation

## Functions and Definitions

In [23]:
def predict_top_k_bindings(model, sequence_vector, k=10, device="cpu"):
    """
    Predicts the top-k most likely binding proteins for a single input sequence.

    Parameters:
        model: Trained PyTorch model
        sequence_vector: np.array shape (500, 4) or (2000,) or (2000, 1)
        k: number of top predictions to return
        device: device to run inference on

    Returns:
        List of tuples: (protein_index, probability), sorted by highest probability
    """

    model.eval()

    # Make sure shape is (1, 500, 4)
    if sequence_vector.ndim == 1:
        sequence_vector = sequence_vector.reshape(500, 4)
    elif sequence_vector.shape == (2000,):
        sequence_vector = sequence_vector.reshape(500, 4)
    elif sequence_vector.shape == (2000, 1):
        sequence_vector = sequence_vector.reshape(500, 4)

    # Convert to tensor
    sequence_tensor = torch.tensor(sequence_vector, dtype=torch.float32).unsqueeze(0).to(device)  # shape: [1, 500, 4]

    # Predict
    with torch.no_grad():
        logits = model(sequence_tensor)
        probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()  # shape: [num_labels]

    # Get top-k
    top_k_indices = np.argsort(probs)[::-1][:k]
    top_k_probs = probs[top_k_indices]

    return list(zip(top_k_indices, top_k_probs))


In [24]:
def plot_binding_predictions_bar(top_bindings, label_tensor, mask_tensor, threshold=0.5, show_line=False, cluster_dict=None):
    label_tensor = np.array(label_tensor)
    mask_tensor = np.array(mask_tensor)

    pred_dict = {int(idx): float(prob) for idx, prob in top_bindings}

    for i in range(len(label_tensor)):
        if mask_tensor[i] == 1 and i not in pred_dict:
            pred_dict[i] = 0.0

    filtered = [(idx, prob) for idx, prob in pred_dict.items()
                if prob >= threshold or mask_tensor[idx] == 1]

    filtered_sorted = sorted(filtered, key=lambda x: x[1], reverse=True)
    indices = [idx for idx, _ in filtered_sorted]
    probs = [prob for _, prob in filtered_sorted]

    colors = []
    for idx in indices:
        if mask_tensor[idx] == 1:
            if label_tensor[idx] == 1:
                colors.append(good_purple)  # known binder
            else:
                colors.append('red')   # known non-binder
        else:
            colors.append('gray')      # unknown

    labels = [Index_mapping.get(idx + 1, str(idx + 1)) for idx in indices]

    label_to_cluster = {}
    if cluster_dict is not None:
        for cluster_name, info in cluster_dict.items():
            cluster_id = cluster_name.replace("cluster", "C")
            for protein in info["proteins"]:
                label_to_cluster[protein] = cluster_id

    plt.figure(figsize=(8, max(4, 0.4 * len(labels))), dpi=300)
    y_pos = np.arange(len(labels))
    plt.barh(y_pos, probs, color=colors)
    plt.yticks(y_pos, labels)
    plt.xlabel("Predicted Probability")
    plt.xlim(0, 1.0)
    if show_line:
        plt.axvline(threshold, color='black', linestyle='--', linewidth=1.5)
    plt.gca().invert_yaxis()
    plt.grid(axis='x', linestyle='--', alpha=0.5)

    if cluster_dict is not None:
        ax = plt.gca()
        for i, label in enumerate(labels):
            cluster_id = label_to_cluster.get(label, None)
            if cluster_id and probs[i] >= 0.2:
                ax.text(0.01, i, cluster_id, va='center', ha='left',
                        fontsize=8, color='black', fontweight='bold')

    plt.tight_layout()
    plt.show()


In [25]:
@torch.no_grad()
def binding_rate_per_label(model, dataloader, device="cuda", threshold=0.5, return_counts=False):
    """
    For each label l:
      - rates[l]         = fraction of sequences predicted positive (>= threshold)  [over ALL sequences in loader]
      - success_ratio[l] = (# predicted positive among KNOWN entries) / (# ground-truth positives among KNOWN entries)

    If the dataloader yields (x, y, mask), we use y & mask to compute success_ratio.
    If it yields only x, success_ratio will be NaN (no labels to compare against).

    Returns:
        (rates, success_ratio)                             if return_counts == False
        (rates, success_ratio, total_num_sequences)        if return_counts == True
    """
    model.eval()
    sum_pred_all = None     # predicted positives over all sequences (for 'rates')
    total = 0

    # For success_ratio:
    sum_pred_known = None   # predicted positives among KNOWN entries (mask==1)
    sum_gt_pos = None       # ground-truth positives among KNOWN entries

    for batch in dataloader:
        if isinstance(batch, (list, tuple)):
            if len(batch) == 3:
                batch_x, batch_y, batch_mask = batch
            elif len(batch) == 2:
                batch_x, batch_y = batch
                batch_mask = None
            else:
                batch_x = batch[0]
                batch_y = None
                batch_mask = None
        else:
            batch_x = batch
            batch_y = None
            batch_mask = None

        batch_x = batch_x.to(device, non_blocking=True)

        logits = model(batch_x)
        probs = torch.sigmoid(logits)
        preds = (probs >= threshold).to(torch.int64)  # [B, L]

        # For 'rates': predicted positives across all sequences
        pred_sum = preds.sum(dim=0).cpu().numpy().astype(np.int64)

        if sum_pred_all is None:
            sum_pred_all = pred_sum
        else:
            sum_pred_all += pred_sum

        total += preds.size(0)

        # For 'success_ratio': needs labels & mask
        if batch_y is not None:
            batch_y = batch_y.to(device, non_blocking=True)
            if batch_mask is not None:
                batch_mask = batch_mask.to(device, non_blocking=True).to(torch.int64)
                # count only KNOWN entries
                pred_known = ((preds == 1) & (batch_y == 1) & (batch_mask == 1)).sum(dim=0).cpu().numpy().astype(np.int64) #(TP)/(GT)
                gt_pos = (batch_y.to(torch.int64) * batch_mask).sum(dim=0).cpu().numpy().astype(np.int64)
            else:
                # if no mask, treat all as known
                pred_known = ((preds == 1) & (batch_y == 1)).sum(dim=0).cpu().numpy().astype(np.int64) #(TP)/(GT)
                gt_pos = (batch_y.to(torch.int64)).sum(dim=0).cpu().numpy().astype(np.int64)

            if sum_pred_known is None:
                sum_pred_known = pred_known
                sum_gt_pos = gt_pos
            else:
                sum_pred_known += pred_known
                sum_gt_pos += gt_pos

    # rates over all sequences
    if total > 0:
        rates = sum_pred_all / float(total)
    else:
        rates = np.zeros_like(sum_pred_all, dtype=np.float32)

    # success_ratio over known entries
    if sum_pred_known is not None and sum_gt_pos is not None:
        denom = np.where(sum_gt_pos > 0, sum_gt_pos, np.nan)  # avoid /0 → NaN
        success_ratio = sum_pred_known / denom
    else:
        # no labels available from loader
        success_ratio = np.full_like(rates, np.nan, dtype=np.float32)

    if return_counts:
        return rates, success_ratio, total
    return rates, success_ratio


In [26]:
def find_indices_with_label(dataset, label_index, max_results=20):
    results = []
    for i in range(len(dataset)):
        seq, label, mask = dataset[i]
        if mask[label_index] == 1 and label[label_index] == 1:
            results.append(i)
            if len(results) >= max_results:
                break
    return results

def find_indices_with_multilabel(dataset, max_results=20):
    results = []
    for i in range(len(dataset)):
        seq, label, mask = dataset[i]
        if sum(label) > 1:
            results.append(i)
            if len(results) >= max_results:
                break
    return results


In [27]:
def plot_roc_curves(y_true, y_pred, label_names=None, mask=None, num_prots=10):

    import numpy as np
    import matplotlib.pyplot as plt
    from sklearn.metrics import roc_curve, auc

    num_labels = y_true.shape[1]
    auc_dict = {}
    roc_data = {}

    for i in range(num_labels):
        if mask is not None:
            valid = mask[:, i].astype(bool)
            if not np.any(valid):
                auc_dict[label_names[i] if label_names else f"Label {i}"] = None
                continue
            y_t = y_true[valid, i]
            y_p = y_pred[valid, i]
        else:
            y_t = y_true[:, i]
            y_p = y_pred[:, i]

        label = label_names[i] if label_names else f"Label {i}"

        if np.unique(y_t).size < 2:
            auc_dict[label] = None
            continue

        fpr, tpr, _ = roc_curve(y_t, y_p)
        roc_auc = auc(fpr, tpr)
        auc_dict[label] = roc_auc
        roc_data[label] = (fpr, tpr)

    valid_roc = [(label, auc) for label, auc in auc_dict.items() if auc is not None]
    sorted_roc = sorted(valid_roc, key=lambda x: x[1], reverse=True)

    if num_prots >= len(sorted_roc):
        selected = [label for label, _ in sorted_roc]
    else:
        step = max(1, len(sorted_roc) // (num_prots - 1))
        selected = [sorted_roc[0][0]]  # top AUC
        selected += [sorted_roc[i][0] for i in range(step, len(sorted_roc), step)][:num_prots - 1]

    plt.figure(figsize=(12, 10))
    for label in selected:
        fpr, tpr = roc_data[label]
        plt.plot(fpr, tpr, lw=1.5, label=f"{label} (AUC = {auc_dict[label]:.2f})")

    valid_aucs = [v for v in auc_dict.values() if v is not None]
    avg_auc = np.mean(valid_aucs) if valid_aucs else 0.0
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label="Random chance")
    plt.legend(title=f"Avg AUC = {avg_auc:.3f}", loc="lower right", fontsize='medium')

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curves for Selected Labels")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return auc_dict


def plot_sorted_auc_histogram(auc_dict, lines=None, title="Sorted AUC Histogram",
                              xlabels=False, left_color="powderblue", right_color="red", threshold_color=0.8):

    filtered_items = [(k, v) for k, v in auc_dict.items() if v is not None and not np.isnan(v)]
    sorted_items = sorted(filtered_items, key=lambda x: x[1])
    labels, auc_values = zip(*sorted_items)
    auc_values = np.array(auc_values)

    bar_colors = [right_color if val >= threshold_color else left_color for val in auc_values]

    plt.figure(figsize=(max(12, len(labels)*0.15), max(4, len(labels)*0.05)))
    plt.bar(range(len(auc_values)), auc_values, color=bar_colors)

    if lines:
        for line_val, line_col in lines:
            count = np.sum(auc_values >= line_val)
            total = len(auc_values)
            plt.axhline(line_val, linestyle='--', linewidth=2, color=line_col,
                        label=f"AUC ≥ {line_val:.2f}: {count} / {total}")

    if xlabels:
        plt.xticks(ticks=range(len(labels)), labels=labels, rotation=90)
    else:
        plt.xticks([])

    plt.ylim(0.0, 1.0)
    plt.ylabel("AUC Score")

    plt.tight_layout()
    plt.show()

def plot_auc_three_columns(auc_dict, lines=None, title="Sorted AUC Histogram (3 Columns)",
                           ylabels=True, left_color="powderblue", right_color="red", threshold_color=0.8):

    filtered_items = [(k, v) for k, v in auc_dict.items() if v is not None and not np.isnan(v)]
    sorted_items = sorted(filtered_items, key=lambda x: x[1])

    total = len(sorted_items)
    if total == 0:
        print("No valid data to plot.")
        return

    chunk_size = total // 3
    remainder = total % 3

    end_1 = chunk_size + (1 if remainder >= 1 else 0)
    end_2 = end_1 + chunk_size + (1 if remainder >= 2 else 0)

    data_3 = sorted_items[:end_1]        # Lowest scores
    data_2 = sorted_items[end_1:end_2]   # Middle scores
    data_1 = sorted_items[end_2:]        # Highest scores

    fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(8.27, 10.5), dpi=300)

    def plot_on_axis(ax, data, subtitle):
        if not data:
            ax.axis('off')
            return

        labels, values = zip(*data)
        values = np.array(values)
        y_pos = np.arange(len(labels))

        colors = [right_color if v >= threshold_color else left_color for v in values]

        ax.barh(y_pos, values, color=colors, height=0.8, alpha=0.6)

        for i, v in enumerate(values):
            ax.text(0.52, i, f"{v:.3f}", va='center', fontsize=7, color='black')

        if lines:
            for line_val, line_col in lines:
                if 0.5 <= line_val <= 1.0:
                    count = np.sum(values >= line_val)
                    ax.axvline(line_val, linestyle='--', linewidth=1.5, color=line_col,
                               label=f"≥{line_val:.2f} ({count})")

        if ylabels:
            ax.set_yticks(y_pos)
            ax.set_yticklabels(labels, fontsize=10)
        else:
            ax.set_yticks([])

        ax.set_ylim(-1, len(labels))

        ax.set_xlim(0.5, 1.0)
        ax.set_xlabel("AUC Score", fontsize=10)
        ax.set_xticks([0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
        ax.tick_params(axis='x', labelsize=10)

        ax.grid(axis='x', linestyle=':', alpha=0.5)

    plot_on_axis(axes[0], data_1, "High")
    plot_on_axis(axes[1], data_2, "Mid")
    plot_on_axis(axes[2], data_3, "Low")

    plt.tight_layout(rect=[0, 0, 1, 0.97])

    plt.savefig("auc_columns_labeled.png", dpi=300, bbox_inches='tight')
    plt.show()

In [28]:
@torch.no_grad()
def compute_codependence_matrix(model, dataloader, device="gpu", threshold=0.5, return_counts=False):
    """
    Computes a num_labels x num_labels matrix M where:
      - First we accumulate co-occurrence counts: C = sum_b (b^T @ b) over batches,
        where b is the binarized prediction matrix for a batch (thresholded).
      - Then we column-normalize by the diagonal: M[:, j] = C[:, j] / C[j, j].
        This yields M[j, j] = 1 (when C[j, j] > 0), and for i != j, M[i, j]
        is the fraction of times label i co-appears when label j appears.

    Returns:
      M: np.ndarray [L, L]  (float32), column-normalized co-dependence
      diag_counts: np.ndarray [L]      (int64), the diagonal counts (if return_counts=True)
    """
    model.eval()
    C = None
    L = None

    for batch in dataloader:
        # Supports datasets that yield (x, y, mask) or (x, y)
        x = batch[0] if isinstance(batch, (list, tuple)) else batch
        x = x.to(device, non_blocking=True)

        logits = model(x)
        probs = torch.sigmoid(logits)            # [B, L]
        preds = (probs >= threshold).to(torch.int64)  # [B, L]

        if L is None:
            L = preds.size(1)
        if C is None:
            C = np.zeros((L, L), dtype=np.int64)

        # Co-occurrence for the batch: preds^T @ preds
        # Move to CPU once per batch for efficiency
        P = preds.cpu().numpy().astype(np.int64)          # [B, L]
        C += P.T @ P                                      # [L, L]

    if C is None:
        raise ValueError("Dataloader produced no batches.")

    diag = C.diagonal().astype(np.float32)               # [L]
    M = C.astype(np.float32)

    # Column-normalize by the diagonal (avoid division by zero)
    for j in range(L):
        if diag[j] > 0:
            M[:, j] /= diag[j]
        else:
            M[:, j] = 0.0  # no positives for column j → set column to zeros

    if return_counts:
        return M, C, diag
    return M


In [29]:
def plot_codependence_heatmap(M, Index_mapping, diag_counts=None, top_k=None, title="Co-dependence (squared scale)", power=None):
    """
    Plot a heatmap of the column-normalized co-dependence matrix M (shape [L, L]).

    Args:
        M: np.ndarray [L, L], M[:, j] = P(i | j)
        Index_mapping: dict {label_id (1..L) or negative ids: name}; we use positive ids 1..L
        diag_counts: np.ndarray [L] of raw diagonal counts (optional; for top_k selection)
        top_k: int or None. If provided, selects the top_k labels by diag_counts (or by column sums if diag_counts None).
        title: plot title
    """
    L = M.shape[0]
    assert M.shape[1] == L, "M must be square"

    # Choose which labels to show
    if top_k is not None and top_k < L:
        if diag_counts is not None:
            order = np.argsort(diag_counts)[::-1]
        else:
            order = np.argsort(M.sum(axis=0))[::-1]
        keep = order[:top_k]
        keep = np.sort(keep)
        M_show = M[np.ix_(keep, keep)]
        labels = [Index_mapping.get(i+1, str(i+1)) for i in keep]
    else:
        M_show = M
        labels = [Index_mapping.get(i+1, str(i+1)) for i in range(L)]

    # Apply square transform to emphasize values near 1
    if power is not None:
        M_sq = np.power(M_show, power)
    else:
        M_sq = M_show


    # Plot heatmap
    plt.figure(figsize=(max(6, 0.35*len(labels)), max(5, 0.35*len(labels))))
    # im = plt.imshow(M_sq, aspect='auto', interpolation='nearest', cmap='viridis')
    im = plt.imshow(M_sq, aspect='auto', interpolation='nearest', cmap='PuRd', vmin=0.0, vmax=1.0)
    cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
    cbar.set_label("Squared P(i | j)")

    plt.xticks(ticks=np.arange(len(labels)), labels=labels, rotation=90)
    plt.yticks(ticks=np.arange(len(labels)), labels=labels)
    plt.title(title)
    plt.xlabel("Conditioned on protein j")
    plt.ylabel("Co-appearing protein i")
    plt.tight_layout()
    plt.show()

In [30]:
def split_cluster_by_brightness(M_ord, initial_thresh=0.6, brightness_thresh=0.3, min_block_size=2):
    """
    Recursively splits clusters based on brightness inside heatmap blocks.
    Ensures only meaningful, non-overlapping clusters are returned.
    """
    def recursive_split(indices, thresh):
        if len(indices) <= min_block_size:
            return [indices]

        submatrix = M_ord[np.ix_(indices, indices)]
        bright_mask = submatrix > brightness_thresh
        structure = np.ones((3, 3))
        labeled, num_features = label(bright_mask, structure=structure)

        if num_features <= 1:
            return [indices]

        # Reclustering
        try:
            D = pdist(submatrix + 1e-9, metric="cosine")
            Z_sub = linkage(D, method="average")
            clusters = fcluster(Z_sub, t=thresh * 0.8, criterion='distance')
        except:
            return [indices]  # fallback if linkage fails

        unique_clusters = np.unique(clusters)
        if len(unique_clusters) == 1:
            return [indices]

        results = []
        for uc in unique_clusters:
            group = [indices[i] for i in range(len(indices)) if clusters[i] == uc]
            if len(group) < min_block_size:
                continue
            results.extend(recursive_split(group, thresh * 0.8))
        return results

    # Initial global clustering
    D = pdist(M_ord + 1e-9, metric="cosine")
    Z = linkage(D, method="average")
    cluster_ids = fcluster(Z, t=initial_thresh, criterion='distance')

    all_clusters = []
    for cluster_id in np.unique(cluster_ids):
        members = np.where(cluster_ids == cluster_id)[0]
        if len(members) < min_block_size:
            continue
        all_clusters.extend(recursive_split(members.tolist(), initial_thresh))

    # Overlap filtering: keep only the smallest non-contained clusters
    final_clusters = []
    used = np.zeros(len(M_ord), dtype=bool)
    for cluster in sorted(all_clusters, key=lambda x: len(x)):
        if any(used[i] for i in cluster):
            continue
        final_clusters.append(cluster)
        for i in cluster:
            used[i] = True

    return final_clusters


In [31]:
def cluster_codependence_matrix(M, Index_mapping, top_k=None):
    """
    Performs clustering on the co-dependence matrix M.
    Returns ordered matrix, labels, and all data needed for plotting.
    """
    L = M.shape[0]
    assert M.shape[1] == L, "M must be square"

    # Subset Data (if top_k provided)
    if top_k is not None and top_k < L:
        order_by_mass = np.argsort(M.sum(axis=0))[::-1][:top_k]
        order_by_mass = np.sort(order_by_mass)
        M_use = M[np.ix_(order_by_mass, order_by_mass)]
        labels = [Index_mapping.get(i+1, str(i+1)) for i in order_by_mass]
    else:
        M_use = M
        labels = [Index_mapping.get(i+1, str(i+1)) for i in range(L)]

    # Hierarchical Clustering
    S = 0.5 * (M_use + M_use.T) + 1e-9
    D = pdist(S, metric="cosine")
    Z = linkage(D, method="average")
    leaves = leaves_list(Z)

    M_ord = M_use[leaves][:, leaves]
    labels_ord_temp = [labels[i] for i in leaves]

    #  Visual Block Detection (Recursive)
    clusters = split_cluster_by_brightness(M_ord, initial_thresh=0.6)

    # Sort clusters by first index
    clusters_sorted = sorted(clusters, key=lambda c: min(c))
    flat_indices = [i for cluster in clusters_sorted for i in cluster]

    #  Final Reordering
    M_ord = M_ord[np.ix_(flat_indices, flat_indices)]
    labels_ord = [labels_ord_temp[i] for i in flat_indices]

    # Build Cluster Info & Plot Data
    cluster_dict = {}
    clusters_data_list = [] # Stores sub-matrices for potential plotting

    for i, cluster in enumerate(clusters_sorted):
        cluster_name = f"cluster{i+1}"
        # Map back to indices in the final reordered list
        cluster_indices = [j for j in range(len(labels_ord)) if flat_indices[j] in cluster]
        proteins = [labels_ord[j] for j in cluster_indices]

        # Compute cohesion and brightness
        if len(cluster_indices) > 1:
            subM = M_ord[np.ix_(cluster_indices, cluster_indices)]
            subS = 0.5 * (subM + subM.T)
            cohesion = 1 - np.mean(pdist(subS + 1e-9, metric='cosine'))
            brightness = np.mean(subM)

            # Save data needed for individual cluster plots
            clusters_data_list.append({
                'name': cluster_name,
                'matrix': subM,
                'labels': proteins,
                'brightness': brightness,
                'cohesion': cohesion
            })
        else:
            cohesion = 1.0
            brightness = 0.0

        strength = 0.5 * cohesion + 0.5 * brightness

        cluster_dict[cluster_name] = {
            "proteins": proteins,
            "cohesion": cohesion,
            "brightness": brightness,
            "strength": strength
        }

    return {
        "M_ord": M_ord,
        "labels_ord": labels_ord,
        "cluster_dict": cluster_dict,
        "clusters_sorted": clusters_sorted,
        "flat_indices": flat_indices,
        "Z": Z,
        "clusters_data_list": clusters_data_list,
        "original_labels": labels
    }

In [32]:
def get_dynamic_fontsize(n_items, length_inches):
    if n_items == 0: return 26

    axis_length_points = length_inches * 0.90 * 72
    space_per_item = axis_length_points / n_items

    if n_items > 15:
        return max(6, min(space_per_item * 0.98, 40))
    else:
        return max(10, min(space_per_item * 0.85, 26))

def get_main_plot_fontsize(n_items, length_inches):
    if n_items == 0: return 20

    axis_length_points = length_inches * 0.90 * 72
    space_per_item = axis_length_points / n_items

    return max(8, min(space_per_item * 0.80, 22))

def plot_clustered_heatmap(data_pack, title="Co-dependence (clustered)", power=None,
                           plot_squares=True, plot_dendrogram=False, plot_clusters=False,
                           download=False):

    generated_files = []

    # Unpack data
    M_ord = data_pack["M_ord"]
    labels_ord = data_pack["labels_ord"]
    clusters_sorted = data_pack["clusters_sorted"]
    Z = data_pack["Z"]
    original_labels = data_pack["original_labels"]
    clusters_data_list = data_pack["clusters_data_list"]

    # Custom Colormap
    cividis_bg = plt.get_cmap('cividis')(0.0)
    mid_purple = '#800080'
    bright_pink = '#FF1493'

    thesis_cmap = mcolors.LinearSegmentedColormap.from_list(
        'CividisPurplePink',
        [(0.0, cividis_bg), (0.5, mid_purple), (1.0, bright_pink)],
        N=256
    )

    if power is not None:
        M_plot = np.power(M_ord, power)
        cbar_label = f"P(i|j)^{power}"
    else:
        M_plot = M_ord
        cbar_label = "P(i|j)"

    # Main Heatmap
    FIG_W, FIG_H = 14, 12
    plt.figure(figsize=(FIG_W, FIG_H), dpi=300)

    im = plt.imshow(M_plot, aspect='auto', interpolation='nearest',
                    cmap=thesis_cmap, vmin=0.0, vmax=1.0)

    # Calc Font Size (Main Plot - Conservative)
    n_clusters = len(clusters_sorted)
    font_size_main = get_main_plot_fontsize(n_clusters, min(FIG_W, FIG_H))

    # Colorbar
    cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
    cbar.set_label(cbar_label, fontsize=font_size_main)
    cbar.ax.tick_params(labelsize=font_size_main)

    # Ticks
    cluster_ticks = []
    cluster_tick_labels = []
    current_idx = 0
    for i, cluster in enumerate(clusters_sorted):
        cluster_size = len(cluster)
        center_pos = current_idx + (cluster_size / 2) - 0.5
        if cluster_size > 1:
            cluster_ticks.append(center_pos)
            cluster_tick_labels.append(f"C{i+1}")
        current_idx += cluster_size

    # Apply Font Size
    plt.xticks(cluster_ticks, cluster_tick_labels, fontsize=font_size_main, fontweight='bold', rotation=90)
    plt.yticks(cluster_ticks, cluster_tick_labels, fontsize=font_size_main, fontweight='bold')

    # Squares
    current_pos = 0
    for cluster in clusters_sorted:
        size = len(cluster)
        if size < 2:
            current_pos += size
            continue
        if plot_squares:
            rect = plt.Rectangle((current_pos - 0.5, current_pos - 0.5), size, size,
                                 edgecolor='cyan', facecolor='none', linewidth=2)
            plt.gca().add_patch(rect)
        current_pos += size

    plt.tight_layout()
    main_filename = "thesis_clustered_heatmap.png"
    plt.savefig(main_filename, dpi=300, bbox_inches='tight')
    plt.close()
    display(Image(main_filename))
    generated_files.append(main_filename)

    # Dendrogram
    if plot_dendrogram:
        plt.figure(figsize=(12, 8), dpi=300)
        dendrogram(Z, labels=original_labels, leaf_rotation=90)
        plt.title("Protein Clustering Dendrogram", fontsize=22)
        plt.xlabel("Proteins", fontsize=22)
        plt.ylabel("Distance", fontsize=22)
        plt.tick_params(axis='both', which='major', labelsize=14)
        plt.tight_layout()
        dendro_filename = "thesis_dendrogram.png"
        plt.savefig(dendro_filename, dpi=300)
        plt.close()
        display(Image(dendro_filename))
        generated_files.append(dendro_filename)

    # Individual Clusters
    if plot_clusters and clusters_data_list:
        print("\n--- Individual Cluster Views ---")
        for c_data in clusters_data_list:
            n_items = len(c_data['labels'])

            # Figure size
            fig_size = max(6, 0.8 * n_items)

            plt.figure(figsize=(fig_size, fig_size), dpi=300)

            if power is not None:
                sub_matrix = np.power(c_data['matrix'], power)
            else:
                sub_matrix = c_data['matrix']

            plt.imshow(sub_matrix, aspect='equal', interpolation='nearest',
                       cmap=thesis_cmap, vmin=0.0, vmax=1.0)
            plt.grid(False)

            labels = c_data['labels']

            # Grid
            for i in range(n_items - 1):
                if labels[i] != labels[i+1]:
                    plt.axvline(x=i + 0.5, color='white', linewidth=2)
                    plt.axhline(y=i + 0.5, color='white', linewidth=2)

            cluster_fs = get_dynamic_fontsize(n_items, fig_size)

            cb = plt.colorbar(fraction=0.046, pad=0.04)
            cb.set_label(cbar_label, fontsize=cluster_fs)
            cb.ax.tick_params(labelsize=cluster_fs)

            plt.xticks(range(n_items), labels, rotation=90, fontsize=cluster_fs)
            plt.yticks(range(n_items), labels, fontsize=cluster_fs)
            plt.tight_layout()

            cluster_filename = f"cluster_{c_data['name']}.png"
            plt.savefig(cluster_filename, dpi=300, bbox_inches='tight')
            plt.close()
            display(Image(cluster_filename))
            generated_files.append(cluster_filename)

    # Download
    if download:
        zip_filename = "thesis_plots.zip"
        with zipfile.ZipFile(zip_filename, 'w') as zipf:
            for file in generated_files:
                if os.path.exists(file):
                    zipf.write(file)
        files.download(zip_filename)

In [34]:
def print_cluster_dict(cluster_dict):
    for cname, info in cluster_dict.items():
        proteins_str = " ".join(info["proteins"])
        cohesion = round(float(info["cohesion"]), 3)
        brightness = round(float(info["brightness"]), 3)
        strength = round(float(info["strength"]), 3)
        print(f'{cname}: {{')
        print(f'  "proteins": [{proteins_str}],')
        print(f'  "cohesion": {cohesion},')
        print(f'  "brightness": {brightness},')
        print(f'  "strength": {strength}')
        print("}")


def save_and_download_clusters(cluster_dict, output_dir="cluster_protein_files"):

    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir)

    for cname, info in cluster_dict.items():
        proteins = info["proteins"]
        filename = f"{cname}.txt"
        file_path = os.path.join(output_dir, filename)

        with open(file_path, "w") as f:
            f.write("\n".join(proteins))

    print(f"Saved {len(cluster_dict)} files to '{output_dir}/'")

    zip_filename = f"{output_dir}"
    shutil.make_archive(zip_filename, 'zip', output_dir)
    print(f"Created zip archive: {zip_filename}.zip")

    files.download(f"{zip_filename}.zip")


## Loading model and dataset

In [ ]:
load_option = "from_path" # "from_path" / "from_train"
loaded_batch_size = 256
time_stamp = "20250812_123241"

if load_option == "from_path":
    #load dataset
    csv_path = "/content/drive/MyDrive/intRBP/dataset_K562_multilabel_with_NEGs.csv"
    print("Loading CSV and preprocessing...")
    sequences, targets, masks, label_counts = load_masked_multilabel_dataset(csv_path)
    print(f"Loaded {len(sequences)} sequences.")

    dataset = MaskedMultilabelDataset(sequences, targets, masks)

    # Load indices
    split_load_path = f"/content/drive/MyDrive/intRBP/Models/data_split_{time_stamp}.json"
    with open(split_load_path, "r") as f:
        split_dict = json.load(f)

    loaded_train_idx = split_dict["train_idx"]
    loaded_val_idx = split_dict["val_idx"]

    print(f"Loaded {len(loaded_train_idx)} train sequences.")
    print(f"Loaded {len(loaded_val_idx)} validation sequences.")

    loaded_train_loader = DataLoader(Subset(dataset, loaded_train_idx), batch_size=loaded_batch_size, num_workers=2, shuffle=True)
    loaded_val_loader = DataLoader(Subset(dataset, loaded_val_idx), batch_size=loaded_batch_size, num_workers=2)

    loaded_model = CNNMultiLabelResidual(num_labels=targets.shape[1])
    model_load_path = f"/content/drive/MyDrive/intRBP/Models/CNNMultiLabelResidual_{time_stamp}.pth"
    loaded_model.load_state_dict(torch.load(model_load_path, map_location=device))
    loaded_model.to(device)

elif load_option == "from_train":
    loaded_train_loader = train_loader
    loaded_val_loader = val_loader
    loaded_model = model
    loaded_model.to(device)


In [ ]:
loaded_aurocs, loaded_f1s, loaded_f1s_05, loaded_mAP, loaded_thresholds, loaded_confusions, loaded_y_true, loaded_y_pred, loaded_y_mask, loaded_avg_output_sums = evaluate_masked(loaded_model, loaded_val_loader, device)
loaded_mask = np.vstack([m.cpu().numpy() for _, _, m in loaded_val_loader])

## Main evaluate

In [ ]:
prot = "IGF2BP2"
label_idx = Class_mapping[prot] - 1  # your label index
positive_indices = find_indices_with_label(loaded_val_loader.dataset, label_idx)
print(positive_indices)

multilabel_indices = find_indices_with_multilabel(loaded_val_loader.dataset)
print(multilabel_indices)

In [ ]:
random_idx = random.randint(0, len(loaded_val_loader.dataset) - 1)
sequence_tensor, label_tensor, mask_tensor = loaded_val_loader.dataset[random_idx]
sequence_vector = sequence_tensor.numpy()  # shape: (500, 4)

top_bindings = predict_top_k_bindings(loaded_model, sequence_vector, k=122, device=device)

decoded_sequence = ""
for base in sequence_tensor:  # shape: (500, 4)
    idx = base.argmax().item()  # Find which nucleotide is '1'
    decoded_sequence += Index_Character_mapping.get(idx, 'N')  # Use 'N' if invalid

print("Decoded sequence (50 bases per line):")
for i in range(0, len(decoded_sequence), 50):
    print(decoded_sequence[i:i+50])

# Extract labels
positive_labels = [i+1 for i in range(len(label_tensor)) if mask_tensor[i] == 1 and label_tensor[i] == 1]
negative_labels = [i+1 for i in range(len(label_tensor)) if mask_tensor[i] == 1 and label_tensor[i] == 0]

# Print binding/non-binding labels
print("\nKnown binding proteins (+):", [f"{Index_mapping[j]} ({j})" for j in positive_labels] if positive_labels else "None")
print("Known non-binding proteins (−):", [f"{Index_mapping[j]} ({j})" for j in negative_labels] if negative_labels else "None")

print("\nModel predictions:")
plot_binding_predictions_bar(top_bindings, label_tensor.numpy(), mask_tensor.numpy(), threshold=0.8, show_line=False, cluster_dict=None)


In [ ]:
plot_all_label_histogram_masked(
    loaded_y_true, loaded_y_pred, loaded_mask,
    thresholds=loaded_thresholds,
    bins=50, scale=None
)

In [ ]:
protein_names = [Index_mapping.get(i+1, str(i+1)) for i in range(loaded_y_true.shape[1])]
auc_dict = plot_roc_curves(loaded_y_true, loaded_y_pred, label_names=protein_names, mask=loaded_y_mask, num_prots=10)

In [ ]:
lines=[(0.8, "#C20072"), (0.95, "#6B0037")]

plot_sorted_auc_histogram(auc_dict, lines=lines, xlabels=True, left_color="#B2CADD", right_color="#2e76b0", threshold_color=1)
plot_auc_three_columns(auc_dict, lines=lines, ylabels=True, left_color="#1f78b4", right_color="#2e76b0", threshold_color=1)

In [ ]:
# Load Data
df = pd.read_excel("intRBP_DeepRiPe_comparison.xlsx")

# Process Data
df_my = df.iloc[:, [0, 1]].dropna()
df_my.columns = ['Protein', 'My_AUC']
df_my['Protein'] = df_my['Protein'].astype(str).str.strip().str.upper()

df_other = df.iloc[:, [3, 4]].dropna()
df_other.columns = ['Protein', 'Other_AUC']
df_other['Protein'] = df_other['Protein'].astype(str).str.strip().str.upper()

merged = pd.merge(df_my, df_other, on='Protein')
merged['Diff'] = merged['My_AUC'] - merged['Other_AUC']
merged = merged.sort_values('Diff', ascending=False)

# Plotting Setup
n_cols = 2
n_items = len(merged)
chunk_size = math.ceil(n_items / n_cols)

fig, axes = plt.subplots(1, n_cols, figsize=(20, 20), dpi=300)
sns.set_context("poster", font_scale=1.6)
plt.rcParams['mathtext.default'] = 'regular'

for i in range(n_cols):
    ax = axes[i]
    start = i * chunk_size
    end = start + chunk_size
    chunk = merged.iloc[start:end]

    if chunk.empty:
        ax.axis('off')
        continue

    # Colors
    colors_list = ['#6B0037' if x > 0 else '#C20072' for x in chunk['Diff']]
    palette_dict = dict(zip(chunk['Protein'], colors_list))

    # Plot
    sns.barplot(
        x='Diff', y='Protein', hue='Protein',
        data=chunk, palette=palette_dict, ax=ax, dodge=False
    )

    if ax.legend_: ax.legend_.remove()

    # Limits & Formatting
    ax.axvline(0, color='black', linewidth=1.5)
    ax.set_xlim(-0.43, 0.43)
    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.tick_params(axis='y', labelsize=20)
    ax.tick_params(axis='x', labelsize=20)

    ax.text(-0.25, -0.8, "DeepRiPe", ha='center', va='bottom',
            fontsize=24, fontweight='bold', color='#C20072')

    ax.text(0.25, -0.8, "intRBP", ha='center', va='bottom',
            fontsize=24, fontweight='bold', color='#6B0037')

    for idx, (row_idx, row) in enumerate(chunk.iterrows()):
        diff = row['Diff']
        my_auc = row['My_AUC']
        other_auc = row['Other_AUC']

        if my_auc >= other_auc:
            label_text = r"{:.2f} | $\bf{{ {:.2f} }}$".format(other_auc, my_auc)
        else:
            label_text = r"$\bf{{ {:.2f} }}$ | {:.2f}".format(other_auc, my_auc)

        if diff >= 0:
            x_pos = diff + 0.01
            ha = 'left'
        else:
            x_pos = diff - 0.01
            ha = 'right'

        ax.text(x_pos, idx, label_text, va='center', ha=ha, fontsize=20, color='black')

fig.text(0.5, 0.02, "AUC Difference", ha='center', va='top', fontsize=30)

plt.tight_layout()
plt.subplots_adjust(bottom=0.06, top=0.95)
plt.show()

In [ ]:
M, C_raw, diag_counts = compute_codependence_matrix(
    loaded_model, loaded_val_loader, device=device, threshold=0.8, return_counts=True
)

plot_codependence_heatmap(M, Index_mapping, diag_counts=diag_counts, top_k=None,
                          title="Co-dependence Heatmap (All Labels)", power=None)


In [ ]:
data_pack = cluster_codependence_matrix(M, Index_mapping, top_k=None)

M_ord = data_pack["M_ord"]
labels_ord = data_pack["labels_ord"]
cluster_dict = data_pack["cluster_dict"]

plot_clustered_heatmap(
    data_pack,
    title="Clustered co-dependence",
    power=None,
    plot_squares=True,
    plot_clusters=True,
    plot_dendrogram=False,
    download=True
)

In [ ]:
save_and_download_clusters(cluster_dict)
print_cluster_dict(cluster_dict)

# GO Analysis

In [ ]:
# --- CONFIGURATION ---
INPUT_EXCEL = "GO-Analysis-Fisher-Calculation.xlsx"
OUTPUT_EXCEL = "GO-Analysis-Fisher-Final.xlsx"
ZIP_NAME = "GO enrichment results.zip"
BASE_DIR = "results"

# UNZIP
if not os.path.exists(BASE_DIR):
    if os.path.exists(ZIP_NAME):
        print(f"Unzipping {ZIP_NAME}...")
        with zipfile.ZipFile(ZIP_NAME, 'r') as zip_ref:
            zip_ref.extractall(".")
    else:
        print(f"Error: {ZIP_NAME} not found. Please upload it.")

def get_path(name):
    for root, dirs, files in os.walk("."):
        for d in dirs:
            if d.lower() == name.lower(): return os.path.join(root, d)
    return None

BP_DIR = get_path("biological function")
MF_DIR = get_path("molecular function")

# HELPERS
def clean_term(text):
    if not text: return ""
    text = str(text).split(' (GO:')[0].split(' (FDR')[0]
    return text.strip().lower()

def extract_genome_N(filepath):
    """Finds total genome size (e.g. 20580)"""
    if not os.path.exists(filepath): return 20580
    with open(filepath, 'r') as f:
        head = f.read(1024)
    match = re.search(r'REFLIST \((\d+)\)', head)
    if match: return int(match.group(1))
    return 20580

def load_universe_data(filepath):
    """Returns dict Term -> (K_genome, K_list)"""
    data_map = {}
    if not os.path.exists(filepath): return data_map
    with open(filepath, 'r') as f:
        lines = f.readlines()
    start = -1
    for i, line in enumerate(lines):
        if "FDR" in line and "Enrichment" in line:
            start = i + 1
            break
    if start != -1:
        for line in lines[start:]:
            parts = line.split('\t')
            if len(parts) < 3: continue
            try:
                term = clean_term(parts[0])
                k_genome = int(parts[1]) # Col 1: Genome Ref
                k_list = int(parts[2])   # Col 2: List (Correct K for N=122)
                data_map[term] = (k_genome, k_list)
            except: continue
    return data_map

def load_cluster_k(filepath):
    k_map = {}
    if not os.path.exists(filepath): return k_map
    with open(filepath, 'r') as f:
        lines = f.readlines()
    start = -1
    for i, line in enumerate(lines):
        if "FDR" in line and "Enrichment" in line:
            start = i + 1
            break
    if start != -1:
        for line in lines[start:]:
            parts = line.split('\t')
            if len(parts) < 3: continue
            try:
                term = clean_term(parts[0])
                count = int(parts[2])
                k_map[term] = count
            except: continue
    return k_map

# INDEXING
print("Indexing Data...")
N_genome_BP = extract_genome_N("universe_BP.txt")
N_genome_MF = extract_genome_N("universe_MF.txt")

uni_bp_map = load_universe_data("universe_BP.txt")
uni_mf_map = load_universe_data("universe_MF.txt")

cluster_bp_k = {}
cluster_mf_k = {}
for i in range(1, 15):
    fname = f"analysis_cluster{i}.txt"
    if BP_DIR: cluster_bp_k[i] = load_cluster_k(os.path.join(BP_DIR, fname))
    if MF_DIR: cluster_mf_k[i] = load_cluster_k(os.path.join(MF_DIR, fname))

# EXCEL PROCESSING
if not os.path.exists(INPUT_EXCEL):
    print(f"Error: {INPUT_EXCEL} not found.")
else:
    print("Processing Excel with FISHER EXACT TEST...")
    wb = openpyxl.load_workbook(INPUT_EXCEL)
    ws = wb.active

    # Config columns
    COL_ID = 1
    COL_NAMES = 3
    COL_BP = 4
    COL_MF = 7

    # OPTION 1: GENOME (J-S)
    J, K_col, L, M, N_col = 10, 11, 12, 13, 14
    O, P_col, Q, R, S = 15, 16, 17, 18, 19

    # OPTION 2: 122 STRICT (T-AC)
    T, U, V, W, X = 20, 21, 22, 23, 24
    Y, Z, AA, AB, AC = 25, 26, 27, 28, 29

    # --- FISHER EXACT CALCULATION FUNCTION ---
    def calc_fisher_p(N, K, n, k):
        if K is None: return "NF"
        if k > n: k = n # Safety correction

        try:
            # Contingency Table for Fisher:
            #        | Cluster | Background
            # -------|---------|-----------
            # Yes GO |    k    |   K - k
            # No  GO |  n - k  | (N-n) - (K-k)

            table = [[k, K - k],
                     [n - k, (N - n) - (K - k)]]

            oddsratio, p_value = stats.fisher_exact(table, alternative='greater')
            return p_value
        except Exception as e:
            return "Err"

    curr_id, curr_n = None, None

    for row in range(2, ws.max_row + 1):
        # Parse ID/N
        val_id = ws.cell(row, COL_ID).value
        val_names = ws.cell(row, COL_NAMES).value

        # Parse ID
        if val_id:
            clean_id = str(val_id).replace("Cluster","").strip()
            if clean_id.isdigit(): curr_id = int(clean_id)

        # Parse N
        if val_names:
            names_list = [x.strip() for x in str(val_names).split(',') if x.strip()]
            curr_n = len(names_list)

        if not curr_id or not curr_n: continue

        # --- PROCESSOR ---
        def run_row(term_raw, uni_map, clust_map, N_gen, N_122, col_gen, col_122):
            if not term_raw or "No significant" in str(term_raw): return
            term = clean_term(str(term_raw))

            K_gen, K_list = uni_map.get(term, (None, None))
            k = clust_map.get(curr_id, {}).get(term, 0)

            # Write Option 1 (Genome)
            ws.cell(row, col_gen).value = N_gen
            ws.cell(row, col_gen+1).value = K_gen if K_gen else "NF"
            ws.cell(row, col_gen+2).value = curr_n
            ws.cell(row, col_gen+3).value = k
            if K_gen:
                p = calc_fisher_p(N_gen, K_gen, curr_n, k)
                ws.cell(row, col_gen+4).value = p
                if isinstance(p, float): ws.cell(row, col_gen+4).number_format = '0.00E+00'

            # Write Option 2 (122 Strict)
            ws.cell(row, col_122).value = N_122
            ws.cell(row, col_122+1).value = K_list if K_list else "NF"
            ws.cell(row, col_122+2).value = curr_n
            ws.cell(row, col_122+3).value = k
            if K_list:
                p = calc_fisher_p(N_122, K_list, curr_n, k)
                ws.cell(row, col_122+4).value = p
                if isinstance(p, float): ws.cell(row, col_122+4).number_format = '0.00E+00'

        # BP
        run_row(ws.cell(row, COL_BP).value, uni_bp_map, cluster_bp_k, N_genome_BP, 122, J, T)
        # MF
        run_row(ws.cell(row, COL_MF).value, uni_mf_map, cluster_mf_k, N_genome_MF, 122, O, Y)

    print("Saving...")
    wb.save(OUTPUT_EXCEL)
    files.download(OUTPUT_EXCEL)
    print("Done! Fisher Exact Test applied.")